# Baseline: pipeline lengkap

Model asli, data, pretraining, post-training, evaluasi, dan UI source.

Notebook ini menyertakan 100 file source yang diperlukan. Jalankan di runtime Google Colab, bukan local runtime laptop. Sel latihan tidak aktif sampai sakelar `RUN_...` diubah.

Runtime Colab dapat berakhir dan file di `/content` atau `/ephemeral` hilang. Salin checkpoint penting ke Drive.


## 1. Siapkan runtime

Pilih GPU di menu Runtime Colab. PyTorch umumnya sudah tersedia; sel ini memasang paket pendukung. Kode sumber dan sel training di bawah tidak dijalankan saat file dibuat.


In [2]:
%pip install -q h5py numpy tqdm tiktoken zstandard requests datasets wandb


In [3]:
from pathlib import Path
import os, sys
PROJECT_ROOT = Path('/content/train-llm-from-scratch')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
for folder in ['.streamlit', 'config', 'configs', 'configs/smoke', 'data_loader', 'docs/javascripts', 'images', 'scripts', 'src', 'src/models', 'src/post_training', 'src/post_training/rewards', 'tests', 'ui', 'ui/pages']:
    Path(folder).mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT = Path('/ephemeral')
for folder in ('data', 'ckpts', 'logs'):
    (ARTIFACT_ROOT / folder).mkdir(parents=True, exist_ok=True)
print('Project:', PROJECT_ROOT, '| Artifacts:', ARTIFACT_ROOT)


Project: /content/train-llm-from-scratch | Artifacts: /ephemeral


## 2. Source proyek

Setiap sel berikut berisi isi asli satu file. `%%writefile` membuat struktur paket agar import antarmodul bekerja di Colab.


In [4]:
%%writefile .streamlit/config.toml
[theme]
primaryColor = "#16a085"
backgroundColor = "#ffffff"
secondaryBackgroundColor = "#f3faf8"
textColor = "#0a3d33"
font = "sans serif"

[server]
headless = true
runOnSave = true

[browser]
gatherUsageStats = false


Writing .streamlit/config.toml


In [5]:
%%writefile config/__init__.py
"""Configuration package: legacy ``config.config`` (pretraining), the post-training
dataclasses (``config.post_training_config``), and the JSON ``config.loader``."""


Writing config/__init__.py


In [6]:
%%writefile config/config.py
# --- Configuration ---

import torch

# Define vocabulary size and transformer configuration (3 Billion)
VOCAB_SIZE = 50304          # Number of unique tokens in the vocabulary
CONTEXT_LENGTH = 512        # Maximum sequence length for the model
N_EMBED = 2048              # Dimension of the embedding space
N_HEAD = 16                 # Number of attention heads in each transformer block
N_BLOCKS = 64               # Number of transformer blocks in the model

# Paths to training and development datasets
TRAIN_PATH = "data/train/pile_train.h5"  # File path for the training dataset
DEV_PATH = "data/val/pile_dev.h5"      # File path for the validation dataset

# Transformer training parameters
T_BATCH_SIZE = 32          # Number of samples per training batch
T_CONTEXT_LENGTH = 16      # Context length for training batches
T_TRAIN_STEPS = 200000     # Total number of training steps
T_EVAL_STEPS = 1000        # Frequency (in steps) to perform evaluation
T_EVAL_ITERS = 250         # Number of iterations to evaluate the model
T_LR_DECAY_STEP = 50000    # Step at which to decay the learning rate
T_LR = 5e-4                # Initial learning rate for training
T_LR_DECAYED = 5e-5        # Learning rate after decay
T_OUT_PATH = "models/transformer_B.pt"  # Path to save the trained model
T_CHECKPOINT_STEPS = 0      # Save periodic checkpoints every N steps (0 disables)
T_KEEP_LAST_CHECKPOINTS = 3 # Number of periodic checkpoints to keep (0 keeps all)
T_CHECKPOINT_DIR = None     # Optional checkpoint directory override

# Memory-optimisation knobs (all OFF by default => unchanged behaviour/numerics).
# These let large configs fit in less VRAM; enable them on the CLI or here. See issue #5.
USE_AMP = False                 # bf16/fp16 autocast (CUDA only; ignored on CPU)
AMP_DTYPE = "bf16"              # "bf16" (no GradScaler) or "fp16" (GradScaler)
USE_GRADIENT_CHECKPOINTING = False  # recompute block activations in backward to save VRAM
GRAD_ACCUM_STEPS = 1           # micro-batches per optimizer step (effective batch x N)
REPORT_MEMORY_BUDGET = False   # print a rough VRAM budget before training (CUDA only)

# Device configuration
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Store all configurations in a dictionary for easy access and modification
default_config = {
    'vocab_size': VOCAB_SIZE,
    'context_length': CONTEXT_LENGTH,
    'n_embed': N_EMBED,
    'n_head': N_HEAD,
    'n_blocks': N_BLOCKS,
    'train_path': TRAIN_PATH,
    'dev_path': DEV_PATH,
    't_batch_size': T_BATCH_SIZE,
    't_context_length': T_CONTEXT_LENGTH,
    't_train_steps': T_TRAIN_STEPS,
    't_eval_steps': T_EVAL_STEPS,
    't_eval_iters': T_EVAL_ITERS,
    't_lr_decay_step': T_LR_DECAY_STEP,
    't_lr': T_LR,
    't_lr_decayed': T_LR_DECAYED,
    't_out_path': T_OUT_PATH,
    't_checkpoint_steps': T_CHECKPOINT_STEPS,
    't_keep_last_checkpoints': T_KEEP_LAST_CHECKPOINTS,
    't_checkpoint_dir': T_CHECKPOINT_DIR,
    'use_amp': USE_AMP,
    'amp_dtype': AMP_DTYPE,
    'use_gradient_checkpointing': USE_GRADIENT_CHECKPOINTING,
    'grad_accum_steps': GRAD_ACCUM_STEPS,
    'report_memory_budget': REPORT_MEMORY_BUDGET,
    'device': DEVICE,
}


Writing config/config.py


In [7]:
%%writefile config/loader.py
"""
JSON config loader for the post-training stages.

Industry-standard, learning-friendly: each stage's knobs live in a small editable JSON file
under ``configs/`` (typed by the dataclasses in :mod:`config.post_training_config`). This
loader resolves a final dataclass instance by merging four layers, lowest precedence first:

    1. dataclass field defaults        (config.post_training_config.<Stage>Config)
    2. configs/base.json               (shared model + runtime fields)
    3. the stage JSON (configs/sft.json, ...)   (that stage's hyperparameters)
    4. CLI --field overrides           (highest precedence)

JSON ``null`` maps to Python ``None`` cleanly -- the right way to set ``str | None`` fields
like ``amp_dtype``. Unknown keys are warned about, not fatal.

When ``json_path`` lives in a sub-dir with its own ``base.json`` (e.g. ``configs/smoke/sft.json``),
that sibling ``base.json`` is used automatically -- so the smoke configs shrink the model too.
"""

from __future__ import annotations

import json
import os
from dataclasses import fields
from typing import Any


def _deep_merge(dst: dict, src: dict) -> dict:
    """Recursively merge ``src`` into ``dst`` (nested-dict aware; future-proof)."""
    for k, v in src.items():
        if isinstance(v, dict) and isinstance(dst.get(k), dict):
            _deep_merge(dst[k], v)
        else:
            dst[k] = v
    return dst


def _resolve_base(json_path: str | None, base_path: str | None) -> str:
    if base_path is not None:
        return base_path
    if json_path:
        sibling = os.path.join(os.path.dirname(json_path), "base.json")
        if os.path.exists(sibling):
            return sibling
    return "configs/base.json"


def load_config(
    cfg_cls,
    json_path: str | None = None,
    overrides: dict[str, Any] | None = None,
    *,
    base_path: str | None = None,
):
    """
    Resolve ``cfg_cls`` from ``base.json`` + the stage JSON + CLI overrides.

    Args:
        cfg_cls: the stage dataclass (e.g. ``SFTConfig``).
        json_path: path to the stage JSON (e.g. ``configs/sft.json``); None = base + defaults.
        overrides: parsed CLI ``--field`` values (None values are ignored).
        base_path: shared base JSON; if None, uses the sibling ``base.json`` of ``json_path``
            (so ``configs/smoke/sft.json`` picks up ``configs/smoke/base.json``), else
            ``configs/base.json``.

    Returns:
        an instance of ``cfg_cls`` with the resolved values.
    """
    base = _resolve_base(json_path, base_path)
    merged: dict[str, Any] = {}
    for path in (base, json_path):
        if path and os.path.exists(path):
            with open(path) as fh:
                _deep_merge(merged, json.load(fh))

    field_names = {f.name for f in fields(cfg_cls)}
    for key in list(merged):
        if key not in field_names:
            print(f"[config] ignoring unknown key '{key}' for {cfg_cls.__name__}")
            merged.pop(key)

    if overrides:
        merged.update({k: v for k, v in overrides.items() if k in field_names and v is not None})

    return cfg_cls(**merged)


Writing config/loader.py


In [8]:
%%writefile config/post_training_config.py
"""
Configuration for the from-scratch post-training suite.

Kept entirely separate from ``config/config.py`` (which import-executes for the original
pretraining path and must stay untouched). Each stage is a frozen-ish dataclass that
inherits the shared :class:`BaseModelConfig` model/runtime fields and adds its own
hyperparameters. Construct with overrides, e.g. ``SFTConfig(lr=2e-5, batch_size=16)``.

The default base model is ~400M parameters (n_embed=1024, n_head=16, n_blocks=24,
context_length=1024) -- the "mid" size chosen so real datasets (Alpaca, HH-RLHF, GSM8K)
give meaningful results while still fitting comfortably on one H100 and training in a
reasonable time on 2x H100. A tiny ``SMOKE`` variant is provided for fast CPU/1-GPU tests.
"""

from __future__ import annotations

from dataclasses import dataclass, field, replace


# Shared paths (all heavy artifacts live on the 1.5TB /ephemeral disk).
EPHEMERAL = "/ephemeral"
CKPT_DIR = f"{EPHEMERAL}/ckpts"
DATA_DIR = f"{EPHEMERAL}/data"
LOG_DIR = f"{EPHEMERAL}/logs"


@dataclass
class BaseModelConfig:
    # --- model architecture (must match across all stages + the pretrained ckpt) ---
    vocab_size: int = 50304
    context_length: int = 1024
    n_embed: int = 1024
    n_head: int = 16
    n_blocks: int = 24

    # --- runtime ---
    device: str = "cuda"
    amp_dtype: str | None = "bf16"      # None | "bf16"; bf16 needs no GradScaler on H100
    seed: int = 1337
    compile: bool = False               # torch.compile the model (big speedup, slow 1st step)
    ckpt_dir: str = CKPT_DIR
    log_dir: str = LOG_DIR
    use_wandb: bool = False
    wandb_project: str = "train-llm-from-scratch-posttrain"


@dataclass
class PretrainConfig(BaseModelConfig):
    """Pretrain the mid base model from scratch on the Pile HDF5 (mix in task text late)."""
    train_path: str = "data/train/pile_train.h5"
    dev_path: str = "data/val/pile_dev.h5"
    batch_size: int = 24                # per-GPU micro-batch
    grad_accum: int = 8                 # effective batch = batch_size * grad_accum * world
    train_steps: int = 200_000
    eval_steps: int = 1_000
    eval_iters: int = 100
    warmup_steps: int = 2_000
    lr: float = 3e-4
    min_lr: float = 3e-5
    weight_decay: float = 0.1
    grad_clip: float = 1.0
    out_ckpt: str = f"{CKPT_DIR}/base_pretrained.pt"
    save_every: int = 2_000


@dataclass
class SFTConfig(BaseModelConfig):
    pretrained_ckpt: str = f"{CKPT_DIR}/base_pretrained.pt"
    data_path: str = f"{DATA_DIR}/sft_packed.h5"
    out_ckpt: str = f"{CKPT_DIR}/sft.pt"
    batch_size: int = 16
    grad_accum: int = 2
    epochs: int = 3
    max_steps: int = -1                 # -1 = run full epochs
    eval_steps: int = 200
    warmup_steps: int = 100
    lr: float = 1e-5
    min_lr: float = 1e-6
    weight_decay: float = 0.0
    grad_clip: float = 1.0
    save_every: int = 500


@dataclass
class RewardConfig(BaseModelConfig):
    sft_ckpt: str = f"{CKPT_DIR}/sft.pt"
    pref_path: str = f"{DATA_DIR}/preferences.jsonl"
    out_ckpt: str = f"{CKPT_DIR}/reward.pt"
    batch_size: int = 8                 # pairs per step (2x sequences through the model)
    epochs: int = 1
    eval_steps: int = 200
    warmup_steps: int = 50
    lr: float = 1e-5
    weight_decay: float = 0.0
    grad_clip: float = 1.0
    max_len: int = 768
    save_every: int = 500


@dataclass
class DPOConfig(BaseModelConfig):
    sft_ckpt: str = f"{CKPT_DIR}/sft.pt"        # init policy + frozen reference
    pref_path: str = f"{DATA_DIR}/preferences.jsonl"
    out_ckpt: str = f"{CKPT_DIR}/dpo.pt"
    loss_type: str = "dpo"              # "dpo" | "orpo" | "kto"
    beta: float = 0.1
    orpo_lambda: float = 1.0           # ORPO odds-ratio weight (loss_type="orpo")
    batch_size: int = 8
    epochs: int = 1
    eval_steps: int = 200
    warmup_steps: int = 50
    lr: float = 5e-7
    weight_decay: float = 0.0
    grad_clip: float = 1.0
    max_len: int = 768
    save_every: int = 500


@dataclass
class PPOConfig(BaseModelConfig):
    sft_ckpt: str = f"{CKPT_DIR}/sft.pt"
    reward_ckpt: str = f"{CKPT_DIR}/reward.pt"   # used when reward_source="rm"
    prompt_path: str = f"{DATA_DIR}/rl_prompts_train.jsonl"
    eval_prompt_path: str = f"{DATA_DIR}/rl_prompts_test.jsonl"
    out_ckpt: str = f"{CKPT_DIR}/ppo.pt"
    reward_source: str = "verifier"    # "verifier" (GSM8K checker) | "rm" (reward model)
    iterations: int = 1_000
    prompts_per_iter: int = 32         # prompts sampled per PPO iteration (per rank)
    rollout_len: int = 300
    temperature: float = 1.0
    top_p: float = 1.0
    ppo_epochs: int = 4
    minibatch_size: int = 16
    clip: float = 0.2
    vf_clip: float = 0.2
    vf_coef: float = 0.5
    ent_coef: float = 0.0
    gamma: float = 1.0
    gae_lambda: float = 0.95
    kl_coef: float = 0.05              # penalty on KL(policy || ref) added to reward
    lr: float = 1e-6
    grad_clip: float = 1.0
    eval_every: int = 50
    save_every: int = 100


@dataclass
class GRPOConfig(BaseModelConfig):
    sft_ckpt: str = f"{CKPT_DIR}/sft.pt"
    prompt_path: str = f"{DATA_DIR}/rl_prompts_train.jsonl"
    eval_prompt_path: str = f"{DATA_DIR}/rl_prompts_test.jsonl"
    curriculum_path: str = f"{DATA_DIR}/arithmetic_prompts.jsonl"  # warm-up before GSM8K
    curriculum_iters: int = 100        # iterations on the arithmetic warm-up before GSM8K
    out_ckpt: str = f"{CKPT_DIR}/grpo.pt"
    iterations: int = 1_000
    prompts_per_iter: int = 8          # distinct prompts per iter (per rank)
    group_size: int = 8               # samples per prompt (group)
    rollout_len: int = 300
    temperature: float = 1.0
    top_p: float = 1.0
    grpo_epochs: int = 1
    clip: float = 0.2
    kl_coef: float = 0.04             # KL(policy || ref) penalty term in the loss
    lr: float = 1e-6
    grad_clip: float = 1.0
    eval_every: int = 50
    save_every: int = 100


# Tiny config for fast smoke tests (CPU or a single GPU, seconds not hours).
SMOKE = dict(
    vocab_size=256, context_length=64, n_embed=64, n_head=4, n_blocks=2, device="cpu", amp_dtype=None
)


def smoke(cfg_cls):
    """Return an instance of ``cfg_cls`` shrunk to the tiny SMOKE model dims."""
    return replace(cfg_cls(), **SMOKE)


Writing config/post_training_config.py


In [9]:
%%writefile configs/base.json
{
  "vocab_size": 50304,
  "context_length": 1024,
  "n_embed": 1024,
  "n_head": 16,
  "n_blocks": 24,
  "device": "cuda",
  "amp_dtype": "bf16",
  "seed": 1337,
  "compile": false,
  "ckpt_dir": "/ephemeral/ckpts",
  "log_dir": "/ephemeral/logs",
  "use_wandb": false,
  "wandb_project": "train-llm-from-scratch-posttrain"
}


Writing configs/base.json


In [10]:
%%writefile configs/dpo.json
{
  "sft_ckpt": "/ephemeral/ckpts/sft.pt",
  "pref_path": "/ephemeral/data/preferences.jsonl",
  "out_ckpt": "/ephemeral/ckpts/dpo.pt",
  "loss_type": "dpo",
  "beta": 0.1,
  "orpo_lambda": 1.0,
  "batch_size": 8,
  "epochs": 1,
  "eval_steps": 200,
  "warmup_steps": 50,
  "lr": 5e-07,
  "weight_decay": 0.0,
  "grad_clip": 1.0,
  "max_len": 768,
  "save_every": 500
}


Writing configs/dpo.json


In [11]:
%%writefile configs/grpo.json
{
  "sft_ckpt": "/ephemeral/ckpts/sft.pt",
  "prompt_path": "/ephemeral/data/rl_prompts_train.jsonl",
  "eval_prompt_path": "/ephemeral/data/rl_prompts_test.jsonl",
  "curriculum_path": "/ephemeral/data/arithmetic_prompts.jsonl",
  "curriculum_iters": 100,
  "out_ckpt": "/ephemeral/ckpts/grpo.pt",
  "iterations": 1000,
  "prompts_per_iter": 8,
  "group_size": 8,
  "rollout_len": 300,
  "temperature": 1.0,
  "top_p": 1.0,
  "grpo_epochs": 1,
  "clip": 0.2,
  "kl_coef": 0.04,
  "lr": 1e-06,
  "grad_clip": 1.0,
  "eval_every": 50,
  "save_every": 100
}


Writing configs/grpo.json


In [12]:
%%writefile configs/ppo.json
{
  "sft_ckpt": "/ephemeral/ckpts/sft.pt",
  "reward_ckpt": "/ephemeral/ckpts/reward.pt",
  "prompt_path": "/ephemeral/data/rl_prompts_train.jsonl",
  "eval_prompt_path": "/ephemeral/data/rl_prompts_test.jsonl",
  "out_ckpt": "/ephemeral/ckpts/ppo.pt",
  "reward_source": "verifier",
  "iterations": 1000,
  "prompts_per_iter": 32,
  "rollout_len": 300,
  "temperature": 1.0,
  "top_p": 1.0,
  "ppo_epochs": 4,
  "minibatch_size": 16,
  "clip": 0.2,
  "vf_clip": 0.2,
  "vf_coef": 0.5,
  "ent_coef": 0.0,
  "gamma": 1.0,
  "gae_lambda": 0.95,
  "kl_coef": 0.05,
  "lr": 1e-06,
  "grad_clip": 1.0,
  "eval_every": 50,
  "save_every": 100
}


Writing configs/ppo.json


In [13]:
%%writefile configs/pretrain.json
{
  "train_path": "/ephemeral/data/pile_train.h5",
  "dev_path": "/ephemeral/data/pile_dev.h5",
  "batch_size": 8,
  "grad_accum": 12,
  "train_steps": 50000,
  "eval_steps": 1000,
  "eval_iters": 100,
  "warmup_steps": 2000,
  "lr": 0.0003,
  "min_lr": 3e-05,
  "weight_decay": 0.1,
  "grad_clip": 1.0,
  "out_ckpt": "/ephemeral/ckpts/base_pretrained.pt",
  "save_every": 1000
}


Writing configs/pretrain.json


In [14]:
%%writefile configs/reward.json
{
  "sft_ckpt": "/ephemeral/ckpts/sft.pt",
  "pref_path": "/ephemeral/data/preferences.jsonl",
  "out_ckpt": "/ephemeral/ckpts/reward.pt",
  "batch_size": 8,
  "epochs": 1,
  "eval_steps": 200,
  "warmup_steps": 50,
  "lr": 1e-05,
  "weight_decay": 0.0,
  "grad_clip": 1.0,
  "max_len": 768,
  "save_every": 500
}


Writing configs/reward.json


In [15]:
%%writefile configs/sft.json
{
  "pretrained_ckpt": "/ephemeral/ckpts/base_pretrained.pt",
  "data_path": "/ephemeral/data/sft_packed.h5",
  "out_ckpt": "/ephemeral/ckpts/sft.pt",
  "batch_size": 16,
  "grad_accum": 2,
  "epochs": 3,
  "max_steps": -1,
  "eval_steps": 200,
  "warmup_steps": 100,
  "lr": 1e-05,
  "min_lr": 1e-06,
  "weight_decay": 0.0,
  "grad_clip": 1.0,
  "save_every": 500
}


Writing configs/sft.json


In [16]:
%%writefile configs/smoke/base.json
{
  "vocab_size": 50304,
  "context_length": 256,
  "n_embed": 128,
  "n_head": 4,
  "n_blocks": 2,
  "device": "cpu",
  "amp_dtype": null
}


Writing configs/smoke/base.json


In [17]:
%%writefile configs/smoke/dpo.json
{
  "batch_size": 4,
  "epochs": 1,
  "eval_steps": 10000,
  "warmup_steps": 2,
  "max_len": 256,
  "save_every": 10000
}


Writing configs/smoke/dpo.json


In [18]:
%%writefile configs/smoke/grpo.json
{
  "iterations": 2,
  "prompts_per_iter": 2,
  "group_size": 4,
  "rollout_len": 32,
  "curriculum_iters": 1,
  "grpo_epochs": 1,
  "eval_every": 10000,
  "save_every": 10000
}


Writing configs/smoke/grpo.json


In [19]:
%%writefile configs/smoke/ppo.json
{
  "iterations": 2,
  "prompts_per_iter": 4,
  "rollout_len": 32,
  "ppo_epochs": 2,
  "minibatch_size": 2,
  "eval_every": 10000,
  "save_every": 10000
}


Writing configs/smoke/ppo.json


In [20]:
%%writefile configs/smoke/pretrain.json
{
  "batch_size": 4,
  "grad_accum": 1,
  "train_steps": 20,
  "eval_steps": 10000,
  "warmup_steps": 2,
  "save_every": 10000,
  "train_path": "/ephemeral/data/pile_train.h5",
  "dev_path": "/ephemeral/data/pile_dev.h5"
}


Writing configs/smoke/pretrain.json


In [21]:
%%writefile configs/smoke/reward.json
{
  "batch_size": 4,
  "epochs": 1,
  "eval_steps": 10000,
  "warmup_steps": 2,
  "max_len": 256,
  "save_every": 10000
}


Writing configs/smoke/reward.json


In [22]:
%%writefile configs/smoke/sft.json
{
  "max_steps": 10,
  "batch_size": 4,
  "eval_steps": 10000,
  "warmup_steps": 2,
  "save_every": 10000
}


Writing configs/smoke/sft.json


In [23]:
%%writefile data_loader/__init__.py
"""Data loaders: pretraining HDF5 iterator + SFT / preference / RL-prompt iterators."""


Writing data_loader/__init__.py


In [24]:
%%writefile data_loader/data_loader.py
import torch
import numpy as np
import h5py
from typing import Iterator, Tuple

def get_batch_iterator(data_path: str, batch_size: int, context_length: int, device: str = "cpu") -> Iterator[Tuple[torch.Tensor, torch.Tensor]]:
    """
    Creates an iterator for generating batches of data from an HDF5 file.

    Args:
        data_path (str): Path to the HDF5 file containing tokenized data.
        batch_size (int): Number of sequences in each batch.
        context_length (int): Length of each sequence.
        device (str, optional): Device to load the data onto ('cpu' or 'cuda'). Defaults to "cpu".

    Yields:
        tuple: A tuple containing input sequences (xb) and target sequences (yb).
    """
    # Open the HDF5 file in read mode
    with h5py.File(data_path, 'r') as hdf5_file:

        # Extract the dataset of tokenized sequences
        dataset = hdf5_file['tokens']

        # Get the total size of the dataset
        dataset_size = dataset.shape[0]

        # Calculate the number of examples (sequences) that can be made from the data
        n_examples = (dataset_size - 1) // context_length

        # Create an array of indices for examples and shuffle them for randomness
        example_idxs = np.arange(n_examples)
        np.random.shuffle(example_idxs)

        # Initialize epoch counter and example counter
        epochs = 0
        counter = 0

        while True:
            # Check if the current batch exceeds the number of available examples
            if counter + batch_size > n_examples:
                # Shuffle the indices again and reset the counter to 0
                np.random.shuffle(example_idxs)
                counter = 0
                print(f"Finished epoch {epochs}")  # Print epoch number when an epoch finishes
                epochs += 1  # Increment the epoch counter

            # Select a batch of random indices to generate sequences
            random_indices = example_idxs[counter:counter+batch_size] * context_length

            # Retrieve sequences from the dataset based on the random indices
            random_samples = torch.tensor(np.array([dataset[idx:idx+context_length+1] for idx in random_indices]))

            # Separate the input sequences (xb) and target sequences (yb)
            xb = random_samples[:, :context_length].to(device)  # Input sequence (first half of the random sample)
            yb = random_samples[:, 1:context_length+1].to(device)  # Target sequence (second half of the random sample)

            # Increment the counter to move to the next batch
            counter += batch_size

            # Yield the input and target sequences as a tuple for the current batch
            yield xb, yb

if __name__ == '__main__':
    # Example Usage (requires a dummy HDF5 file for testing)
    # Create a dummy HDF5 file
    import os
    dummy_data_path = "dummy_data.h5"
    if not os.path.exists(dummy_data_path):
        with h5py.File(dummy_data_path, 'w') as f:
            f.create_dataset('tokens', data=np.arange(1000))

    batch_size = 4
    context_length = 10
    for xb, yb in get_batch_iterator(dummy_data_path, batch_size, context_length):
        print("Input Batch Shape:", xb.shape)
        print("Target Batch Shape:", yb.shape)
        break


Writing data_loader/data_loader.py


In [25]:
%%writefile data_loader/preference_dataset.py
"""
Batch iterator over preference pairs for reward-model and DPO training.

Reads a JSONL file of ``{"prompt", "chosen", "rejected"}`` (produced by
``scripts/prepare_preference_data.py``). Each side is rendered through the chat template
so we get, for the chosen and rejected responses to the same prompt:
  - token ids of ``prompt + response + EOT``
  - a response mask (1 over the completion, used by DPO)
  - the true sequence length (used by the reward model to read the last-token reward)

Right-padding is safe here because the model's attention is causal: the last real token
never attends to padding that comes after it, and the response mask zeros padded
positions in the loss.
"""

from __future__ import annotations

import json
from typing import Iterator

import numpy as np
import torch

from src.post_training.chat_template import EOT_ID, encode_chat


def _encode_side(prompt: str, response: str, max_len: int) -> tuple[list[int], list[int]]:
    ids, mask = encode_chat([{"role": "user", "content": prompt},
                             {"role": "assistant", "content": response}])
    return ids[:max_len], mask[:max_len]


def _collate(rows: list[dict], max_len: int, device: str) -> dict:
    enc = [( _encode_side(r["prompt"], r["chosen"], max_len),
             _encode_side(r["prompt"], r["rejected"], max_len)) for r in rows]
    # Pad chosen and rejected to a single common length so they can share one forward.
    L = max(max(len(c[0]), len(j[0])) for c, j in enc)

    def pad(seq, fill):
        return seq + [fill] * (L - len(seq))

    ch_ids, ch_mask, ch_len, rj_ids, rj_mask, rj_len = [], [], [], [], [], []
    for (cids, cmask), (jids, jmask) in enc:
        ch_len.append(len(cids)); rj_len.append(len(jids))
        ch_ids.append(pad(cids, EOT_ID)); ch_mask.append(pad(cmask, 0))
        rj_ids.append(pad(jids, EOT_ID)); rj_mask.append(pad(jmask, 0))

    t = lambda a, dt: torch.tensor(a, dtype=dt, device=device)
    return {
        "chosen_ids": t(ch_ids, torch.long), "chosen_mask": t(ch_mask, torch.long), "chosen_len": t(ch_len, torch.long),
        "rejected_ids": t(rj_ids, torch.long), "rejected_mask": t(rj_mask, torch.long), "rejected_len": t(rj_len, torch.long),
    }


def get_preference_iterator(
    path: str,
    batch_size: int,
    max_len: int,
    device: str = "cpu",
    *,
    rank: int = 0,
    world_size: int = 1,
    shuffle: bool = True,
    infinite: bool = True,
) -> Iterator[dict]:
    """Yield collated preference batches (dict of tensors). Rows are sharded across ranks."""
    with open(path) as f:
        rows = [json.loads(line) for line in f if line.strip()]
    rows = rows[rank::world_size]
    rng = np.random.default_rng(7 + rank)
    while True:
        order = np.arange(len(rows))
        if shuffle:
            rng.shuffle(order)
        for s in range(0, len(order) - batch_size + 1, batch_size):
            batch = [rows[i] for i in order[s:s + batch_size]]
            yield _collate(batch, max_len, device)
        if not infinite:
            return


Writing data_loader/preference_dataset.py


In [26]:
%%writefile data_loader/prompt_dataset.py
"""
Prompt iterator for RL (PPO / GRPO). Reads a JSONL of ``{"prompt", "gold"}`` (gold is the
verifiable numeric answer, or null) and yields lists of sampled rows. Rows are sharded
across ranks so each GPU optimizes on its own prompts.
"""

from __future__ import annotations

import json
from typing import Iterator

import numpy as np


def load_prompt_rows(path: str) -> list[dict]:
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]


def get_prompt_iterator(
    path: str,
    prompts_per_iter: int,
    *,
    rank: int = 0,
    world_size: int = 1,
    seed: int = 0,
    shuffle: bool = True,
) -> Iterator[list[dict]]:
    """Infinitely yield lists of ``prompts_per_iter`` rows (this rank's shard)."""
    rows = load_prompt_rows(path)[rank::world_size]
    rng = np.random.default_rng(seed + rank)
    n = len(rows)
    while True:
        idx = rng.permutation(n) if shuffle else np.arange(n)
        for s in range(0, n - prompts_per_iter + 1, prompts_per_iter):
            yield [rows[i] for i in idx[s:s + prompts_per_iter]]


Writing data_loader/prompt_dataset.py


In [27]:
%%writefile data_loader/sft_dataset.py
"""
Batch iterator for packed SFT data (mirrors ``data_loader.data_loader.get_batch_iterator``
but carries the per-token loss mask alongside the tokens).

The HDF5 file has two aligned datasets, ``tokens`` and ``loss_mask``, both shape
(N, context_length), produced by ``scripts/prepare_sft_data.py``.
"""

from __future__ import annotations

from typing import Iterator

import h5py
import numpy as np
import torch


def get_sft_batch_iterator(
    data_path: str,
    batch_size: int,
    device: str = "cpu",
    *,
    rank: int = 0,
    world_size: int = 1,
    shuffle: bool = True,
    infinite: bool = True,
) -> Iterator[tuple[torch.Tensor, torch.Tensor, int]]:
    """
    Yield ``(tokens, loss_mask, epoch)`` batches from a packed SFT HDF5 file.

    Rows are sharded across ranks (each rank sees a disjoint stride) so DDP data-parallel
    training covers the dataset once per epoch. Set ``infinite=False`` to stop after one
    pass (used for evaluation).
    """
    with h5py.File(data_path, "r") as f:
        tokens = f["tokens"]
        masks = f["loss_mask"]
        n = tokens.shape[0]
        idxs = np.arange(rank, n, world_size)
        epoch = 0
        rng = np.random.default_rng(1234 + rank)
        while True:
            if shuffle:
                rng.shuffle(idxs)
            for start in range(0, len(idxs) - batch_size + 1, batch_size):
                batch = np.sort(idxs[start: start + batch_size])  # sorted for h5py fancy-index
                tk = torch.tensor(np.asarray(tokens[batch]), dtype=torch.long, device=device)
                mk = torch.tensor(np.asarray(masks[batch]), dtype=torch.long, device=device)
                yield tk, mk, epoch
            epoch += 1
            if not infinite:
                return


Writing data_loader/sft_dataset.py


In [28]:
%%writefile docs/javascripts/mathjax.js
window.MathJax = {
  tex: {
    inlineMath: [["\\(", "\\)"]],
    displayMath: [["\\[", "\\]"]],
    processEscapes: true,
    processEnvironments: true
  },
  options: {
    ignoreHtmlClass: ".*|",
    processHtmlClass: "arithmatex"
  }
};


Writing docs/javascripts/mathjax.js


In [29]:
%%writefile images/make_diagrams.py
"""
Generate every README diagram as a Mermaid .mmd file, then render to PNG with the hand-drawn
look. Reproducible: edit a graph below, re-run, re-render.

Render step (run from this folder, needs Node.js for npx):
    for f in *.mmd; do
      npx -y @mermaid-js/mermaid-cli -i "$f" -o "${f%.mmd}.png" -c mmdc.json -p puppeteer.json -b white -s 3
    done
"""
import os

OUT = os.path.dirname(os.path.abspath(__file__))

# Shared palette: soft fill + strong border per role (data, store, proc, model, rl, loss, ckpt, eval, out, op).
PALETTE = """
  classDef data fill:#cdeccd,stroke:#2e7d32,stroke-width:2px,color:#143d1a;
  classDef store fill:#cfeee9,stroke:#1f9e8f,stroke-width:2px,color:#0c3a34;
  classDef proc fill:#cfe3fb,stroke:#1565c0,stroke-width:2px,color:#0d2c52;
  classDef model fill:#ffe8a3,stroke:#d48806,stroke-width:2px,color:#5a3d00;
  classDef rl fill:#ffd9b3,stroke:#e67e22,stroke-width:2px,color:#6b3500;
  classDef loss fill:#ffd4d4,stroke:#c0392b,stroke-width:2px,color:#5c1212;
  classDef ckpt fill:#ececec,stroke:#666666,stroke-width:2px,color:#222222;
  classDef eval fill:#e7d6fb,stroke:#8e44ad,stroke-width:2px,color:#3d1a5a;
  classDef out fill:#e7d6fb,stroke:#7b3fbf,stroke-width:2px,color:#3d1a5a;
  classDef op fill:#ffffff,stroke:#888888,stroke-width:2px,color:#333333;
"""

DIAGRAMS = {
"00_pipeline": """flowchart LR
  PILE([The Pile<br/>raw text]):::data --> PRE{{Pretrain<br/>base ~400M}}:::model
  PRE --> SFT{{SFT<br/>instruct}}:::model
  SFT --> RM{{Reward Model<br/>Bradley-Terry}}:::rl
  SFT --> DPO{{DPO / ORPO / KTO}}:::rl
  SFT --> GRPO{{GRPO / RLVR}}:::rl
  RM -->|reward signal| PPO{{PPO}}:::rl
  PPO --> EVAL([GSM8K eval<br/>+ chat]):::eval
  DPO --> EVAL
  GRPO --> EVAL
""",

"01_data": """flowchart LR
  A1([Pretrain: The Pile]):::data --> A2[tiktoken r50k_base<br/>stream + encode]:::proc --> A3[(pile_train.h5<br/>flat int32 tokens)]:::store
  B1([SFT: Alpaca · Dolly · GSM8K]):::data --> B2[chat template<br/>+ mask the prompt]:::proc --> B3[(sft_packed.h5<br/>tokens + loss_mask)]:::store
  C1([Prefs: HH-RLHF · UltraFeedback]):::data --> C2[split prompt /<br/>chosen / rejected]:::proc --> C3[(preferences.jsonl)]:::store
  D1([RL: GSM8K · arithmetic]):::data --> D2[keep the gold answer]:::proc --> D3[(rl_prompts.jsonl)]:::store
""",

"02_tokenization": """flowchart LR
  T1["raw text<br/>'Once upon a time'"]:::data --> T2["tiktoken<br/>r50k_base"]:::proc
  T2 --> T3["token ids<br/>[7454, 2402, 257, 640]"]:::store
  T3 --> T4["append &lt;|endoftext|&gt;<br/>(id 50256)"]:::proc
  T4 --> T5["fixed windows of<br/>context_length + 1"]:::store
  T5 --> T6["x = window[:-1]<br/>y = window[1:]"]:::out
""",

"03_mlp": """flowchart LR
  M1["x  (B, T, n_embed)"]:::data --> M2["Linear<br/>n_embed -> 4 · n_embed"]:::proc
  M2 --> M3["ReLU"]:::proc
  M3 --> M4["Linear<br/>4 · n_embed -> n_embed"]:::proc
  M4 --> M5["out  (B, T, n_embed)"]:::out
""",

"04_attention_head": """flowchart LR
  X["x : token vectors<br/>(B, T, C)"]:::data --> Q["Q = x Wq"]:::proc
  X --> K["K = x Wk"]:::proc
  X --> V["V = x Wv"]:::proc
  Q --> S["scores = Q Kᵀ / √d"]:::rl
  K --> S
  S --> M["causal mask<br/>+ softmax"]:::rl
  M --> O["weights · V"]:::rl
  V --> O
  O --> H["head output<br/>(B, T, head_size)"]:::out
""",

"05_multi_head_attention": """flowchart LR
  X["x  (B, T, C)"]:::data --> H1["head 1"]:::proc
  X --> H2["head 2"]:::proc
  X --> H3["...  (n_head heads)"]:::proc
  H1 --> CC["concat heads<br/>(B, T, C)"]:::rl
  H2 --> CC
  H3 --> CC
  CC --> PJ["output projection<br/>Linear C -> C"]:::proc
  PJ --> HO["multi-head output"]:::out
""",

"06_transformer_block": """flowchart LR
  X["x"]:::data --> LN1["LayerNorm"]:::proc --> AT["Multi-Head Attention"]:::rl --> A1(("+")):::op
  X --> A1
  A1 --> LN2["LayerNorm"]:::proc --> ML["MLP"]:::proc --> A2(("+")):::op
  A1 --> A2
  A2 --> BO["block output"]:::out
""",

"07_transformer": """flowchart LR
  I["token ids  (B, T)"]:::data --> TE["token embedding"]:::proc
  I --> PE["position embedding"]:::proc
  TE --> SUM(("+")):::op
  PE --> SUM
  SUM --> BLK["N_BLOCKS ×<br/>Transformer Block"]:::model
  BLK --> LNF["final LayerNorm"]:::proc
  LNF --> LMH["lm_head<br/>Linear -> vocab_size"]:::proc
  LMH --> LOG["logits  (B, T, vocab)"]:::out
  LOG -. targets present .-> CE["cross-entropy loss"]:::loss
""",

"08_training_loop": """flowchart LR
  D[(pile_train.h5)]:::store --> IT["get_batch_iterator<br/>random windows"]:::proc --> FW["forward<br/>(bf16 autocast)"]:::model --> CE["cross-entropy"]:::loss --> BW["backward<br/>× grad_accum"]:::proc --> CL["clip grad norm 1.0"]:::proc --> ST["AdamW step<br/>cosine LR + warmup"]:::model --> CK[(checkpoint)]:::ckpt
  ST -. next step .-> IT
""",

"09_sft": """flowchart LR
  S0[(sft_packed.h5<br/>tokens + loss_mask)]:::store --> S1["Transformer forward"]:::model --> S2["shift: predict t+1"]:::proc --> S3["per-token cross-entropy"]:::loss
  S0 --> S4["loss_mask = 1 only on<br/>assistant tokens"]:::proc --> S5["mean over masked tokens"]:::loss
  S3 --> S5 --> S6["AdamW step"]:::model
""",

"10_reward_model": """flowchart LR
  R0["prompt + chosen / rejected"]:::data --> R1["SFT backbone<br/>forward_hidden"]:::model --> R2["take last real token"]:::proc --> R3["reward head<br/>Linear -> 1"]:::proc
  R3 --> R4["r_chosen , r_rejected"]:::rl --> R5["Bradley-Terry loss<br/>-log σ(r_chosen - r_rejected)"]:::loss --> R6["AdamW step"]:::model
""",

"11_dpo": """flowchart LR
  P0["chosen / rejected pair"]:::data --> P1["policy (trainable)<br/>sequence log-probs"]:::model
  P0 --> P2["reference (frozen SFT)<br/>sequence log-probs"]:::ckpt
  P1 --> P3["DPO loss<br/>-log σ(β · Δ)"]:::loss
  P2 --> P3 --> P4["AdamW step"]:::model
""",

"12_ppo": """flowchart LR
  Q0([GSM8K prompts]):::data --> Q1["rollout<br/>generate + log-probs"]:::model --> Q2["score<br/>verifier or reward model"]:::rl --> Q3["+ per-token KL to ref"]:::rl --> Q4["GAE<br/>advantages + returns"]:::proc --> Q5["clipped policy + value<br/>(K epochs)"]:::model --> Q6[(ppo.pt)]:::ckpt
  Q5 -. sync old policy .-> Q1
""",

"13_grpo": """flowchart LR
  G0([prompt]):::data --> G1["sample a GROUP<br/>of G answers"]:::model --> G2["verifier reward<br/>per answer"]:::rl --> G3["group advantage<br/>(r - mean) / std"]:::proc --> G4["clipped surrogate<br/>+ k3 KL to ref"]:::loss --> G5["policy update"]:::model
  G5 -. next prompt .-> G0
""",

"14_evaluation": """flowchart LR
  E0[(stage checkpoint)]:::ckpt --> E1["greedy generate"]:::model --> E2["extract_answer<br/>&lt;answer&gt;N&lt;/answer&gt;"]:::proc --> E3{"== gold?"}:::rl --> E4["GSM8K accuracy<br/>Base -> SFT -> ... -> GRPO"]:::eval
""",

"15_inference": """flowchart LR
  C0[(any checkpoint)]:::ckpt --> C1["load dims from cfg"]:::proc --> C2{"chat or raw?"}:::rl
  C2 -->|instruct| C3["wrap in chat template"]:::proc
  C2 -->|base| C4["raw prefix"]:::proc
  C3 --> C5["generate<br/>temperature / top-p / greedy"]:::model
  C4 --> C5 --> C6["decode -> reply"]:::out
""",
}

def main():
    for name, body in DIAGRAMS.items():
        path = os.path.join(OUT, name + ".mmd")
        with open(path, "w", encoding="utf-8") as f:
            f.write(body.rstrip() + "\n" + PALETTE)
        print("wrote", name + ".mmd")
    print("total:", len(DIAGRAMS), "diagrams")

if __name__ == "__main__":
    main()


Writing images/make_diagrams.py


In [30]:
%%writefile images/plot_artifacts.py
"""
Plot the real training artifacts produced by the GPU run into the README images:
  - images/loss_curve.png       from the pretraining stdout (step / train loss / dev loss)
  - images/gsm8k_accuracy.png   from the across-stages GSM8K table (stage_table.jsonl)

Usage:
    python images/plot_artifacts.py <artifacts_dir>
where <artifacts_dir> holds pretrain_stdout.txt and stage_table.jsonl.
"""
import json, os, re, sys
import matplotlib as mpl
mpl.use("Agg")
import matplotlib.pyplot as plt

OUT = os.path.dirname(os.path.abspath(__file__))
ART = sys.argv[1] if len(sys.argv) > 1 else OUT

mpl.rcParams["font.family"] = ["Comic Sans MS", "Ink Free", "DejaVu Sans"]
BLUE, ORANGE, PURPLE = "#1565c0", "#e8730c", "#7b3fbf"

def plot_loss():
    path = os.path.join(ART, "pretrain_stdout.txt")
    if not os.path.exists(path):
        print("no pretrain_stdout.txt, skipping loss curve"); return
    txt = open(path, encoding="utf-8", errors="ignore").read()
    train = [(int(s), float(l)) for s, l in re.findall(r"^step (\d+) \| loss ([\d.]+)", txt, re.M)]
    ev = re.findall(r"\[eval\] step (\d+) \| train ([\d.]+) \| dev ([\d.]+)", txt)
    ev = [(int(s), float(a), float(b)) for s, a, b in ev]
    if not train:
        print("no step lines found, skipping loss curve"); return
    fig, ax = plt.subplots(figsize=(9, 5.2))
    xs, ys = zip(*train)
    ax.plot(xs, ys, color=BLUE, lw=2.2, label="train loss")
    if ev:
        es, et, ed = zip(*ev)
        ax.plot(es, ed, color=ORANGE, lw=2.2, marker="o", ms=5, label="dev loss")
    ax.axhline(10.83, color="#999", ls="--", lw=1.2, label="uniform-guess loss  ln(50304)")
    ax.set_xlabel("training step"); ax.set_ylabel("cross-entropy loss")
    ax.set_title("Pretraining a 77M base on The Pile (2x L40)", fontsize=15)
    ax.grid(True, alpha=0.25); ax.legend()
    fig.tight_layout(); fig.savefig(os.path.join(OUT, "loss_curve.png"), dpi=150, facecolor="white")
    print("wrote loss_curve.png  (last train=%.3f%s)" % (ys[-1], (", dev=%.3f" % ed[-1]) if ev else ""))

def plot_gsm8k():
    path = os.path.join(ART, "stage_table.jsonl")
    if not os.path.exists(path):
        print("no stage_table.jsonl, skipping accuracy chart"); return
    rows = [json.loads(l) for l in open(path) if l.strip()]
    order = ["base_pretrained", "sft", "dpo", "ppo", "grpo"]
    rows.sort(key=lambda r: order.index(r["label"]) if r["label"] in order else 99)
    labels = [r["label"].replace("base_pretrained", "base") for r in rows]
    accs = [100.0 * r["accuracy"] for r in rows]
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(labels, accs, color=[BLUE, "#d48806", "#e67e22", "#c0392b", PURPLE][:len(labels)],
                  edgecolor="#333", linewidth=1.5)
    for b, a in zip(bars, accs):
        ax.text(b.get_x()+b.get_width()/2, a, f"{a:.1f}%", ha="center", va="bottom", fontsize=11)
    ax.set_ylabel("GSM8K accuracy (%)")
    ax.set_title("GSM8K accuracy across stages (77M, 2x L40)", fontsize=15)
    ax.grid(True, axis="y", alpha=0.25)
    fig.tight_layout(); fig.savefig(os.path.join(OUT, "gsm8k_accuracy.png"), dpi=150, facecolor="white")
    print("wrote gsm8k_accuracy.png ", dict(zip(labels, [round(a,1) for a in accs])))

if __name__ == "__main__":
    plot_loss()
    plot_gsm8k()


Writing images/plot_artifacts.py


In [31]:
%%writefile mkdocs.yml
site_name: Train LLM From Scratch
site_description: From pretraining to RLHF/GRPO — every algorithm hand-written in pure PyTorch.
site_url: https://FareedKhan-dev.github.io/train-llm-from-scratch/
repo_url: https://github.com/FareedKhan-dev/train-llm-from-scratch
repo_name: FareedKhan-dev/train-llm-from-scratch
docs_dir: docs
# Files present under docs/ but intentionally not page nodes in the nav.
not_in_nav: |
  /diagrams/README.md
# Keep the raw mermaid sources out of the built site.
exclude_docs: |
  /diagrams/src/

theme:
  name: material
  icon:
    repo: fontawesome/brands/github
  palette:
    - media: "(prefers-color-scheme: light)"
      scheme: default
      primary: teal
      accent: teal
      toggle: {icon: material/weather-night, name: Switch to dark mode}
    - media: "(prefers-color-scheme: dark)"
      scheme: slate
      primary: teal
      accent: teal
      toggle: {icon: material/weather-sunny, name: Switch to light mode}
  features:
    - navigation.tabs
    - navigation.sections
    - navigation.top
    - navigation.indexes
    - toc.follow
    - search.suggest
    - search.highlight
    - content.code.copy
    - content.code.annotate

plugins:
  - search

markdown_extensions:
  - admonition
  - attr_list
  - md_in_html
  - footnotes
  - toc:
      permalink: true
  - pymdownx.highlight:
      anchor_linenums: true
  - pymdownx.inlinehilite
  - pymdownx.arithmatex:
      generic: true
  - pymdownx.snippets:
      base_path: ["."]          # so howto/commands.md can include the root POST_TRAINING.md
  - pymdownx.superfences:
      custom_fences:
        - name: mermaid
          class: mermaid
          format: !!python/name:pymdownx.superfences.fence_code_format
  - pymdownx.tabbed:
      alternate_style: true

extra_javascript:
  - https://unpkg.com/mermaid@11/dist/mermaid.min.js   # live hand-drawn mermaid in <details>
  - javascripts/mathjax.js
  - https://unpkg.com/mathjax@3/es5/tex-mml-chtml.js

nav:
  - Home: README.md
  - Foundations:
      - Overview: foundations/README.md
      - Tokenization & Data Shapes: foundations/tokenization.md
      - Decoder-Only Transformer: foundations/transformer.md
      - Attention, Masks & Heads: foundations/attention.md
      - Objectives, Losses & Perplexity: foundations/objectives.md
      - Optimization & Training Systems: foundations/optimization.md
      - Generation & Sampling: foundations/generation.md
  - Theory & Pipeline:
      - Data handling: 01_data_pipeline.md
      - Pretraining: 02_pretraining.md
      - SFT: 03_sft.md
      - Reward Model: 04_reward_model.md
      - DPO / ORPO / KTO: 05_dpo.md
      - PPO: 06_ppo.md
      - GRPO / RLVR: 07_grpo.md
  - How-to:
      - Configure (JSON): howto/configs.md
      - Train (UI & CLI): howto/train.md
      - The control-panel UI: howto/ui.md
      - Evaluate: 08_evaluation.md
      - Inference / Chat: 09_inference.md
  - Reference:
      - Command cheatsheet: howto/commands.md


Writing mkdocs.yml


In [32]:
%%writefile pyproject.toml
[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.build_meta"

[project]
name = "train-llm-from-scratch"
version = "0.1.0"
description = "Train an LLM from scratch in pure PyTorch — pretraining through SFT, Reward Modeling, PPO, DPO and GRPO/RLVR."
readme = "README.md"
requires-python = ">=3.9"
license = { text = "MIT" }
# Core deps for the model + data path. (The teaching pretraining path also has
# requirements.txt with cu118 wheels; this project install does not pin a CUDA build —
# install the right torch wheel for your machine first if needed.)
dependencies = [
    "torch",
    "numpy",
    "h5py",
    "tqdm",
    "tiktoken",
    "zstandard",
    "requests",
]

[project.optional-dependencies]
# Post-training extras (datasets + optional experiment logging).
train = ["datasets", "wandb"]
# The Streamlit control-panel UI.
ui = ["streamlit", "pandas", "altair"]
# The MkDocs Material documentation site.
docs = ["mkdocs", "mkdocs-material", "pymdown-extensions"]
# Everything, for development.
all = ["datasets", "wandb", "streamlit", "pandas", "altair", "mkdocs", "mkdocs-material", "pymdown-extensions"]

# Make the existing import roots installable so `pip install -e .` removes the need for
# `PYTHONPATH=.`. These packages already exist in the repo (additive __init__.py only).
[tool.setuptools]
packages = ["config", "data_loader", "src", "src.models", "src.post_training", "src.post_training.rewards", "ui"]

[tool.setuptools.package-data]
# ship the JSON configs alongside the package
"*" = ["*.json"]


Writing pyproject.toml


In [33]:
%%writefile requirements-post.txt
# Post-training stack for the from-scratch SFT / RM / PPO / DPO / GRPO suite.
# Targets H100 + CUDA 12.x (the base requirements.txt pins cu118 for the original
# pretraining path; this file does NOT overwrite it). Install into a fresh venv:
#
#   python3 -m venv /ephemeral/venv && source /ephemeral/venv/bin/activate
#   pip install -r requirements-post.txt
#
# torchrun (multi-GPU) ships with torch.

--extra-index-url https://download.pytorch.org/whl/cu121

torch
numpy
h5py
tqdm
tiktoken
zstandard
datasets          # HuggingFace dataset downloads (Alpaca, HH-RLHF, GSM8K, ...)
wandb             # optional experiment logging (every script also writes JSONL)


Writing requirements-post.txt


In [34]:
%%writefile requirements.txt
--extra-index-url https://download.pytorch.org/whl/cu118
torch
torchvision
torchaudio
numpy
h5py
requests
tqdm
zstandard
tiktoken


Writing requirements.txt


In [35]:
%%writefile scripts/chat.py
"""
Chat / inference CLI for any stage checkpoint (base, SFT, DPO, PPO, GRPO).

Model dimensions are read from the checkpoint, so you only pass the path. Use the chat
template for instruction-tuned models, or --raw for base-model continuation.

One-shot:
    PYTHONPATH=. python scripts/chat.py --ckpt /ephemeral/ckpts/sft.pt --prompt "What is 13 + 29?"
    PYTHONPATH=. python scripts/chat.py --ckpt /ephemeral/ckpts/grpo.pt --prompt "..." --greedy
    PYTHONPATH=. python scripts/chat.py --ckpt /ephemeral/ckpts/base_pretrained.pt --raw --prompt "Once upon a time"
Interactive REPL (no --prompt):
    PYTHONPATH=. python scripts/chat.py --ckpt /ephemeral/ckpts/sft.pt
"""

from __future__ import annotations

import argparse

import torch

from src.post_training.inference import generate_reply, load_model_from_ckpt


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--ckpt", required=True)
    p.add_argument("--prompt", default=None, help="one-shot prompt; omit for interactive REPL")
    p.add_argument("--system", default=None, help="optional system message (chat mode)")
    p.add_argument("--raw", action="store_true", help="base-model continuation (no chat template)")
    p.add_argument("--max_new_tokens", type=int, default=256)
    p.add_argument("--temperature", type=float, default=0.8)
    p.add_argument("--top_p", type=float, default=0.95)
    p.add_argument("--top_k", type=int, default=None)
    p.add_argument("--greedy", action="store_true", help="deterministic argmax decoding")
    p.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    args = p.parse_args()

    model = load_model_from_ckpt(args.ckpt, args.device)
    n = sum(p.numel() for p in model.parameters())
    print(f"loaded {args.ckpt} ({n/1e6:.0f}M params) on {args.device} | "
          f"mode={'raw' if args.raw else 'chat'} {'greedy' if args.greedy else f'T={args.temperature} top_p={args.top_p}'}")

    def reply(text):
        return generate_reply(model, text, device=args.device, system=args.system, raw=args.raw,
                              max_new_tokens=args.max_new_tokens, temperature=args.temperature,
                              top_p=args.top_p if args.top_p < 1 else None, top_k=args.top_k, greedy=args.greedy)

    if args.prompt is not None:
        print(reply(args.prompt))
        return

    print("Interactive chat (Ctrl-D / 'exit' to quit).")
    while True:
        try:
            text = input("\nyou> ").strip()
        except EOFError:
            break
        if text in ("exit", "quit"):
            break
        if text:
            print("bot>", reply(text))


if __name__ == "__main__":
    main()


Writing scripts/chat.py


In [36]:
%%writefile scripts/data_download.py
import os
import argparse
import requests
from tqdm import tqdm
from typing import List

# Base URL for the dataset files
BASE_URL = "https://huggingface.co/datasets/monology/pile-uncopyrighted/resolve/main"
VAL_URL = f"{BASE_URL}/val.jsonl.zst"  # URL for the validation dataset
TRAIN_URLS = [f"{BASE_URL}/train/{i:02d}.jsonl.zst" for i in range(65)]  # URLs for 65 training files (adjust the range if needed)

def download_file(url: str, file_name: str) -> None:
    """
    Downloads a file from the given URL and saves it with the specified file name.
    Displays a progress bar using tqdm.

    Args:
        url (str): The URL of the file to download.
        file_name (str): The local path where the file will be saved.
    """
    print(f"Downloading: {file_name}...")
    response = requests.get(url, stream=True)  # Stream the file content
    total_size = int(response.headers.get('content-length', 0))  # Get total file size if available
    block_size = 1024  # Size of each block for the progress bar
    with open(file_name, 'wb') as f:  # Open file for writing in binary mode
        for chunk in tqdm(response.iter_content(block_size), total=total_size // block_size, desc="Downloading", leave=True):
            f.write(chunk)  # Write each chunk to the file

def download_dataset(val_url: str, train_urls: List[str], val_dir: str, train_dir: str, max_train_files: int) -> None:
    """
    Manages downloading of the dataset, including both validation and training files.

    Args:
        val_url (str): URL for the validation dataset.
        train_urls (list): List of URLs for the training dataset files.
        val_dir (str): Directory where the validation file will be stored.
        train_dir (str): Directory where the training files will be stored.
        max_train_files (int): Maximum number of training files to download.
    """
    # Define the path for the validation file
    val_file_path = os.path.join(val_dir, "val.jsonl.zst")
    if not os.path.exists(val_file_path):  # Check if the validation file already exists
        print(f"Validation file not found. Downloading from {val_url}...")
        download_file(val_url, val_file_path)  # Download the validation file
    else:
        print("Validation data already present. Skipping download.")

    # Loop through the training file URLs and download if not already present
    for idx, url in enumerate(train_urls[:max_train_files]):  # Limit to max_train_files
        file_name = f"{idx:02d}.jsonl.zst"  # Format file name (e.g., 00.jsonl.zst)
        file_path = os.path.join(train_dir, file_name)  # Construct the full file path
        if not os.path.exists(file_path):  # Check if the file already exists
            print(f"Training file {file_name} not found. Downloading...")
            download_file(url, file_path)  # Download the training file
        else:
            print(f"Training file {file_name} already present. Skipping download.")

def main() -> None:
    """
    Main function to parse arguments and orchestrate the dataset download process.
    """
    # Parse command-line arguments using argparse
    parser = argparse.ArgumentParser(description="Download PILE dataset.")  # Description of the script
    parser.add_argument('--train_max', type=int, default=1, help="Max number of training files to download.")  # Max training files
    parser.add_argument('--train_dir', default="data/train", help="Directory for storing training data.")  # Training directory
    parser.add_argument('--val_dir', default="data/val", help="Directory for storing validation data.")  # Validation directory

    args = parser.parse_args()  # Parse the arguments provided by the user

    # Ensure directories for training and validation data exist
    os.makedirs(args.train_dir, exist_ok=True)  # Create training directory if it doesn't exist
    os.makedirs(args.val_dir, exist_ok=True)  # Create validation directory if it doesn't exist

    # Start downloading the dataset
    download_dataset(VAL_URL, TRAIN_URLS, args.val_dir, args.train_dir, args.train_max)

    print("Dataset downloaded successfully.")  # Indicate successful download

if __name__ == "__main__":
    # Entry point of the script
    main()


Writing scripts/data_download.py


In [37]:
%%writefile scripts/data_preprocess.py
import os
import json
import zstandard as zstd
import tiktoken
import h5py
from tqdm import tqdm
import argparse
from typing import Optional

def process_files(input_dir: str, output_file: str, tokenizer_name: str, max_data: Optional[int] = None) -> None:
    """
    Process a specified number of lines from each .jsonl.zst file in the input directory
    and save encoded tokens to an HDF5 file.

    Args:
        input_dir (str): Directory containing input .jsonl.zst files.
        output_file (str): Path to the output HDF5 file.
        tokenizer_name (str): Name of the tiktoken tokenizer to use (e.g., 'r50k_base').
        max_data (int, optional): Maximum number of lines to process from each file.
                                  If None, process all lines.
    """
    # Print processing strategy based on max_data
    if max_data is not None:
        print(f"You have chosen max_data = {max_data}. Processing only the top {max_data} JSON objects from each file.")
    else:
        print("Processing all available JSON objects from each file.")

    # Load the tokenizer using the provided tokenizer name
    enc = tiktoken.get_encoding(tokenizer_name)

    # Create an HDF5 file for output
    with h5py.File(output_file, 'w') as out_f:
        # Initialize the dataset for storing tokenized data
        dataset = out_f.create_dataset('tokens', (0,), maxshape=(None,), dtype='i')
        start_index = 0  # Track the starting index for the next batch of tokens

        # Process each .jsonl.zst file in the input directory
        for filename in sorted(os.listdir(input_dir)):
            if filename.endswith(".jsonl.zst"):  # Only process .jsonl.zst files
                in_file = os.path.join(input_dir, filename)
                print(f"Processing: {in_file}")

                processed_lines = 0  # Counter for processed lines in the current file

                # Open the compressed .jsonl.zst file for reading
                with zstd.open(in_file, 'rt', encoding='utf-8') as in_f:
                    # Iterate over each line in the file
                    for line in tqdm(in_f, desc=f"Processing {filename}", total=max_data if max_data is not None else None):
                        try:
                            # Parse the line as JSON
                            data = json.loads(line)
                            text = data.get('text')  # Extract the 'text' field from the JSON object

                            if text:
                                # Tokenize the text and append an end-of-text token
                                encoded = enc.encode(text + "<|endoftext|>", allowed_special={'<|endoftext|>'})
                                encoded_len = len(encoded)

                                # Resize the dataset to accommodate new tokens
                                end_index = start_index + encoded_len
                                dataset.resize(dataset.shape[0] + encoded_len, axis=0)

                                # Store the encoded tokens in the dataset
                                dataset[start_index:end_index] = encoded
                                start_index = end_index  # Update the start index
                            else:
                                # Warn if 'text' key is missing in the JSON object
                                print(f"Warning: 'text' key missing in line from {filename}")
                        except json.JSONDecodeError:
                            # Handle JSON decoding errors
                            print(f"Warning: Could not decode JSON from line in {filename}")
                        except Exception as e:
                            # Handle any other errors
                            print(f"An error occurred while processing line in {filename}: {e}")

                        processed_lines += 1
                        # Stop processing if max_data limit is reached
                        if max_data is not None and processed_lines >= max_data:
                            break

def main():
    """
    Main function to parse arguments, validate directories, and process files.
    """
    # Parse command-line arguments
    parser = argparse.ArgumentParser(description="Preprocess PILE dataset files and save tokens to HDF5.")
    parser.add_argument("--train_dir", type=str, default="data/train", help="Directory containing training .jsonl.zst files.")
    parser.add_argument("--val_dir", type=str, default="data/val", help="Directory containing validation .jsonl.zst files.")
    parser.add_argument("--out_train_file", type=str, default="data/train/pile_train.h5", help="Path to the output training HDF5 file.")
    parser.add_argument("--out_val_file", type=str, default="data/val/pile_dev.h5", help="Path to the output validation HDF5 file.")
    parser.add_argument("--tokenizer_name", type=str, default="r50k_base", help="Name of the tiktoken tokenizer to use.")
    parser.add_argument("--max_data", type=int, default=1000, help="Maximum number of json objects to process from each file in both train and val datasets (default: 1000).")

    args = parser.parse_args()

    # Validate the existence of the training and validation directories
    if not os.path.isdir(args.train_dir):
        print(f"Error: Training directory not found: {args.train_dir}")
        return
    if not os.path.isdir(args.val_dir):
        print(f"Error: Validation directory not found: {args.val_dir}")
        return

    # Process training data
    print("Starting training data preprocessing...")
    process_files(args.train_dir, args.out_train_file, args.tokenizer_name, args.max_data)
    print("Training data preprocessing complete.")

    # Process validation data
    print("Starting validation data preprocessing...")
    process_files(args.val_dir, args.out_val_file, args.tokenizer_name, args.max_data)
    print("Validation data preprocessing complete.")

# Entry point of the script
if __name__ == "__main__":
    main()


Writing scripts/data_preprocess.py


In [38]:
%%writefile scripts/eval_post_training.py
"""
Evaluate any stage checkpoint on GSM8K (greedy) and optionally dump sample generations.
Use it to build the headline "GSM8K accuracy across stages" table:

    for s in base_pretrained sft dpo ppo grpo; do
      PYTHONPATH=. python scripts/eval_post_training.py --ckpt /ephemeral/ckpts/$s.pt \
        --label $s --limit 200 --append /ephemeral/logs/stage_table.jsonl
    done
    PYTHONPATH=. python scripts/eval_post_training.py --table /ephemeral/logs/stage_table.jsonl

Model dimensions are read from the checkpoint's stored ``cfg`` so you don't have to repeat
them. Reward checkpoints (which have a reward head, not an LM head only) still load because
we keep just the backbone keys for generation.
"""

from __future__ import annotations

import argparse
import json
import os

import torch

from src.models.transformer import Transformer
from src.post_training.evaluation import gsm8k_accuracy, load_gsm8k_eval


def model_from_ckpt(ckpt_path: str, device: str, overrides: dict | None = None) -> Transformer:
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    cfg = ck.get("cfg", {}) or {}
    cfg = {**cfg, **(overrides or {})}
    model = Transformer(
        n_head=cfg.get("n_head", 16), n_embed=cfg.get("n_embed", 1024),
        context_length=cfg.get("context_length", 1024), vocab_size=cfg.get("vocab_size", 50304),
        N_BLOCKS=cfg.get("n_blocks", 24),
    )
    state = ck["model_state_dict"] if "model_state_dict" in ck else ck
    state = {k.removeprefix("module.").removeprefix("transformer."): v for k, v in state.items()}
    backbone_keys = set(model.state_dict().keys())
    filtered = {k: v for k, v in state.items() if k in backbone_keys}
    model.load_state_dict(filtered, strict=False)
    return model.to(device).eval()


def print_table(path: str):
    rows = [json.loads(l) for l in open(path) if l.strip()]
    print(f"\n{'stage':<18}{'GSM8K acc':>10}{'n':>8}")
    print("-" * 36)
    for r in rows:
        print(f"{r['label']:<18}{r['accuracy']*100:>9.1f}%{r['n']:>8}")


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--ckpt")
    p.add_argument("--label", default="model")
    p.add_argument("--limit", type=int, default=200)
    p.add_argument("--split", default="test")
    p.add_argument("--max_new_tokens", type=int, default=300)
    p.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    p.add_argument("--samples", type=int, default=3)
    p.add_argument("--append", default=None, help="append the result row to this JSONL")
    p.add_argument("--table", default=None, help="just print a stage table from this JSONL and exit")
    args = p.parse_args()

    if args.table:
        print_table(args.table)
        return

    model = model_from_ckpt(args.ckpt, args.device)
    qa = load_gsm8k_eval(args.split, limit=args.limit)
    res = gsm8k_accuracy(model, qa, device=args.device, max_new_tokens=args.max_new_tokens,
                         greedy=True, return_samples=args.samples)
    print(f"[{args.label}] GSM8K {args.split} accuracy: {res['accuracy']*100:.1f}%  ({res['correct']}/{res['n']})")
    for s in res["samples"]:
        print(f"\n  Q: {s['q'][:120]}\n  gold={s['gold']} correct={s['correct']}\n  A: {s['response'][:300]}")

    if args.append:
        os.makedirs(os.path.dirname(args.append) or ".", exist_ok=True)
        with open(args.append, "a") as f:
            f.write(json.dumps({"label": args.label, "accuracy": res["accuracy"],
                                "correct": res["correct"], "n": res["n"]}) + "\n")
        print(f"\nappended -> {args.append}")


if __name__ == "__main__":
    main()


Writing scripts/eval_post_training.py


In [39]:
%%writefile scripts/generate_text.py
import torch
import tiktoken
import argparse
from config.config import default_config as config
from src.models.transformer import Transformer  # Assuming your Transformer class is in this module


def load_checkpoint(model_path: str, device: str):
    try:
        return torch.load(model_path, map_location=torch.device(device), weights_only=False)
    except TypeError:
        return torch.load(model_path, map_location=torch.device(device))

def generate_text(model_path: str, input_text: str, max_new_tokens: int = 100, device: str = 'cuda') -> str:
    """
    Generates text using a pre-trained Transformer model.

    Args:
        model_path (str): Path to the saved model checkpoint.
        input_text (str): The initial text to start generation from.
        max_new_tokens (int): The maximum number of new tokens to generate.
        device (str): 'cuda' or 'cpu', the device to run the model on.

    Returns:
        str: The generated text.
    """
    # Load the model checkpoint
    checkpoint = load_checkpoint(model_path, device)

    # Initialize the model using the configuration from config.py
    model = Transformer(
        n_head=config['n_head'],
        n_embed=config['n_embed'],
        context_length=config['context_length'],
        vocab_size=config['vocab_size'],
        N_BLOCKS=config['n_blocks']
    )
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval().to(device)

    # Load the tokenizer
    enc = tiktoken.get_encoding("r50k_base")

    start_ids = enc.encode_ordinary(input_text)
    context = torch.tensor(start_ids, dtype=torch.long, device=device).unsqueeze(0)

    # Generation process
    with torch.no_grad():
        generated_tokens = model.generate(context, max_new_tokens=max_new_tokens)[0].tolist()

    # Decode the generated tokens
    output_text = enc.decode(generated_tokens)

    return output_text

def main() -> None:
    parser = argparse.ArgumentParser(description="Generate text using a pre-trained Transformer model.")
    parser.add_argument('--model_path', type=str, help='Path to the saved model checkpoint.')
    parser.add_argument('--input_text', type=str, help='The initial text to start generation from.')
    parser.add_argument('--max_new_tokens', type=int, default=100, help='Maximum number of new tokens to generate.')

    args = parser.parse_args()

    generated = generate_text(args.model_path, args.input_text, args.max_new_tokens, config['device'])
    print(f"Generated text:\n{generated}")

if __name__ == "__main__":
    main()


Writing scripts/generate_text.py


In [40]:
%%writefile scripts/prepare_preference_data.py
"""
Build preference data for the reward model and DPO from real public datasets:
  - Anthropic/hh-rlhf                       (helpful/harmless human preferences)
  - HuggingFaceH4/ultrafeedback_binarized   (LLM-judged preference pairs)

Emits JSONL of ``{"prompt", "chosen", "rejected"}`` (train + held-out test). The held-out
test split is what reward-model preference accuracy is measured on.

Example:
    PYTHONPATH=. HF_HOME=/ephemeral/hf_cache python scripts/prepare_preference_data.py \
        --source both --max_per_source 40000 --out_dir /ephemeral/data
"""

from __future__ import annotations

import argparse
import json
import os

os.environ.setdefault("HF_HOME", "/ephemeral/hf_cache")

_ASSISTANT_MARKER = "\n\nAssistant:"


def _split_hh(text: str) -> tuple[str, str] | None:
    """Split an HH-RLHF conversation string into (prompt_context, final_response)."""
    idx = text.rfind(_ASSISTANT_MARKER)
    if idx == -1:
        return None
    prompt = text[:idx].strip()
    response = text[idx + len(_ASSISTANT_MARKER):].strip()
    if not prompt or not response:
        return None
    return prompt, response


def from_hh(max_n: int, split: str) -> list[dict]:
    from datasets import load_dataset

    ds = load_dataset("Anthropic/hh-rlhf", split=split)
    if max_n:
        ds = ds.select(range(min(max_n, len(ds))))
    out = []
    for ex in ds:
        c = _split_hh(ex["chosen"]); r = _split_hh(ex["rejected"])
        if not c or not r:
            continue
        prompt, chosen = c
        _, rejected = r
        if chosen == rejected:
            continue
        out.append({"prompt": prompt, "chosen": chosen, "rejected": rejected})
    return out


def from_ultrafeedback(max_n: int, split: str) -> list[dict]:
    from datasets import load_dataset

    hf_split = "train_prefs" if split == "train" else "test_prefs"
    ds = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split=hf_split)
    if max_n:
        ds = ds.select(range(min(max_n, len(ds))))
    out = []
    for ex in ds:
        prompt = ex["prompt"].strip()
        chosen = ex["chosen"][-1]["content"].strip()
        rejected = ex["rejected"][-1]["content"].strip()
        if not prompt or not chosen or not rejected or chosen == rejected:
            continue
        out.append({"prompt": prompt, "chosen": chosen, "rejected": rejected})
    return out


def collect(source: str, max_n: int, split: str) -> list[dict]:
    rows: list[dict] = []
    if source in ("hh", "both"):
        print(f"Loading Anthropic/hh-rlhf [{split}] ...")
        rows += from_hh(max_n, split)
    if source in ("ultrafeedback", "both"):
        print(f"Loading ultrafeedback_binarized [{split}] ...")
        try:
            rows += from_ultrafeedback(max_n, split)
        except Exception as e:  # noqa: BLE001
            print(f"  (skipping ultrafeedback: {e})")
    return rows


def write_jsonl(rows: list[dict], path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
    print(f"  wrote {len(rows)} pairs -> {path}")


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--source", choices=["hh", "ultrafeedback", "both"], default="both")
    p.add_argument("--max_per_source", type=int, default=40000)
    p.add_argument("--out_dir", default="/ephemeral/data")
    args = p.parse_args()

    train = collect(args.source, args.max_per_source, "train")
    test = collect(args.source, max(2000, args.max_per_source // 20), "test")
    import random
    random.Random(0).shuffle(train)
    write_jsonl(train, os.path.join(args.out_dir, "preferences.jsonl"))
    write_jsonl(test, os.path.join(args.out_dir, "preferences_test.jsonl"))


if __name__ == "__main__":
    main()


Writing scripts/prepare_preference_data.py


In [41]:
%%writefile scripts/prepare_pretrain_data.py
"""
Download + tokenize Pile-uncopyrighted shards into a single flat-token HDF5 for
pretraining. Faster than the original ``data_preprocess.py`` (which resizes the HDF5 per
document): here we stream-decompress, batch-tokenize with tiktoken, and write tokens to
the HDF5 in large chunks.

Writes to /ephemeral by default (the 1.5TB disk).

Examples:
    # dev split from the Pile validation file
    PYTHONPATH=. python scripts/prepare_pretrain_data.py --split val \
        --out /ephemeral/data/pile_dev.h5
    # one training shard
    PYTHONPATH=. python scripts/prepare_pretrain_data.py --split train --num_shards 1 \
        --out /ephemeral/data/pile_train.h5
"""

from __future__ import annotations

import argparse
import io
import json
import os

import h5py
import numpy as np
import requests
import tiktoken
import zstandard as zstd
from tqdm import tqdm

BASE_URL = "https://huggingface.co/datasets/monology/pile-uncopyrighted/resolve/main"
EOT_ID = 50256
WRITE_CHUNK = 8_000_000  # flush tokens to HDF5 in ~8M-token chunks
ENC_BATCH = 1024          # documents per tiktoken batch-encode


def shard_urls(split: str, num_shards: int) -> list[str]:
    if split == "val":
        return [f"{BASE_URL}/val.jsonl.zst"]
    return [f"{BASE_URL}/train/{i:02d}.jsonl.zst" for i in range(num_shards)]


def download(url: str, dest: str) -> str:
    if os.path.exists(dest):
        print(f"  cached: {dest}")
        return dest
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    print(f"  downloading {url}")
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        with open(dest + ".part", "wb") as f:
            for chunk in tqdm(r.iter_content(1 << 20), total=total >> 20, unit="MB", desc="dl"):
                f.write(chunk)
    os.replace(dest + ".part", dest)
    return dest


def iter_texts(zst_path: str):
    dctx = zstd.ZstdDecompressor()
    with open(zst_path, "rb") as fh:
        reader = dctx.stream_reader(fh)
        for line in io.TextIOWrapper(reader, encoding="utf-8"):
            line = line.strip()
            if not line:
                continue
            try:
                txt = json.loads(line).get("text")
            except json.JSONDecodeError:
                continue
            if txt:
                yield txt


def tokenize_to_h5(zst_paths: list[str], out_path: str, max_tokens: int | None) -> int:
    enc = tiktoken.get_encoding("r50k_base")
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    total = 0
    buf: list[int] = []
    with h5py.File(out_path, "w") as f:
        dset = f.create_dataset("tokens", (0,), maxshape=(None,), dtype="i4", chunks=(WRITE_CHUNK,))

        def flush():
            nonlocal total, buf
            if not buf:
                return
            arr = np.asarray(buf, dtype=np.int32)
            dset.resize(total + arr.size, axis=0)
            dset[total: total + arr.size] = arr
            total += arr.size
            buf = []

        for zp in zst_paths:
            print(f"  tokenizing {zp}")
            docs: list[str] = []
            pbar = tqdm(iter_texts(zp), unit="doc", desc="tok")
            for txt in pbar:
                docs.append(txt)
                if len(docs) >= ENC_BATCH:
                    for ids in enc.encode_ordinary_batch(docs):
                        buf.extend(ids)
                        buf.append(EOT_ID)
                    docs = []
                    if len(buf) >= WRITE_CHUNK:
                        flush()
                        pbar.set_postfix(tokens=f"{total/1e6:.1f}M")
                    if max_tokens and total >= max_tokens:
                        break
            if docs:
                for ids in enc.encode_ordinary_batch(docs):
                    buf.extend(ids)
                    buf.append(EOT_ID)
            flush()
            if max_tokens and total >= max_tokens:
                break
    print(f"  wrote {total:,} tokens -> {out_path}")
    return total


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--split", choices=["train", "val"], required=True)
    p.add_argument("--num_shards", type=int, default=1, help="train shards to use")
    p.add_argument("--raw_dir", default="/ephemeral/data/pile_raw")
    p.add_argument("--out", required=True)
    p.add_argument("--max_tokens", type=int, default=None, help="stop after this many tokens")
    args = p.parse_args()

    urls = shard_urls(args.split, args.num_shards)
    local = []
    for u in urls:
        name = "val.jsonl.zst" if args.split == "val" else os.path.basename(u)
        local.append(download(u, os.path.join(args.raw_dir, name)))
    tokenize_to_h5(local, args.out, args.max_tokens)


if __name__ == "__main__":
    main()


Writing scripts/prepare_pretrain_data.py


In [42]:
%%writefile scripts/prepare_rl_prompts.py
"""
Build RL prompt sets ({"prompt", "gold"}) for PPO/GRPO:
  - GSM8K train -> rl_prompts_train.jsonl   (the RL training prompts)
  - GSM8K test  -> rl_prompts_test.jsonl    (held-out benchmark for eval)
  - a programmatic arithmetic set -> arithmetic_prompts.jsonl  (RL curriculum warm-up,
    where even a weak model gets some non-zero reward so RL has signal to start from)

Example:
    PYTHONPATH=. HF_HOME=/ephemeral/hf_cache python scripts/prepare_rl_prompts.py --out_dir /ephemeral/data
"""

from __future__ import annotations

import argparse
import json
import os
import random

os.environ.setdefault("HF_HOME", "/ephemeral/hf_cache")

from src.post_training.rewards import gsm8k_gold_answer


def write_jsonl(rows, path):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
    print(f"  wrote {len(rows)} prompts -> {path}")


def gsm8k_prompts(split: str, limit: int | None):
    from datasets import load_dataset

    ds = load_dataset("openai/gsm8k", "main", split=split)
    if limit:
        ds = ds.select(range(min(limit, len(ds))))
    rows = []
    for ex in ds:
        gold = gsm8k_gold_answer(ex["answer"])
        if gold is not None:
            rows.append({"prompt": ex["question"].strip(), "gold": gold})
    return rows


def arithmetic_prompts(n: int, max_val: int, seed: int):
    rng = random.Random(seed)
    ops = [("+", lambda a, b: a + b), ("-", lambda a, b: a - b), ("*", lambda a, b: a * b)]
    rows = []
    for _ in range(n):
        a, b = rng.randint(0, max_val), rng.randint(0, max_val)
        sym, fn = rng.choice(ops)
        rows.append({"prompt": f"What is {a} {sym} {b}?", "gold": float(fn(a, b))})
    return rows


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--out_dir", default="/ephemeral/data")
    p.add_argument("--train_limit", type=int, default=None)
    p.add_argument("--test_limit", type=int, default=500)
    p.add_argument("--arith_n", type=int, default=5000)
    p.add_argument("--arith_max", type=int, default=20)
    args = p.parse_args()

    print("Loading GSM8K train ...")
    write_jsonl(gsm8k_prompts("train", args.train_limit), os.path.join(args.out_dir, "rl_prompts_train.jsonl"))
    print("Loading GSM8K test ...")
    write_jsonl(gsm8k_prompts("test", args.test_limit), os.path.join(args.out_dir, "rl_prompts_test.jsonl"))
    print("Generating arithmetic curriculum ...")
    write_jsonl(arithmetic_prompts(args.arith_n, args.arith_max, seed=0),
                os.path.join(args.out_dir, "arithmetic_prompts.jsonl"))


if __name__ == "__main__":
    main()


Writing scripts/prepare_rl_prompts.py


In [43]:
%%writefile scripts/prepare_sft_data.py
"""
Build the SFT dataset from real public instruction data + GSM8K, render it through the
chat template (masking prompt tokens), pack to fixed-length rows, and write a packed
HDF5 (``tokens`` + ``loss_mask``) plus a held-out dev file.

Datasets (downloaded via HuggingFace ``datasets`` to /ephemeral/hf_cache):
  - tatsu-lab/alpaca                (general instruction following)
  - databricks/databricks-dolly-15k (general instruction following)
  - openai/gsm8k (main, train)      (math, reformatted into <think>/<answer> so the model
                                     learns the reasoning format the RL verifier rewards)

Example:
    PYTHONPATH=. HF_HOME=/ephemeral/hf_cache python scripts/prepare_sft_data.py \
        --context_length 1024 --out_dir /ephemeral/data
"""

from __future__ import annotations

import argparse
import os
import re

import h5py
import numpy as np

from src.post_training.chat_template import encode_chat, ANSWER_OPEN, ANSWER_CLOSE, THINK_OPEN, THINK_CLOSE
from src.post_training.sft import pack_examples

os.environ.setdefault("HF_HOME", "/ephemeral/hf_cache")
_CALC_RE = re.compile(r"<<[^>]*>>")          # GSM8K calculator annotations
_HASH_RE = re.compile(r"####\s*(.+)\s*$")


def gsm8k_to_messages(question: str, answer: str) -> list[dict]:
    """Reformat a GSM8K (question, answer) into chat messages whose assistant turn uses
    the <think>...</think><answer>N</answer> structure."""
    answer = _CALC_RE.sub("", answer).strip()
    m = _HASH_RE.search(answer)
    final = m.group(1).strip() if m else answer
    reasoning = _HASH_RE.sub("", answer).strip()
    completion = f"{THINK_OPEN}{reasoning}{THINK_CLOSE}{ANSWER_OPEN}{final}{ANSWER_CLOSE}"
    return [{"role": "user", "content": question.strip()},
            {"role": "assistant", "content": completion}]


def alpaca_to_messages(ex: dict) -> list[dict]:
    instr = ex["instruction"].strip()
    inp = (ex.get("input") or "").strip()
    user = f"{instr}\n\n{inp}" if inp else instr
    return [{"role": "user", "content": user}, {"role": "assistant", "content": ex["output"].strip()}]


def dolly_to_messages(ex: dict) -> list[dict]:
    instr = ex["instruction"].strip()
    ctx = (ex.get("context") or "").strip()
    user = f"{instr}\n\n{ctx}" if ctx else instr
    return [{"role": "user", "content": user}, {"role": "assistant", "content": ex["response"].strip()}]


def collect_examples(context_length: int, limit_per_set: int | None) -> list[tuple[list[int], list[int]]]:
    from datasets import load_dataset

    examples: list[tuple[list[int], list[int]]] = []
    n_kept = n_skipped = 0

    def add(messages):
        nonlocal n_kept, n_skipped
        ids, mask = encode_chat(messages)
        if len(ids) <= context_length and sum(mask) > 0:
            examples.append((ids, mask))
            n_kept += 1
        else:
            n_skipped += 1

    print("Loading tatsu-lab/alpaca ...")
    alpaca = load_dataset("tatsu-lab/alpaca", split="train")
    for ex in (alpaca.select(range(min(limit_per_set, len(alpaca)))) if limit_per_set else alpaca):
        add(alpaca_to_messages(ex))

    print("Loading databricks/databricks-dolly-15k ...")
    try:
        dolly = load_dataset("databricks/databricks-dolly-15k", split="train")
        for ex in (dolly.select(range(min(limit_per_set, len(dolly)))) if limit_per_set else dolly):
            add(dolly_to_messages(ex))
    except Exception as e:  # noqa: BLE001
        print(f"  (skipping dolly: {e})")

    print("Loading openai/gsm8k (main/train) ...")
    gsm = load_dataset("openai/gsm8k", "main", split="train")
    for ex in (gsm.select(range(min(limit_per_set, len(gsm)))) if limit_per_set else gsm):
        add(gsm8k_to_messages(ex["question"], ex["answer"]))

    print(f"Collected {n_kept} examples (skipped {n_skipped} that exceeded context_length).")
    return examples


def write_packed(examples, context_length: int, out_path: str) -> int:
    tokens, masks = pack_examples(examples, context_length)
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    with h5py.File(out_path, "w") as f:
        f.create_dataset("tokens", data=tokens)
        f.create_dataset("loss_mask", data=masks)
    print(f"  wrote {tokens.shape[0]} packed rows x {context_length} -> {out_path}")
    return tokens.shape[0]


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--context_length", type=int, default=1024)
    p.add_argument("--out_dir", default="/ephemeral/data")
    p.add_argument("--dev_frac", type=float, default=0.02)
    p.add_argument("--limit_per_set", type=int, default=None, help="cap examples per dataset (debug)")
    p.add_argument("--seed", type=int, default=42)
    args = p.parse_args()

    examples = collect_examples(args.context_length, args.limit_per_set)
    rng = np.random.default_rng(args.seed)
    rng.shuffle(examples)
    n_dev = max(1, int(len(examples) * args.dev_frac))
    dev, train = examples[:n_dev], examples[n_dev:]

    write_packed(train, args.context_length, os.path.join(args.out_dir, "sft_packed.h5"))
    write_packed(dev, args.context_length, os.path.join(args.out_dir, "sft_dev_packed.h5"))


if __name__ == "__main__":
    main()


Writing scripts/prepare_sft_data.py


In [44]:
%%writefile scripts/pretrain_base.py
"""
Pretrain the mid-size (~400M) base model from scratch on the Pile HDF5 corpus.

This is the shared starting checkpoint for every post-training stage. It upgrades the
original ``train_transformer.py`` recipe with the things needed to actually train a
mid-size model on 2x H100: DistributedDataParallel, bf16 autocast, gradient accumulation,
a cosine LR schedule with warmup, weight-decay param groups, and periodic checkpointing.
The original ``train_transformer.py`` is left untouched.

Single GPU:
    PYTHONPATH=. python scripts/pretrain_base.py
Both GPUs:
    PYTHONPATH=. torchrun --standalone --nproc_per_node=2 scripts/pretrain_base.py

Override any config field from the CLI, e.g. ``--batch_size 16 --train_steps 50000``.
"""

from __future__ import annotations

import os
import time

import numpy as np
import torch

from config.post_training_config import PretrainConfig
from data_loader.data_loader import get_batch_iterator
from src.post_training.cli import parse_config_with_json
from src.post_training.distributed import ddp_setup, ddp_wrap, cleanup, reduce_scalar
from src.post_training.logging_utils import MetricsLogger
from src.post_training.optim import configure_optimizer, cosine_lr
from src.post_training.utils import (
    amp_autocast, build_model_from_config, save_stage_ckpt, set_seed, unwrap,
)


@torch.no_grad()
def estimate_loss(model, cfg, ctx, iters: int) -> dict[str, float]:
    model.eval()
    out = {}
    for split, path in [("train", cfg.train_path), ("dev", cfg.dev_path)]:
        if not os.path.exists(path):
            continue
        it = get_batch_iterator(path, cfg.batch_size, cfg.context_length, device=ctx.device)
        losses = torch.zeros(iters)
        for k in range(iters):
            xb, yb = next(it)
            with amp_autocast(cfg.amp_dtype, ctx.device):
                _, loss = model(xb, yb)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


def main():
    cfg, extras = parse_config_with_json(
        PretrainConfig, "configs/pretrain.json",
        extra={"--resume": dict(type=str, default=None, help="checkpoint to resume from")})
    resume = extras.resume
    ctx = ddp_setup(cfg.device)
    # Different data shuffle per rank (the loader shuffles via numpy global RNG).
    set_seed(cfg.seed + ctx.rank)

    model = build_model_from_config(cfg).to(ctx.device)
    start_step = 0
    if resume and os.path.exists(resume):
        ck = torch.load(resume, map_location="cpu", weights_only=False)
        unwrap(model).load_state_dict(ck["model_state_dict"])
        start_step = ck.get("step", 0)
        if ctx.is_main:
            print(f"Resumed from {resume} at step {start_step}")

    if cfg.compile:
        model = torch.compile(model)
    model = ddp_wrap(model, ctx)

    optimizer = configure_optimizer(unwrap(model), cfg.lr, cfg.weight_decay)
    if resume and os.path.exists(resume):
        ck = torch.load(resume, map_location="cpu", weights_only=False)
        if ck.get("optimizer_state_dict"):
            optimizer.load_state_dict(ck["optimizer_state_dict"])

    logger = None
    if ctx.is_main:
        n_params = sum(p.numel() for p in unwrap(model).parameters())
        print(f"Model parameters: {n_params:,} (~{n_params/1e6:.0f}M) | world_size={ctx.world_size}")
        print(f"Effective batch = {cfg.batch_size}*{cfg.grad_accum}*{ctx.world_size} "
              f"= {cfg.batch_size*cfg.grad_accum*ctx.world_size} seqs/step")
        logger = MetricsLogger("pretrain", cfg.log_dir, use_wandb=cfg.use_wandb,
                               wandb_project=cfg.wandb_project, config=vars(cfg).copy() if hasattr(cfg, "__dict__") else None)

    batch_iter = get_batch_iterator(cfg.train_path, cfg.batch_size, cfg.context_length, device=ctx.device)
    tokens_per_step = cfg.batch_size * cfg.context_length * cfg.grad_accum * ctx.world_size

    model.train()
    t0 = time.perf_counter()
    for step in range(start_step, cfg.train_steps):
        lr = cosine_lr(step, warmup_steps=cfg.warmup_steps, max_steps=cfg.train_steps,
                       lr=cfg.lr, min_lr=cfg.min_lr)
        for g in optimizer.param_groups:
            g["lr"] = lr

        optimizer.zero_grad(set_to_none=True)
        accum_loss = 0.0
        for micro in range(cfg.grad_accum):
            xb, yb = next(batch_iter)
            # Only sync grads on the last micro-step (DDP optimization).
            sync = (micro == cfg.grad_accum - 1) or not ctx.enabled
            cm = model.no_sync() if (ctx.enabled and not sync) else _nullcm()
            with cm, amp_autocast(cfg.amp_dtype, ctx.device):
                _, loss = model(xb, yb)
                loss = loss / cfg.grad_accum
            loss.backward()
            accum_loss += loss.item()

        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        optimizer.step()

        if ctx.is_main and step % 20 == 0:
            dt = time.perf_counter() - t0
            tok_s = tokens_per_step * 20 / dt if step > start_step else 0.0
            t0 = time.perf_counter()
            print(f"step {step} | loss {accum_loss:.4f} | lr {lr:.2e} | {tok_s:,.0f} tok/s")
            if logger:
                logger.log(step, {"train_loss": accum_loss, "lr": lr, "tok_per_s": tok_s})

        if step > start_step and step % cfg.eval_steps == 0:
            ev = estimate_loss(model, cfg, ctx, cfg.eval_iters)
            ev = {k: reduce_scalar(v, ctx) for k, v in ev.items()}
            if ctx.is_main:
                print(f"  [eval] step {step} | " + " | ".join(f"{k} {v:.4f}" for k, v in ev.items()))
                if logger:
                    logger.log(step, {f"eval_{k}": v for k, v in ev.items()})

        if ctx.is_main and step > start_step and step % cfg.save_every == 0:
            save_stage_ckpt(cfg.out_ckpt, model, optimizer, stage="pretrain",
                            cfg=cfg, step=step, metrics={"train_loss": accum_loss})
            print(f"  saved checkpoint -> {cfg.out_ckpt} (step {step})")

    if ctx.is_main:
        save_stage_ckpt(cfg.out_ckpt, model, optimizer, stage="pretrain", cfg=cfg,
                        step=cfg.train_steps, metrics={"train_loss": accum_loss})
        print(f"Done. Final checkpoint -> {cfg.out_ckpt}")
        if logger:
            logger.close()
    cleanup(ctx)


import contextlib


def _nullcm():
    return contextlib.nullcontext()


if __name__ == "__main__":
    main()


Writing scripts/pretrain_base.py


In [45]:
%%writefile scripts/render_diagrams.sh
#!/usr/bin/env bash
# Render the hand-drawn, colour-coded diagram sources (docs/diagrams/src/*.mmd) to SVGs
# (docs/diagrams/*.svg) that are embedded as images in the docs. We pre-render because
# GitHub's live Mermaid does not reliably support `look: handDrawn`; embedding the SVG makes
# the hand-drawn look show everywhere.
#
# One-time setup:
#   sudo apt-get install -y nodejs                      # Node >= 18
#   sudo npm install -g @mermaid-js/mermaid-cli         # provides `mmdc`
#   # a Chrome/Chromium for headless rendering (e.g. google-chrome-stable)
#
# Usage (from repo root):
#   bash scripts/render_diagrams.sh
set -euo pipefail
cd "$(dirname "$0")/.."

SRC=docs/diagrams/src
OUT=docs/diagrams
CHROME="${CHROME:-/usr/bin/google-chrome-stable}"
PP=$(mktemp)
echo "{\"executablePath\":\"$CHROME\",\"args\":[\"--no-sandbox\",\"--disable-gpu\",\"--disable-dev-shm-usage\"]}" > "$PP"

for m in "$SRC"/*.mmd; do
  base=$(basename "$m" .mmd)
  mmdc -p "$PP" -i "$m" -o "$OUT/$base.png" -b white -s 2   # PNG @2x: renders in every viewer (GitHub, VS Code preview)
  echo "rendered $OUT/$base.png"
done
rm -f "$PP"
echo "Done. Edit a .mmd in $SRC, re-run this script, and the embedded image updates."


Writing scripts/render_diagrams.sh


In [46]:
%%writefile scripts/run_posttraining.sh
#!/usr/bin/env bash
# Turnkey post-training pipeline: SFT -> Reward Model -> {DPO, PPO} -> GRPO -> eval table.
# Assumes the base model is already pretrained (scripts/pretrain_base.py ->
# /ephemeral/ckpts/base_pretrained.pt) and the datasets are prepared (scripts/prepare_*).
#
# Usage (from repo root):
#   bash scripts/run_posttraining.sh            # use both GPUs (torchrun)
#   NPROC=1 bash scripts/run_posttraining.sh    # single GPU
#
# Each stage writes a checkpoint to /ephemeral/ckpts and metrics JSONL to /ephemeral/logs.
set -euo pipefail

cd "$(dirname "$0")/.."
export PYTHONPATH=. HF_HOME=/ephemeral/hf_cache PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
PY=/ephemeral/venv/bin/python
NPROC=${NPROC:-2}

run() {  # run a training script single- or multi-GPU
  if [ "$NPROC" -gt 1 ]; then
    /ephemeral/venv/bin/torchrun --standalone --nproc_per_node="$NPROC" "$@"
  else
    $PY "$@"
  fi
}

echo "############ 1/5  SFT ############"
run scripts/train_sft.py

echo "############ 2/5  Reward Model ############"
run scripts/train_reward.py

echo "############ 3/5  DPO ############"
run scripts/train_dpo.py --loss_type dpo

echo "############ 4/5  PPO (GSM8K, verifier reward) ############"
run scripts/train_ppo.py --reward_source verifier

echo "############ 5/5  GRPO (arithmetic curriculum -> GSM8K) ############"
run scripts/train_grpo.py

echo "############ Eval: GSM8K accuracy across stages ############"
TABLE=/ephemeral/logs/stage_table.jsonl
rm -f "$TABLE"
for s in base_pretrained sft dpo ppo grpo; do
  [ -f "/ephemeral/ckpts/$s.pt" ] && \
    $PY scripts/eval_post_training.py --ckpt "/ephemeral/ckpts/$s.pt" --label "$s" --limit 200 --append "$TABLE"
done
$PY scripts/eval_post_training.py --table "$TABLE"
echo "Done. Metrics in /ephemeral/logs, checkpoints in /ephemeral/ckpts."


Writing scripts/run_posttraining.sh


In [47]:
%%writefile scripts/train_dpo.py
"""
Direct Preference Optimization (and ORPO / KTO variants) on preference pairs.

The policy is initialized from the SFT checkpoint; a frozen deep copy of it serves as the
DPO/KTO reference (ORPO is reference-free). Reports implicit-reward accuracy on held-out
preferences and GSM8K dev accuracy.

    PYTHONPATH=. python scripts/train_dpo.py --loss_type dpo --beta 0.1
    PYTHONPATH=. torchrun --standalone --nproc_per_node=2 scripts/train_dpo.py
"""

from __future__ import annotations

import time

import torch

from config.post_training_config import DPOConfig
from data_loader.preference_dataset import get_preference_iterator
from src.post_training.cli import parse_config_with_json
from src.post_training.distributed import ddp_setup, ddp_wrap, cleanup, reduce_scalar
from src.post_training.dpo import dpo_loss, orpo_loss, kto_loss, implicit_accuracy
from src.post_training.logging_utils import MetricsLogger
from src.post_training.optim import configure_optimizer, cosine_lr
from src.post_training.rollout import sequence_logprobs
from src.post_training.utils import (
    amp_autocast, load_backbone_from_ckpt, make_frozen_copy, save_stage_ckpt, set_seed, unwrap,
)

TEST_PATH = "/ephemeral/data/preferences_test.jsonl"


def _logps(model, ids, mask, requires_grad):
    return sequence_logprobs(model, ids, mask, requires_grad=requires_grad)


def _compute_losses(policy, ref, batch, cfg, ctx):
    B = batch["chosen_ids"].size(0)
    ids = torch.cat([batch["chosen_ids"], batch["rejected_ids"]], dim=0)
    mask = torch.cat([batch["chosen_mask"], batch["rejected_mask"]], dim=0)

    with amp_autocast(cfg.amp_dtype, ctx.device):
        psum, pn = _logps(policy, ids, mask, requires_grad=True)
    pc, pr, ncn, nrn = psum[:B], psum[B:], pn[:B], pn[B:]

    if cfg.loss_type == "orpo":
        return orpo_loss(pc, pr, ncn, nrn, orpo_lambda=cfg.orpo_lambda)

    with torch.no_grad(), amp_autocast(cfg.amp_dtype, ctx.device):
        rsum, _ = _logps(ref, ids, mask, requires_grad=False)
    rc, rr = rsum[:B], rsum[B:]
    if cfg.loss_type == "kto":
        return kto_loss(pc, pr, rc, rr, beta=cfg.beta)
    return dpo_loss(pc, pr, rc, rr, beta=cfg.beta)


@torch.no_grad()
def eval_implicit_acc(policy, ref, cfg, ctx, max_batches: int = 100) -> tuple[float, float]:
    policy.eval()
    it = get_preference_iterator(TEST_PATH, cfg.batch_size, cfg.max_len, device=ctx.device,
                                 rank=ctx.rank, world_size=ctx.world_size, shuffle=False, infinite=False)
    acc, marg, n = 0.0, 0.0, 0
    for batch in it:
        loss, cr, rr = _compute_losses(policy, ref, batch, cfg, ctx)
        acc += implicit_accuracy(cr, rr).item()
        marg += (cr - rr).mean().item()
        n += 1
        if n >= max_batches:
            break
    policy.train()
    return acc / max(1, n), marg / max(1, n)


def main():
    cfg, _ = parse_config_with_json(DPOConfig, "configs/dpo.json")
    ctx = ddp_setup(cfg.device)
    set_seed(cfg.seed + ctx.rank)

    policy = load_backbone_from_ckpt(cfg, cfg.sft_ckpt, ctx.device)
    ref = make_frozen_copy(policy, device=ctx.device) if cfg.loss_type != "orpo" else None
    policy = ddp_wrap(policy, ctx)
    optimizer = configure_optimizer(unwrap(policy), cfg.lr, cfg.weight_decay)

    with open(cfg.pref_path) as f:
        n_rows = sum(1 for line in f if line.strip())
    total_steps = max(1, (n_rows // (cfg.batch_size * ctx.world_size)) * cfg.epochs)

    logger = None
    if ctx.is_main:
        print(f"DPO[{cfg.loss_type}] from {cfg.sft_ckpt} | {n_rows} pairs | total_steps={total_steps} | beta={cfg.beta}")
        logger = MetricsLogger(f"dpo_{cfg.loss_type}", cfg.log_dir, use_wandb=cfg.use_wandb, wandb_project=cfg.wandb_project)

    train_it = get_preference_iterator(cfg.pref_path, cfg.batch_size, cfg.max_len, device=ctx.device,
                                       rank=ctx.rank, world_size=ctx.world_size, shuffle=True, infinite=True)

    policy.train()
    t0 = time.perf_counter()
    for step in range(total_steps):
        lr = cosine_lr(step, warmup_steps=cfg.warmup_steps, max_steps=total_steps, lr=cfg.lr, min_lr=cfg.lr * 0.1)
        for g in optimizer.param_groups:
            g["lr"] = lr

        batch = next(train_it)
        loss, cr, rr = _compute_losses(policy, ref, batch, cfg, ctx)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), cfg.grad_clip)
        optimizer.step()

        if ctx.is_main and step % 20 == 0:
            acc = implicit_accuracy(cr, rr).item()
            dt = time.perf_counter() - t0; t0 = time.perf_counter()
            print(f"step {step}/{total_steps} | loss {loss.item():.4f} | acc {acc:.3f} | "
                  f"r_chosen {cr.mean().item():.3f} r_rejected {rr.mean().item():.3f} | {dt:.1f}s/20")
            if logger:
                logger.log(step, {"train_loss": loss.item(), "train_acc": acc,
                                  "r_chosen": cr.mean().item(), "r_rejected": rr.mean().item(), "lr": lr})

        if step > 0 and step % cfg.eval_steps == 0:
            acc, marg = eval_implicit_acc(policy, ref, cfg, ctx)
            acc, marg = reduce_scalar(acc, ctx), reduce_scalar(marg, ctx)
            if ctx.is_main:
                print(f"  [eval] step {step} | test_acc {acc:.3f} | margin {marg:.3f}")
                if logger:
                    logger.log(step, {"test_acc": acc, "test_margin": marg})

        if ctx.is_main and step > 0 and step % cfg.save_every == 0:
            save_stage_ckpt(cfg.out_ckpt, policy, optimizer, stage=f"dpo_{cfg.loss_type}", cfg=cfg, step=step,
                            metrics={"train_loss": loss.item()})

    if ctx.is_main:
        # Unwrap for the final eval: other ranks are already at cleanup(), so a collective on
        # the DDP-wrapped policy here would hang (NCCL timeout). The periodic eval runs on all ranks.
        acc, marg = eval_implicit_acc(unwrap(policy), ref, cfg, ctx)
        save_stage_ckpt(cfg.out_ckpt, policy, optimizer, stage=f"dpo_{cfg.loss_type}", cfg=cfg, step=total_steps,
                        metrics={"test_acc": acc, "test_margin": marg})
        print(f"Done DPO[{cfg.loss_type}]. test_acc {acc:.3f} margin {marg:.3f} -> {cfg.out_ckpt}")
        if logger:
            logger.close()
    cleanup(ctx)


if __name__ == "__main__":
    main()


Writing scripts/train_dpo.py


In [48]:
%%writefile scripts/train_grpo.py
"""
GRPO / RLVR on GSM8K (DeepSeek-R1 style), from scratch -- no critic, group-relative
advantages, verifiable reward.

Per iteration: for each prompt sample a group of G completions, score each with the GSM8K
verifier, compute group-relative advantages, and update with a token-level clipped
surrogate + KL-to-reference penalty. An arithmetic warm-up curriculum runs first so the
policy gets non-zero reward variance before facing full GSM8K.

    PYTHONPATH=. python scripts/train_grpo.py
    PYTHONPATH=. torchrun --standalone --nproc_per_node=2 scripts/train_grpo.py
"""

from __future__ import annotations

import time

import torch

from config.post_training_config import GRPOConfig
from data_loader.prompt_dataset import get_prompt_iterator
from src.post_training.cli import parse_config_with_json
from src.post_training.chat_template import decode, encode_prompt
from src.post_training.distributed import ddp_setup, ddp_wrap, cleanup, reduce_scalar
from src.post_training.evaluation import gsm8k_accuracy, load_gsm8k_eval
from src.post_training.grpo import group_advantages, grpo_loss
from src.post_training.logging_utils import MetricsLogger
from src.post_training.optim import configure_optimizer
from src.post_training.rewards import reward_gsm8k
from src.post_training.rollout import compute_logprobs, rollout_prompts
from src.post_training.utils import (
    amp_autocast, load_backbone_from_ckpt, make_frozen_copy, save_stage_ckpt, set_seed, unwrap,
)


def seq_lengths_from_mask(response_mask, prompt_lens):
    N, T = response_mask.shape
    pos = torch.arange(T, device=response_mask.device)
    last = torch.where(response_mask, pos[None, :], torch.full_like(response_mask, -1, dtype=torch.long)).max(dim=1).values
    return torch.where(last >= 0, last + 1, prompt_lens)


def main():
    cfg, _ = parse_config_with_json(GRPOConfig, "configs/grpo.json")
    ctx = ddp_setup(cfg.device)
    set_seed(cfg.seed + ctx.rank)

    policy = load_backbone_from_ckpt(cfg, cfg.sft_ckpt, ctx.device)
    ref = make_frozen_copy(policy, device=ctx.device)
    policy_ddp = ddp_wrap(policy, ctx)
    optimizer = configure_optimizer(unwrap(policy_ddp), cfg.lr, weight_decay=0.0)

    eval_set = None
    logger = MetricsLogger("grpo", cfg.log_dir, use_wandb=cfg.use_wandb, wandb_project=cfg.wandb_project) if ctx.is_main else None
    if ctx.is_main:
        print(f"GRPO from {cfg.sft_ckpt} | group_size={cfg.group_size} | world={ctx.world_size}")

    warm_it = get_prompt_iterator(cfg.curriculum_path, cfg.prompts_per_iter, rank=ctx.rank,
                                  world_size=ctx.world_size, seed=cfg.seed)
    main_it = get_prompt_iterator(cfg.prompt_path, cfg.prompts_per_iter, rank=ctx.rank,
                                  world_size=ctx.world_size, seed=cfg.seed)

    G = cfg.group_size
    for it in range(cfg.iterations):
        rows = next(warm_it if it < cfg.curriculum_iters else main_it)
        # Replicate each prompt G times, group-contiguously.
        base_prompts = [encode_prompt([{"role": "user", "content": r["prompt"]}]) for r in rows]
        prompts = [p for p in base_prompts for _ in range(G)]
        golds = [r.get("gold") for r in rows for _ in range(G)]

        policy.eval()
        with amp_autocast(cfg.amp_dtype, ctx.device):
            seqs, rmask, plens = rollout_prompts(policy, prompts, cfg.rollout_len, device=ctx.device,
                                                 temperature=cfg.temperature, top_p=cfg.top_p if cfg.top_p < 1 else None)
        resp = rmask[:, 1:]

        seq_lens = seq_lengths_from_mask(rmask, plens)
        responses = [decode(seqs[i, plens[i]:seq_lens[i]].tolist()) for i in range(len(prompts))]
        rewards = torch.tensor([reward_gsm8k(responses[i], golds[i]) for i in range(len(prompts))],
                               device=ctx.device, dtype=torch.float32)
        adv = group_advantages(rewards, G)

        with torch.no_grad(), amp_autocast(cfg.amp_dtype, ctx.device):
            old_logp, _ = compute_logprobs(policy, seqs, rmask, temperature=cfg.temperature, requires_grad=False)
            ref_logp, _ = compute_logprobs(ref, seqs, rmask, temperature=cfg.temperature, requires_grad=False)
        old_logp, ref_logp = old_logp.float(), ref_logp.float()

        policy.train()
        N = seqs.size(0)
        agg = {"loss": 0.0, "kl": 0.0, "clipfrac": 0.0, "n": 0}
        for _ in range(cfg.grpo_epochs):
            perm = torch.randperm(N, device=ctx.device)
            for s in range(0, N, max(1, G)):  # minibatch ~ one group's worth
                mb = perm[s:s + max(1, G)]
                with amp_autocast(cfg.amp_dtype, ctx.device):
                    new_logp, _ = compute_logprobs(policy_ddp, seqs[mb], rmask[mb], temperature=cfg.temperature, requires_grad=True)
                loss, st = grpo_loss(new_logp.float(), old_logp[mb], ref_logp[mb], adv[mb], resp[mb],
                                     clip=cfg.clip, kl_coef=cfg.kl_coef)
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(policy_ddp.parameters(), cfg.grad_clip)
                optimizer.step()
                agg["loss"] += loss.item(); agg["kl"] += st["kl"]; agg["clipfrac"] += st["clipfrac"]; agg["n"] += 1

        mean_reward = reduce_scalar(rewards.mean().item(), ctx)
        # Fraction of groups with non-zero reward spread (informative groups).
        grp_std = rewards.view(-1, G).std(dim=1)
        informative = reduce_scalar((grp_std > 1e-6).float().mean().item(), ctx)
        resp_len = reduce_scalar(resp.float().sum(1).mean().item(), ctx)
        n = max(1, agg["n"])
        if ctx.is_main and it % 5 == 0:
            phase = "warmup" if it < cfg.curriculum_iters else "gsm8k"
            print(f"iter {it}[{phase}] | reward {mean_reward:.3f} | informative {informative:.2f} | "
                  f"loss {agg['loss']/n:.4f} | KL {agg['kl']/n:.4f} | clipfrac {agg['clipfrac']/n:.3f} | resp_len {resp_len:.0f}")
            if logger:
                logger.log(it, {"reward": mean_reward, "informative_groups": informative,
                                "loss": agg["loss"]/n, "kl": agg["kl"]/n, "resp_len": resp_len})

        if ctx.is_main and it > 0 and it % cfg.eval_every == 0:
            if eval_set is None:
                eval_set = load_gsm8k_eval("test", limit=200)
            res = gsm8k_accuracy(unwrap(policy_ddp), eval_set, device=ctx.device, max_new_tokens=cfg.rollout_len)
            print(f"  [eval] iter {it} | GSM8K test acc {res['accuracy']:.3f} ({res['correct']}/{res['n']})")
            if logger:
                logger.log(it, {"gsm8k_acc": res["accuracy"]})
        if ctx.is_main and it > 0 and it % cfg.save_every == 0:
            save_stage_ckpt(cfg.out_ckpt, unwrap(policy_ddp), optimizer, stage="grpo", cfg=cfg, step=it,
                            metrics={"reward": mean_reward})

    if ctx.is_main:
        save_stage_ckpt(cfg.out_ckpt, unwrap(policy_ddp), optimizer, stage="grpo", cfg=cfg, step=cfg.iterations, metrics={})
        print(f"Done GRPO -> {cfg.out_ckpt}")
        if logger:
            logger.close()
    cleanup(ctx)


if __name__ == "__main__":
    main()


Writing scripts/train_grpo.py


In [49]:
%%writefile scripts/train_ppo.py
"""
PPO RLHF on GSM8K (the classic InstructGPT recipe), from scratch.

Per iteration: roll out completions with the current policy, score them (verifiable GSM8K
reward, or a trained reward model), add a per-token KL-to-reference penalty, compute GAE
advantages with the shared value head, then run several clipped-surrogate update epochs.
Reports mean reward, KL, value loss, clip fraction, and held-out GSM8K accuracy.

    PYTHONPATH=. python scripts/train_ppo.py --reward_source verifier
    PYTHONPATH=. torchrun --standalone --nproc_per_node=2 scripts/train_ppo.py
"""

from __future__ import annotations

import time

import torch
import torch.nn.functional as F

from config.post_training_config import PPOConfig
from data_loader.prompt_dataset import get_prompt_iterator
from src.post_training.cli import parse_config_with_json
from src.post_training.chat_template import EOT_ID, decode, encode_prompt
from src.post_training.distributed import ddp_setup, ddp_wrap, cleanup, reduce_scalar
from src.post_training.evaluation import gsm8k_accuracy, load_gsm8k_eval
from src.post_training.logging_utils import MetricsLogger
from src.post_training.optim import configure_optimizer
from src.post_training.ppo import compute_gae, whiten, ppo_policy_loss, ppo_value_loss, approx_kl
from src.post_training.reward_model import load_reward_model
from src.post_training.rewards import reward_gsm8k
from src.post_training.rollout import compute_logprobs, rollout_prompts
from src.post_training.utils import (
    amp_autocast, load_backbone_from_ckpt, make_frozen_copy, masked_mean, save_stage_ckpt, set_seed, unwrap,
)
from src.post_training.value_head import TransformerWithValueHead


def actor_logp_values(actor, seqs, temperature):
    """One forward through the actor-critic -> (logp, values) in the action frame (B,T-1)."""
    logits, values = actor(seqs)
    logits = logits[:, :-1, :]
    logp_all = F.log_softmax(logits.float() / max(temperature, 1e-6), dim=-1)
    logp = logp_all.gather(-1, seqs[:, 1:, None]).squeeze(-1)
    return logp, values[:, :-1]


def seq_lengths_from_mask(response_mask, prompt_lens):
    """Real token count per row = last response position + 1 (or prompt_len if no response)."""
    N, T = response_mask.shape
    pos = torch.arange(T, device=response_mask.device)
    last = torch.where(response_mask, pos[None, :], torch.full_like(response_mask, -1, dtype=torch.long)).max(dim=1).values
    return torch.where(last >= 0, last + 1, prompt_lens)


def main():
    cfg, _ = parse_config_with_json(PPOConfig, "configs/ppo.json")
    ctx = ddp_setup(cfg.device)
    set_seed(cfg.seed + ctx.rank)

    backbone = load_backbone_from_ckpt(cfg, cfg.sft_ckpt, ctx.device)
    ref = make_frozen_copy(backbone, device=ctx.device)
    actor = TransformerWithValueHead(backbone).to(ctx.device)
    actor_ddp = ddp_wrap(actor, ctx)
    optimizer = configure_optimizer(unwrap(actor_ddp), cfg.lr, weight_decay=0.0)

    rm = load_reward_model(cfg, cfg.reward_ckpt, ctx.device) if cfg.reward_source == "rm" else None

    eval_set = None  # loaded lazily on first eval to keep startup/smoke runs offline
    logger = MetricsLogger("ppo", cfg.log_dir, use_wandb=cfg.use_wandb, wandb_project=cfg.wandb_project) if ctx.is_main else None
    if ctx.is_main:
        print(f"PPO from {cfg.sft_ckpt} | reward={cfg.reward_source} | world={ctx.world_size}")

    prompt_it = get_prompt_iterator(cfg.prompt_path, cfg.prompts_per_iter, rank=ctx.rank,
                                    world_size=ctx.world_size, seed=cfg.seed)

    for it in range(cfg.iterations):
        rows = next(prompt_it)
        prompts = [encode_prompt([{"role": "user", "content": r["prompt"]}]) for r in rows]
        golds = [r.get("gold") for r in rows]

        # --- rollout ---
        actor.eval()
        with amp_autocast(cfg.amp_dtype, ctx.device):
            seqs, rmask, plens = rollout_prompts(actor, prompts, cfg.rollout_len, device=ctx.device,
                                                 temperature=cfg.temperature, top_p=cfg.top_p if cfg.top_p < 1 else None)
        resp = rmask[:, 1:]                       # action-frame response mask (N, T-1)

        # --- score ---
        seq_lens = seq_lengths_from_mask(rmask, plens)
        responses = [decode(seqs[i, plens[i]:seq_lens[i]].tolist()) for i in range(len(rows))]
        if cfg.reward_source == "rm":
            with torch.no_grad(), amp_autocast(cfg.amp_dtype, ctx.device):
                task_r = rm(seqs, seq_lengths=seq_lens).float().tolist()
        else:
            task_r = [reward_gsm8k(responses[i], golds[i]) for i in range(len(rows))]

        # --- per-token rewards: KL penalty everywhere + task reward at last response token ---
        with torch.no_grad(), amp_autocast(cfg.amp_dtype, ctx.device):
            old_logp, old_values = actor_logp_values(actor, seqs, cfg.temperature)
            ref_logp, _ = compute_logprobs(ref, seqs, rmask, temperature=cfg.temperature, requires_grad=False)
        old_logp, old_values, ref_logp = old_logp.float(), old_values.float(), ref_logp.float()

        rewards = -cfg.kl_coef * (old_logp - ref_logp) * resp.float()
        last_idx = seq_lens - 2                    # action index of the last response token
        last_idx = last_idx.clamp(min=0)
        task_t = torch.tensor(task_r, device=ctx.device, dtype=torch.float32)
        rewards[torch.arange(len(rows), device=ctx.device), last_idx] += task_t

        values_next = torch.cat([old_values[:, 1:], torch.zeros_like(old_values[:, :1])], dim=1)
        adv, returns = compute_gae(rewards, old_values, values_next, resp, gamma=cfg.gamma, lam=cfg.gae_lambda)
        adv = whiten(adv, resp)

        # --- clipped PPO update epochs over minibatches of rollout rows ---
        actor.train()
        N = seqs.size(0)
        stats = {"policy_loss": 0.0, "value_loss": 0.0, "clipfrac": 0.0, "kl": 0.0, "n": 0}
        for _ in range(cfg.ppo_epochs):
            perm = torch.randperm(N, device=ctx.device)
            for s in range(0, N, cfg.minibatch_size):
                mb = perm[s:s + cfg.minibatch_size]
                with amp_autocast(cfg.amp_dtype, ctx.device):
                    new_logp, new_values = actor_logp_values(actor_ddp, seqs[mb], cfg.temperature)
                m = resp[mb]
                p_loss, clipf = ppo_policy_loss(new_logp.float(), old_logp[mb], adv[mb], m, clip=cfg.clip)
                v_loss = ppo_value_loss(new_values.float(), old_values[mb], returns[mb], m, vf_clip=cfg.vf_clip)
                loss = p_loss + cfg.vf_coef * v_loss
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(actor_ddp.parameters(), cfg.grad_clip)
                optimizer.step()
                stats["policy_loss"] += p_loss.item(); stats["value_loss"] += v_loss.item()
                stats["clipfrac"] += clipf.item()
                stats["kl"] += approx_kl(new_logp.float(), old_logp[mb], m).item(); stats["n"] += 1

        mean_reward = reduce_scalar(float(sum(task_r) / max(1, len(task_r))), ctx)
        kl_ref = reduce_scalar(masked_mean(old_logp - ref_logp, resp).item(), ctx)
        resp_len = reduce_scalar(resp.float().sum(1).mean().item(), ctx)
        n = max(1, stats["n"])
        if ctx.is_main and it % 5 == 0:
            print(f"iter {it} | reward {mean_reward:.3f} | KL_ref {kl_ref:.3f} | "
                  f"ploss {stats['policy_loss']/n:.4f} | vloss {stats['value_loss']/n:.4f} | "
                  f"clipfrac {stats['clipfrac']/n:.3f} | resp_len {resp_len:.0f}")
            if logger:
                logger.log(it, {"reward": mean_reward, "kl_ref": kl_ref, "policy_loss": stats["policy_loss"]/n,
                                "value_loss": stats["value_loss"]/n, "clipfrac": stats["clipfrac"]/n, "resp_len": resp_len})

        if ctx.is_main and it > 0 and it % cfg.eval_every == 0:
            if eval_set is None:
                eval_set = load_gsm8k_eval("test", limit=200)
            res = gsm8k_accuracy(unwrap(actor_ddp), eval_set, device=ctx.device, max_new_tokens=cfg.rollout_len)
            print(f"  [eval] iter {it} | GSM8K test acc {res['accuracy']:.3f} ({res['correct']}/{res['n']})")
            if logger:
                logger.log(it, {"gsm8k_acc": res["accuracy"]})
        if ctx.is_main and it > 0 and it % cfg.save_every == 0:
            save_stage_ckpt(cfg.out_ckpt, unwrap(actor_ddp).transformer, optimizer, stage="ppo", cfg=cfg, step=it,
                            metrics={"reward": mean_reward})

    if ctx.is_main:
        save_stage_ckpt(cfg.out_ckpt, unwrap(actor_ddp).transformer, optimizer, stage="ppo", cfg=cfg, step=cfg.iterations, metrics={})
        print(f"Done PPO -> {cfg.out_ckpt}")
        if logger:
            logger.close()
    cleanup(ctx)


if __name__ == "__main__":
    main()


Writing scripts/train_ppo.py


In [50]:
%%writefile scripts/train_reward.py
"""
Train the reward model on preference pairs with the Bradley-Terry loss.

Initializes the reward backbone from the SFT checkpoint, adds a scalar reward head, and
trains so chosen responses score above rejected ones. Reports held-out preference accuracy.

Single GPU:
    PYTHONPATH=. python scripts/train_reward.py
Both GPUs:
    PYTHONPATH=. torchrun --standalone --nproc_per_node=2 scripts/train_reward.py
"""

from __future__ import annotations

import time

import torch

from config.post_training_config import RewardConfig
from data_loader.preference_dataset import get_preference_iterator
from src.post_training.cli import parse_config_with_json
from src.post_training.distributed import ddp_setup, ddp_wrap, cleanup, reduce_scalar
from src.post_training.logging_utils import MetricsLogger
from src.post_training.optim import configure_optimizer, cosine_lr
from src.post_training.reward_model import RewardModel
from src.post_training.reward_train import bradley_terry_loss, preference_accuracy, reward_margin
from src.post_training.utils import amp_autocast, load_backbone_from_ckpt, save_stage_ckpt, set_seed, unwrap

TEST_PATH = "/ephemeral/data/preferences_test.jsonl"


def _pair_rewards(rm, batch, cfg, ctx):
    """Forward chosen+rejected in one pass; return (chosen_rewards, rejected_rewards)."""
    B = batch["chosen_ids"].size(0)
    ids = torch.cat([batch["chosen_ids"], batch["rejected_ids"]], dim=0)
    lens = torch.cat([batch["chosen_len"], batch["rejected_len"]], dim=0)
    with amp_autocast(cfg.amp_dtype, ctx.device):
        rewards = rm(ids, seq_lengths=lens).float()
    return rewards[:B], rewards[B:]


@torch.no_grad()
def eval_accuracy(rm, cfg, ctx, max_batches: int = 100) -> tuple[float, float]:
    rm.eval()
    it = get_preference_iterator(TEST_PATH, cfg.batch_size, cfg.max_len, device=ctx.device,
                                 rank=ctx.rank, world_size=ctx.world_size, shuffle=False, infinite=False)
    acc, marg, n = 0.0, 0.0, 0
    for batch in it:
        cr, rr = _pair_rewards(rm, batch, cfg, ctx)
        acc += preference_accuracy(cr, rr).item()
        marg += reward_margin(cr, rr).item()
        n += 1
        if n >= max_batches:
            break
    rm.train()
    return acc / max(1, n), marg / max(1, n)


def main():
    cfg, _ = parse_config_with_json(RewardConfig, "configs/reward.json")
    ctx = ddp_setup(cfg.device)
    set_seed(cfg.seed + ctx.rank)

    backbone = load_backbone_from_ckpt(cfg, cfg.sft_ckpt, ctx.device)
    rm = RewardModel(backbone).to(ctx.device)
    # find_unused_parameters=True: the reward model uses the backbone's forward_hidden + a
    # reward head and never its lm_head, so lm_head params get no gradient. Without this flag
    # DDP errors on the first backward.
    rm = ddp_wrap(rm, ctx, find_unused_parameters=True)
    optimizer = configure_optimizer(unwrap(rm), cfg.lr, cfg.weight_decay)

    import json
    with open(cfg.pref_path) as f:
        n_rows = sum(1 for line in f if line.strip())
    total_steps = max(1, (n_rows // (cfg.batch_size * ctx.world_size)) * cfg.epochs)

    logger = None
    if ctx.is_main:
        print(f"Reward model from {cfg.sft_ckpt} | {n_rows} pairs | total_steps={total_steps}")
        logger = MetricsLogger("reward", cfg.log_dir, use_wandb=cfg.use_wandb, wandb_project=cfg.wandb_project)

    train_it = get_preference_iterator(cfg.pref_path, cfg.batch_size, cfg.max_len, device=ctx.device,
                                       rank=ctx.rank, world_size=ctx.world_size, shuffle=True, infinite=True)

    rm.train()
    t0 = time.perf_counter()
    for step in range(total_steps):
        lr = cosine_lr(step, warmup_steps=cfg.warmup_steps, max_steps=total_steps, lr=cfg.lr, min_lr=cfg.lr * 0.1)
        for g in optimizer.param_groups:
            g["lr"] = lr

        batch = next(train_it)
        cr, rr = _pair_rewards(rm, batch, cfg, ctx)
        loss = bradley_terry_loss(cr, rr)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(rm.parameters(), cfg.grad_clip)
        optimizer.step()

        if ctx.is_main and step % 20 == 0:
            acc = preference_accuracy(cr, rr).item()
            dt = time.perf_counter() - t0; t0 = time.perf_counter()
            print(f"step {step}/{total_steps} | loss {loss.item():.4f} | train_acc {acc:.3f} | lr {lr:.2e} | {dt:.1f}s/20")
            if logger:
                logger.log(step, {"train_loss": loss.item(), "train_acc": acc, "lr": lr})

        if step > 0 and step % cfg.eval_steps == 0:
            acc, marg = eval_accuracy(rm, cfg, ctx)
            acc, marg = reduce_scalar(acc, ctx), reduce_scalar(marg, ctx)
            if ctx.is_main:
                print(f"  [eval] step {step} | test_acc {acc:.3f} | margin {marg:.3f}")
                if logger:
                    logger.log(step, {"test_acc": acc, "test_margin": marg})

        if ctx.is_main and step > 0 and step % cfg.save_every == 0:
            save_stage_ckpt(cfg.out_ckpt, rm, optimizer, stage="reward", cfg=cfg, step=step,
                            metrics={"train_loss": loss.item()})

    if ctx.is_main:
        # Unwrap for the final eval: other ranks are already at cleanup(), so a collective on
        # the DDP-wrapped model here would hang (NCCL timeout). The periodic eval runs on all ranks.
        acc, marg = eval_accuracy(unwrap(rm), cfg, ctx)
        save_stage_ckpt(cfg.out_ckpt, rm, optimizer, stage="reward", cfg=cfg, step=total_steps,
                        metrics={"test_acc": acc, "test_margin": marg})
        print(f"Done RM. test_acc {acc:.3f} margin {marg:.3f} -> {cfg.out_ckpt}")
        if logger:
            logger.close()
    cleanup(ctx)


if __name__ == "__main__":
    main()


Writing scripts/train_reward.py


In [51]:
%%writefile scripts/train_sft.py
"""
Supervised Fine-Tuning of the pretrained base on packed instruction data.

Loads the base checkpoint, trains with the prompt-masked SFT loss, periodically reports
masked dev loss, and saves an SFT checkpoint. DDP + bf16, single code path for 1 or N GPUs.

Single GPU:
    PYTHONPATH=. python scripts/train_sft.py
Both GPUs:
    PYTHONPATH=. torchrun --standalone --nproc_per_node=2 scripts/train_sft.py
"""

from __future__ import annotations

import contextlib
import math
import time

import torch

from config.post_training_config import SFTConfig
from data_loader.sft_dataset import get_sft_batch_iterator
from src.post_training.cli import parse_config_with_json
from src.post_training.distributed import ddp_setup, ddp_wrap, cleanup, reduce_scalar
from src.post_training.logging_utils import MetricsLogger
from src.post_training.optim import configure_optimizer, cosine_lr
from src.post_training.sft import sft_loss
from src.post_training.utils import amp_autocast, load_backbone_from_ckpt, save_stage_ckpt, set_seed, unwrap

DEV_PATH = "/ephemeral/data/sft_dev_packed.h5"


@torch.no_grad()
def eval_dev(model, cfg, ctx, dev_path: str, max_batches: int = 50) -> float:
    model.eval()
    it = get_sft_batch_iterator(dev_path, cfg.batch_size, device=ctx.device,
                                rank=ctx.rank, world_size=ctx.world_size, shuffle=False, infinite=False)
    total, n = 0.0, 0
    for tokens, mask, _ in it:
        with amp_autocast(cfg.amp_dtype, ctx.device):
            logits, _ = model(tokens)
            loss = sft_loss(logits, tokens, mask)
        total += loss.item(); n += 1
        if n >= max_batches:
            break
    model.train()
    return total / max(1, n)


def main():
    cfg, _ = parse_config_with_json(SFTConfig, "configs/sft.json")
    ctx = ddp_setup(cfg.device)
    set_seed(cfg.seed + ctx.rank)

    model = load_backbone_from_ckpt(cfg, cfg.pretrained_ckpt, ctx.device)
    if cfg.compile:
        model = torch.compile(model)
    model = ddp_wrap(model, ctx)
    optimizer = configure_optimizer(unwrap(model), cfg.lr, cfg.weight_decay)

    logger = None
    if ctx.is_main:
        print(f"SFT from {cfg.pretrained_ckpt} | world_size={ctx.world_size}")
        logger = MetricsLogger("sft", cfg.log_dir, use_wandb=cfg.use_wandb, wandb_project=cfg.wandb_project)

    train_it = get_sft_batch_iterator(cfg.data_path, cfg.batch_size, device=ctx.device,
                                      rank=ctx.rank, world_size=ctx.world_size, shuffle=True, infinite=True)

    # Estimate total steps for the cosine schedule from dataset size.
    import h5py
    with h5py.File(cfg.data_path, "r") as f:
        n_rows = f["tokens"].shape[0]
    steps_per_epoch = max(1, n_rows // (cfg.batch_size * ctx.world_size))
    total_steps = cfg.max_steps if cfg.max_steps > 0 else steps_per_epoch * cfg.epochs
    if ctx.is_main:
        print(f"{n_rows} packed rows | ~{steps_per_epoch} steps/epoch | total_steps={total_steps}")

    model.train()
    t0 = time.perf_counter()
    for step in range(total_steps):
        lr = cosine_lr(step, warmup_steps=cfg.warmup_steps, max_steps=total_steps, lr=cfg.lr, min_lr=cfg.min_lr)
        for g in optimizer.param_groups:
            g["lr"] = lr

        tokens, mask, epoch = next(train_it)
        if epoch >= cfg.epochs and cfg.max_steps <= 0:
            break
        optimizer.zero_grad(set_to_none=True)
        with amp_autocast(cfg.amp_dtype, ctx.device):
            logits, _ = model(tokens)
            loss = sft_loss(logits, tokens, mask)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        optimizer.step()

        if ctx.is_main and step % 20 == 0:
            dt = time.perf_counter() - t0; t0 = time.perf_counter()
            print(f"step {step}/{total_steps} | loss {loss.item():.4f} | ppl {math.exp(min(20, loss.item())):.2f} | lr {lr:.2e} | {dt:.1f}s/20")
            if logger:
                logger.log(step, {"train_loss": loss.item(), "lr": lr})

        if step > 0 and step % cfg.eval_steps == 0:
            dev = reduce_scalar(eval_dev(model, cfg, ctx, DEV_PATH), ctx)
            if ctx.is_main:
                print(f"  [eval] step {step} | dev_loss {dev:.4f} | dev_ppl {math.exp(min(20, dev)):.2f}")
                if logger:
                    logger.log(step, {"dev_loss": dev})

        if ctx.is_main and step > 0 and step % cfg.save_every == 0:
            save_stage_ckpt(cfg.out_ckpt, model, optimizer, stage="sft", cfg=cfg, step=step,
                            metrics={"train_loss": loss.item()})

    if ctx.is_main:
        # Use the unwrapped model for the final eval: the other ranks have already reached
        # cleanup(), so calling the DDP-wrapped model here would launch a collective with no
        # peer and hang (NCCL timeout). The periodic eval above runs on all ranks, so it is fine.
        dev = eval_dev(unwrap(model), cfg, ctx, DEV_PATH)
        save_stage_ckpt(cfg.out_ckpt, model, optimizer, stage="sft", cfg=cfg, step=total_steps,
                        metrics={"dev_loss": dev})
        print(f"Done SFT. dev_loss {dev:.4f} -> {cfg.out_ckpt}")
        if logger:
            logger.close()
    cleanup(ctx)


if __name__ == "__main__":
    main()


Writing scripts/train_sft.py


In [52]:
%%writefile scripts/train_transformer.py
from __future__ import annotations

import argparse
import contextlib
import os
import re
import sys
import tempfile
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
from tqdm import tqdm

ROOT_DIR = Path(__file__).resolve().parents[1]
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from config.config import default_config as config
from src.models.transformer import Transformer


# --- Runtime Diagnostics Helpers ---

def bytes_to_gib(num_bytes: int) -> float:
    """Convert a byte count to gibibytes for human-readable memory reports."""
    return num_bytes / (1024 ** 3)


def get_device_report(device: str) -> str:
    """
    Build a short report describing the runtime environment: PyTorch/CUDA
    versions and, when running on a GPU, its name, capability, and total VRAM.
    This makes it easy to collect comparable training reports across machines.
    """
    lines = [
        f"PyTorch version: {torch.__version__}",
        f"Configured device: {device}",
        f"CUDA available: {torch.cuda.is_available()}",
        f"CUDA version: {torch.version.cuda}",
    ]

    if device.startswith('cuda') and torch.cuda.is_available():
        device_index = torch.cuda.current_device()
        props = torch.cuda.get_device_properties(device_index)
        total_vram_gib = bytes_to_gib(props.total_memory)
        lines.extend([
            f"GPU name: {torch.cuda.get_device_name(device_index)}",
            f"GPU capability: {props.major}.{props.minor}",
            f"Total VRAM: {total_vram_gib:.2f} GiB",
        ])
    else:
        lines.append("GPU name: N/A (running without CUDA)")

    return "\n".join(lines)


def get_peak_memory_report(device: str) -> str:
    """Report peak GPU memory (allocated/reserved) since the last reset, or N/A on CPU."""
    if device.startswith('cuda') and torch.cuda.is_available():
        peak_allocated = bytes_to_gib(torch.cuda.max_memory_allocated())
        peak_reserved = bytes_to_gib(torch.cuda.max_memory_reserved())
        return (
            f"Peak VRAM allocated: {peak_allocated:.2f} GiB | "
            f"Peak VRAM reserved: {peak_reserved:.2f} GiB"
        )
    return "Peak VRAM allocated: N/A | Peak VRAM reserved: N/A"


def estimate_memory_budget(num_params: int, device: str, use_amp: bool) -> str:
    """
    Print a rough training VRAM budget so users can predict OOM before launching.

    AdamW keeps fp32 weights + grads + two moment buffers (~16 bytes/param). This is an
    estimate of optimizer/parameter state only (activations depend on batch/context and
    are reduced a lot by gradient checkpointing). CUDA-only; returns N/A otherwise.
    """
    if not (device.startswith("cuda") and torch.cuda.is_available()):
        return "VRAM budget: N/A (no CUDA device)"
    # weights(4) + grad(4) + Adam m(4) + Adam v(4); AMP adds bf16/fp16 copies but keeps
    # the fp32 master state, so ~16 B/param is a reasonable floor either way.
    state_gib = bytes_to_gib(num_params * 16)
    props = torch.cuda.get_device_properties(torch.cuda.current_device())
    total_gib = bytes_to_gib(props.total_memory)
    note = " (+ activations; reduce with --grad-checkpointing / --grad-accum)"
    return (
        f"VRAM budget: ~{state_gib:.2f} GiB params+optimizer state vs {total_gib:.2f} GiB "
        f"total on {torch.cuda.get_device_name()}{note}"
    )


# --- Checkpoint Helpers ---

CHECKPOINT_RE = re.compile(r"checkpoint_step_(\d+)\.pt$")


def load_checkpoint_file(path: str, device: str) -> Dict[str, Any]:
    """Load a checkpoint while supporting both newer and older PyTorch versions."""
    try:
        return torch.load(path, map_location=torch.device(device), weights_only=False)
    except TypeError:
        return torch.load(path, map_location=torch.device(device))


def default_checkpoint_dir(out_path: str) -> str:
    """Return a checkpoint directory tied to the configured final model path."""
    model_path = Path(out_path)
    return str(model_path.with_suffix("")) + "_checkpoints"


def checkpoint_path(checkpoint_dir: str, step: int) -> str:
    """Build a stable checkpoint path for the last completed training step."""
    return os.path.join(checkpoint_dir, f"checkpoint_step_{step:08d}.pt")


def checkpoint_step(path: str) -> int:
    """Extract the step number from a checkpoint filename."""
    match = CHECKPOINT_RE.search(os.path.basename(path))
    if not match:
        return -1
    return int(match.group(1))


def list_checkpoints(checkpoint_dir: str) -> List[str]:
    """Return periodic checkpoints sorted by training step."""
    if not os.path.isdir(checkpoint_dir):
        return []
    paths = [
        os.path.join(checkpoint_dir, name)
        for name in os.listdir(checkpoint_dir)
        if CHECKPOINT_RE.search(name)
    ]
    return sorted(paths, key=checkpoint_step)


def resolve_resume_path(resume: Optional[str], checkpoint_dir: str) -> Optional[str]:
    """
    Resolve a resume argument.

    ``--resume`` with no value uses the latest periodic checkpoint in checkpoint_dir.
    ``--resume path/to/file.pt`` loads that exact checkpoint.
    """
    if resume is None:
        return None
    if resume == "latest":
        checkpoints = list_checkpoints(checkpoint_dir)
        if not checkpoints:
            raise FileNotFoundError(f"No checkpoints found in {checkpoint_dir}")
        return checkpoints[-1]
    return resume


def current_lr(optimizer: torch.optim.Optimizer) -> float:
    """Read the learning rate from the first optimizer parameter group."""
    return float(optimizer.param_groups[0]["lr"])


def lr_for_step(train_config: Dict[str, Any], step: int) -> float:
    """Return the learning rate that should be active at a given step."""
    if step > train_config['t_lr_decay_step']:
        return float(train_config['t_lr_decayed'])
    return float(train_config['t_lr'])


def set_optimizer_lr(optimizer: torch.optim.Optimizer, lr: float) -> None:
    """Set all optimizer parameter groups to the same learning rate."""
    for group in optimizer.param_groups:
        group["lr"] = lr


def save_training_checkpoint(
    path: str,
    model: Transformer,
    optimizer: torch.optim.Optimizer,
    train_config: Dict[str, Any],
    losses: List[float],
    *,
    step: int,
    train_loss: Optional[float] = None,
    dev_loss: Optional[float] = None,
    is_final: bool = False,
) -> None:
    """
    Save model, optimizer, loss history, and LR schedule metadata.

    ``step`` is the last completed zero-based training step, so resume starts at
    ``step + 1``.
    """
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    payload = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'losses': losses,
        'train_loss': train_loss,
        'dev_loss': dev_loss,
        'step': step,
        'last_completed_step': step,
        'steps': step + 1,
        'is_final': is_final,
        'config': dict(train_config),
        'device': train_config['device'],
        'pytorch_version': torch.__version__,
        'cuda_version': torch.version.cuda,
        'lr_state': {
            'current_lr': current_lr(optimizer),
            'initial_lr': train_config['t_lr'],
            'decayed_lr': train_config['t_lr_decayed'],
            'decay_step': train_config['t_lr_decay_step'],
        },
    }
    target_dir = os.path.dirname(path) or "."
    with tempfile.NamedTemporaryFile(
        dir=target_dir,
        prefix=f".{os.path.basename(path)}.",
        suffix=".tmp",
        delete=False,
    ) as tmp_file:
        tmp_path = tmp_file.name
    try:
        torch.save(payload, tmp_path)
        os.replace(tmp_path, path)
    except Exception:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)
        raise


def restore_training_checkpoint(
    path: str,
    model: Transformer,
    optimizer: torch.optim.Optimizer,
    train_config: Dict[str, Any],
    device: str,
) -> Tuple[int, List[float]]:
    """
    Restore model/optimizer state and return ``(next_step, losses)``.

    Older checkpoints did not have ``last_completed_step``. For those, ``steps``
    is treated as the number of completed optimizer steps.
    """
    checkpoint = load_checkpoint_file(path, device)
    model.load_state_dict(checkpoint['model_state_dict'])

    optimizer_state = checkpoint.get('optimizer_state_dict')
    if optimizer_state:
        optimizer.load_state_dict(optimizer_state)

    if 'last_completed_step' in checkpoint:
        last_completed_step = int(checkpoint['last_completed_step'])
        next_step = last_completed_step + 1
    else:
        next_step = int(checkpoint.get('steps', 0))
        last_completed_step = next_step - 1

    if not optimizer_state:
        set_optimizer_lr(optimizer, lr_for_step(train_config, next_step))

    losses = [float(loss) for loss in checkpoint.get('losses', [])]
    print(
        f"Resumed from {path}. "
        f"Last completed step: {last_completed_step}. Next step: {next_step}."
    )
    return next_step, losses


def prune_old_checkpoints(checkpoint_dir: str, keep_last: int) -> None:
    """Keep only the most recent N periodic checkpoints when requested."""
    if keep_last <= 0:
        return
    checkpoints = list_checkpoints(checkpoint_dir)
    for old_path in checkpoints[:-keep_last]:
        os.remove(old_path)


def unique_output_path(out_path: str) -> str:
    """Avoid overwriting an existing final model checkpoint."""
    modified_model_out_path = out_path
    save_tries = 0
    while os.path.exists(modified_model_out_path):
        save_tries += 1
        model_out_name = os.path.splitext(out_path)[0]
        modified_model_out_path = model_out_name + f"_{save_tries}" + ".pt"
    return modified_model_out_path


def as_float(value: Any) -> Optional[float]:
    """Convert scalar tensors/numbers to plain floats for checkpoint metadata."""
    if value is None:
        return None
    if hasattr(value, "item"):
        return float(value.item())
    return float(value)


# --- Training / Evaluation ---

@torch.no_grad()
def estimate_loss(model: Transformer, train_config: Dict[str, Any], steps: int) -> Dict[str, float]:
    """
    Evaluate the model on training and development datasets and calculate average loss.

    Args:
        model (Transformer): The model being trained.
        train_config (dict): Training configuration values.
        steps (int): Number of steps to evaluate.

    Returns:
        dict: Dictionary containing average losses for 'train' and 'dev' splits.
    """
    out = {}
    model.eval()  # Set the model to evaluation mode.
    from data_loader.data_loader import get_batch_iterator

    for split in ['train', 'dev']:
        # Select the appropriate data path for the current split.
        data_path = train_config['train_path'] if split == 'train' else train_config['dev_path']

        # Create a batch iterator for evaluation.
        batch_iterator_eval = get_batch_iterator(
            data_path,
            train_config['t_batch_size'],
            train_config['t_context_length'],
            device=train_config['device'],
        )

        # Track loss values for each evaluation step.
        losses_eval = []
        for _ in range(steps):
            try:
                # Fetch a batch and calculate the loss.
                xb, yb = next(batch_iterator_eval)
                _, loss = model(xb, yb)
                losses_eval.append(float(loss.item()))
            except StopIteration:
                # Handle the case where the data iterator ends early.
                print(f"Warning: Iterator for {split} ended early.")
                break

        # Compute the mean loss for the current split.
        out[split] = float(np.mean(losses_eval)) if losses_eval else float("nan")

    model.train()  # Restore the model to training mode.
    return out


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Train the Transformer model from scratch.")
    parser.add_argument(
        "--resume",
        nargs="?",
        const="latest",
        default=None,
        help=(
            "Resume from a checkpoint path. Pass --resume with no value, or "
            "--resume latest, to use the newest checkpoint in the checkpoint directory."
        ),
    )
    parser.add_argument(
        "--checkpoint-every",
        type=int,
        default=None,
        help="Save a periodic checkpoint every N completed steps. 0 disables periodic checkpoints.",
    )
    parser.add_argument(
        "--checkpoint-dir",
        type=str,
        default=None,
        help="Directory for periodic checkpoints. Defaults to a directory next to t_out_path.",
    )
    parser.add_argument(
        "--keep-last",
        type=int,
        default=None,
        help="Keep only the most recent N periodic checkpoints. 0 keeps all.",
    )
    # --- Memory-optimisation flags (opt-in; all default to the config values, which are OFF) ---
    parser.add_argument(
        "--amp",
        dest="amp",
        action="store_true",
        default=None,
        help="Enable bf16/fp16 mixed-precision autocast (CUDA only; ignored on CPU).",
    )
    parser.add_argument(
        "--amp-dtype",
        type=str,
        choices=["bf16", "fp16"],
        default=None,
        help="Autocast dtype when --amp is set: bf16 (default, no GradScaler) or fp16.",
    )
    parser.add_argument(
        "--grad-checkpointing",
        dest="grad_checkpointing",
        action="store_true",
        default=None,
        help="Recompute transformer-block activations in backward to save VRAM.",
    )
    parser.add_argument(
        "--grad-accum",
        type=int,
        default=None,
        help="Accumulate gradients over N micro-batches per optimizer step (effective batch xN).",
    )
    parser.add_argument(
        "--report-memory",
        dest="report_memory",
        action="store_true",
        default=None,
        help="Print a rough VRAM budget (params + optimizer state) before training (CUDA only).",
    )
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    train_config = dict(config)
    checkpoint_every = (
        args.checkpoint_every
        if args.checkpoint_every is not None
        else train_config.get('t_checkpoint_steps', 0)
    )
    keep_last = (
        args.keep_last
        if args.keep_last is not None
        else train_config.get('t_keep_last_checkpoints', 0)
    )
    checkpoint_dir = (
        args.checkpoint_dir
        or train_config.get('t_checkpoint_dir')
        or default_checkpoint_dir(train_config['t_out_path'])
    )

    # --- Resolve memory-optimisation options (CLI overrides config; all default OFF) ---
    use_amp = args.amp if args.amp is not None else bool(train_config.get('use_amp', False))
    amp_dtype_name = args.amp_dtype or train_config.get('amp_dtype', 'bf16')
    use_grad_ckpt = (
        args.grad_checkpointing if args.grad_checkpointing is not None
        else bool(train_config.get('use_gradient_checkpointing', False))
    )
    grad_accum = max(1, args.grad_accum if args.grad_accum is not None
                     else int(train_config.get('grad_accum_steps', 1)))
    report_memory = (
        args.report_memory if args.report_memory is not None
        else bool(train_config.get('report_memory_budget', False))
    )

    device_is_cuda = train_config['device'].startswith('cuda') and torch.cuda.is_available()
    if use_amp and not device_is_cuda:
        print("[mem-opt] --amp requested but no CUDA device available; disabling AMP.")
        use_amp = False
    amp_dtype = torch.bfloat16 if amp_dtype_name == 'bf16' else torch.float16

    def autocast_ctx():
        if use_amp:
            return torch.autocast(device_type='cuda', dtype=amp_dtype)
        return contextlib.nullcontext()

    # GradScaler is only needed for fp16; bf16 has enough range. A disabled scaler is a no-op,
    # so the scale/unscale_/step/update calls below work unchanged for bf16 and CPU.
    use_scaler = use_amp and amp_dtype == torch.float16
    scaler = torch.amp.GradScaler('cuda', enabled=use_scaler)

    # --- Initialize the Model and Print Parameters ---

    # Print runtime/device diagnostics and reset GPU peak-memory stats before training.
    print(get_device_report(train_config['device']))
    if train_config['device'].startswith('cuda') and torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    model = Transformer(
        n_head=train_config['n_head'],
        n_embed=train_config['n_embed'],
        context_length=train_config['context_length'],
        vocab_size=train_config['vocab_size'],
        N_BLOCKS=train_config['n_blocks'],
    ).to(train_config['device'])

    # Print the total number of parameters.
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total number of parameters in the model: {total_params:,}")

    # Apply opt-in memory optimisations.
    model.gradient_checkpointing = use_grad_ckpt
    if report_memory:
        print(estimate_memory_budget(total_params, train_config['device'], use_amp))
    if use_amp or use_grad_ckpt or grad_accum > 1:
        print(
            f"[mem-opt] amp={use_amp}"
            f"{'(' + amp_dtype_name + ')' if use_amp else ''} "
            f"grad_checkpointing={use_grad_ckpt} grad_accum={grad_accum}"
        )

    # --- Optimizer Setup and Loss Tracking ---

    # Set up the AdamW optimizer with the specified learning rate.
    optimizer = torch.optim.AdamW(model.parameters(), lr=train_config['t_lr'])

    # List to track loss values during training.
    losses: List[float] = []
    start_step = 0
    last_completed_step = -1
    resume_path = resolve_resume_path(args.resume, checkpoint_dir)
    if resume_path is not None:
        start_step, losses = restore_training_checkpoint(
            resume_path,
            model,
            optimizer,
            train_config,
            train_config['device'],
        )
        last_completed_step = start_step - 1

    # Define a window size for averaging recent losses in the training loop.
    avg_window = 64

    # --- Training Loop ---

    from data_loader.data_loader import get_batch_iterator

    # Create a batch iterator for the training data.
    batch_iterator = get_batch_iterator(
        train_config['train_path'],
        train_config['t_batch_size'],
        train_config['t_context_length'],
        device=train_config['device'],
    )

    # Number of tokens processed per optimizer step (batch * context * grad_accum), for throughput.
    tokens_per_step = train_config['t_batch_size'] * train_config['t_context_length'] * grad_accum
    last_eval_time = time.perf_counter()
    latest_train_loss = None
    latest_dev_loss = None

    # Create a progress bar to monitor training progress.
    pbar = tqdm(range(start_step, train_config['t_train_steps']))
    for step in pbar:
        try:
            # Start the step timer.
            step_start_time = time.perf_counter()

            # Accumulate gradients over `grad_accum` micro-batches (==1 => original behaviour).
            optimizer.zero_grad(set_to_none=True)
            step_loss = 0.0
            for _ in range(grad_accum):
                xb, yb = next(batch_iterator)
                with autocast_ctx():
                    _, loss = model(xb, yb)
                    # Scale so the accumulated gradient equals the full-batch mean gradient.
                    loss = loss / grad_accum
                scaler.scale(loss).backward()
                step_loss += float(loss.item())

            # Record the (accumulated) loss for tracking.
            losses.append(step_loss)
            pbar.set_description(f"Train loss: {np.mean(losses[-avg_window:]):.4f}")

            # Clip gradients to prevent exploding gradients (unscale first for fp16 AMP).
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            scaler.step(optimizer)
            scaler.update()
            last_completed_step = step

            # Measure step time and instantaneous throughput for diagnostics.
            step_time = time.perf_counter() - step_start_time
            tokens_per_second = tokens_per_step / step_time if step_time > 0 else float('inf')

            # Periodically evaluate the model on training and development data.
            if step % train_config['t_eval_steps'] == 0:
                evaluation_losses = estimate_loss(model, train_config, train_config['t_eval_iters'])
                latest_train_loss = evaluation_losses['train']
                latest_dev_loss = evaluation_losses['dev']
                # Report timing/throughput for the most recent step and wall-time since last eval.
                now = time.perf_counter()
                elapsed_since_eval = now - last_eval_time
                last_eval_time = now
                print(
                    f"Step: {step}, Train loss: {latest_train_loss:.4f}, Dev loss: {latest_dev_loss:.4f}, "
                    f"Step time: {step_time:.3f}s, Throughput: {tokens_per_second:.2f} tokens/s, "
                    f"Elapsed since last eval: {elapsed_since_eval:.2f}s"
                )
                print(get_peak_memory_report(train_config['device']))

            # Decay the learning rate at the specified step.
            if step == train_config['t_lr_decay_step']:
                print('Decaying learning rate')
                set_optimizer_lr(optimizer, train_config['t_lr_decayed'])

            if checkpoint_every and checkpoint_every > 0 and (step + 1) % checkpoint_every == 0:
                path = checkpoint_path(checkpoint_dir, step)
                save_training_checkpoint(
                    path,
                    model,
                    optimizer,
                    train_config,
                    losses,
                    step=step,
                    train_loss=as_float(latest_train_loss),
                    dev_loss=as_float(latest_dev_loss),
                )
                prune_old_checkpoints(checkpoint_dir, int(keep_last or 0))
                print(f"Saved checkpoint to {path}")
        except StopIteration:
            # Handle the case where the training data iterator ends early.
            print("Training data iterator finished early.")
            break

    # --- Save Model and Final Evaluation ---

    # Perform a final evaluation of the model on training and development datasets.
    evaluation_losses = estimate_loss(model, train_config, 200)
    train_loss = evaluation_losses['train']
    dev_loss = evaluation_losses['dev']

    final_step = max(last_completed_step, start_step - 1)
    modified_model_out_path = unique_output_path(train_config['t_out_path'])

    # Save the model's state dictionary, optimizer state, and training metadata
    # (including the runtime device / PyTorch / CUDA versions for reproducibility).
    save_training_checkpoint(
        modified_model_out_path,
        model,
        optimizer,
        train_config,
        losses,
        step=final_step,
        train_loss=train_loss,
        dev_loss=dev_loss,
        is_final=True,
    )
    print(f"Saved model to {modified_model_out_path}")
    print(get_peak_memory_report(train_config['device']))
    print(f"Finished training. Train loss: {train_loss:.4f}, Dev loss: {dev_loss:.4f}")


if __name__ == "__main__":
    main()


Writing scripts/train_transformer.py


In [53]:
%%writefile src/__init__.py



Writing src/__init__.py


In [54]:
%%writefile src/models/__init__.py
# This file makes the 'src.models' directory a Python package.
from .mlp import MLP
from .attention import Head, MultiHeadAttention
from .transformer_block import Block
from .transformer import Transformer


Writing src/models/__init__.py


In [55]:
%%writefile src/models/attention.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class Head(nn.Module):
    """
    A single attention head.

    This module calculates attention scores and applies them to the values.
    It includes key, query, and value projections, and uses causal masking
    to prevent attending to future tokens.

    Args:
        head_size (int): The dimensionality of the key, query, and value projections.
        n_embed (int): The dimensionality of the input embedding.
        context_length (int): The maximum length of the input sequence, used for causal masking.
    """
    def __init__(self, head_size: int, n_embed: int, context_length: int) -> None:
        """
        Initializes the attention head.

        Args:
            head_size (int): The dimensionality of the key, query, and value projections.
            n_embed (int): The dimensionality of the input embedding.
            context_length (int): The maximum length of the input sequence.
        """
        super().__init__()
        self.key = nn.Linear(n_embed, head_size, bias=False)   # Key projection
        self.query = nn.Linear(n_embed, head_size, bias=False) # Query projection
        self.value = nn.Linear(n_embed, head_size, bias=False) # Value projection
        # Lower triangular matrix for causal masking
        self.register_buffer('tril', torch.tril(torch.ones(context_length, context_length)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass through the attention head.

        Args:
            x (torch.Tensor): Input tensor of shape (B, T, C).

        Returns:
            torch.Tensor: Output tensor after applying attention.
        """
        B, T, C = x.shape
        head_size = self.key.out_features
        k = self.key(x)     # (B, T, head_size)
        q = self.query(x)   # (B, T, head_size)
        scale_factor = 1 / math.sqrt(head_size)
        # Calculate attention weights: (B, T, head_size) @ (B, head_size, T) -> (B, T, T)
        attn_weights = q @ k.transpose(-2, -1) * scale_factor
        # Apply causal masking
        attn_weights = attn_weights.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        attn_weights = F.softmax(attn_weights, dim=-1)
        v = self.value(x)   # (B, T, head_size)
        # Apply attention weights to values
        out = attn_weights @ v # (B, T, T) @ (B, T, head_size) -> (B, T, head_size)
        return out

class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention module.

    This module combines multiple attention heads in parallel. The outputs of each head
    are concatenated and passed through a final linear projection to form the output.

    Args:
        n_head (int): The number of parallel attention heads.
        n_embed (int): The dimensionality of the input embedding.
        context_length (int): The maximum length of the input sequence.
    """
    def __init__(self, n_head: int, n_embed: int, context_length: int) -> None:
        """
        Initializes the multi-head attention module.

        Args:
            n_head (int): The number of parallel attention heads.
            n_embed (int): The dimensionality of the input embedding.
            context_length (int): The maximum length of the input sequence.
        """
        super().__init__()
        self.heads = nn.ModuleList([Head(n_embed // n_head, n_embed, context_length) for _ in range(n_head)])
        self.proj = nn.Linear(n_embed, n_embed)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass through the multi-head attention.

        Args:
            x (torch.Tensor): Input tensor of shape (B, T, C).

        Returns:
            torch.Tensor: Output tensor after concatenating the heads and applying the output projection.
        """
        # Concatenate the output of each head along the last dimension (C)
        x = torch.cat([h(x) for h in self.heads], dim=-1)
        # Apply final linear projection
        x = self.proj(x)
        return x

if __name__ == '__main__':
    # Example Usage (optional, for testing the module independently)
    batch_size = 2
    sequence_length = 5
    embedding_dim = 32
    num_heads = 4
    context_len = 5
    input_tensor = torch.randn(batch_size, sequence_length, embedding_dim)

    multihead_attn = MultiHeadAttention(n_head=num_heads, n_embed=embedding_dim, context_length=context_len)
    output_tensor = multihead_attn(input_tensor)

    print("MultiHeadAttention Input Shape:", input_tensor.shape)
    print("MultiHeadAttention Output Shape:", output_tensor.shape)


Writing src/models/attention.py


In [56]:
%%writefile src/models/mlp.py
import torch
import torch.nn as nn
from torch import Tensor

class MLP(nn.Module):
    """
    A simple Multi-Layer Perceptron with one hidden layer.

    This module is used within the Transformer block for feed-forward processing.
    It expands the input embedding size, applies a ReLU activation, and then projects it back
    to the original embedding size.

    Args:
        n_embed (int): The dimensionality of the input embedding.
    """
    def __init__(self, n_embed: int) -> None:
        """
        Initializes the MLP module.

        Args:
            n_embed (int): The dimensionality of the input embedding.
        """
        super().__init__()
        self.hidden = nn.Linear(n_embed, 4 * n_embed)  # Linear layer to expand embedding size
        self.relu = nn.ReLU()                        # ReLU activation function
        self.proj = nn.Linear(4 * n_embed, n_embed)  # Linear layer to project back to original size

    def forward(self, x: Tensor) -> Tensor:
        """
        Forward pass through the MLP.

        Args:
            x (torch.Tensor): Input tensor of shape (B, T, C), where B is batch size,
                              T is sequence length, and C is embedding size.

        Returns:
            torch.Tensor: Output tensor of the same shape as the input.
        """
        x = self.forward_embedding(x)
        x = self.project_embedding(x)
        return x

    def forward_embedding(self, x: Tensor) -> Tensor:
        """
        Applies the hidden linear layer followed by ReLU activation.

        Args:
            x (torch.Tensor): Input tensor.

        Returns:
            torch.Tensor: Output after the hidden layer and ReLU.
        """
        x = self.relu(self.hidden(x))
        return x

    def project_embedding(self, x: Tensor) -> Tensor:
        """
        Applies the projection linear layer.

        Args:
            x (torch.Tensor): Input tensor.

        Returns:
            torch.Tensor: Output after the projection layer.
        """
        x = self.proj(x)
        return x

if __name__ == '__main__':
    # Example Usage (optional, for testing the module independently)
    batch_size = 2
    sequence_length = 3
    embedding_dim = 16
    input_tensor = torch.randn(batch_size, sequence_length, embedding_dim)

    mlp_module = MLP(n_embed=embedding_dim)
    output_tensor = mlp_module(input_tensor)

    print("MLP Input Shape:", input_tensor.shape)
    print("MLP Output Shape:", output_tensor.shape)


Writing src/models/mlp.py


In [57]:
%%writefile src/models/transformer.py
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as checkpoint
from src.models.transformer_block import Block

class Transformer(nn.Module):
    """
    The main Transformer model.

    This class combines token and position embeddings with a sequence of Transformer blocks
    and a final linear layer for language modeling.

    Args:
        n_head (int): The number of attention heads in each transformer block.
        n_embed (int): The dimensionality of the embedding space.
        context_length (int): The maximum length of the input sequence.
        vocab_size (int): The size of the vocabulary.
        N_BLOCKS (int): The number of transformer blocks in the model.
    """
    def __init__(self, n_head: int, n_embed: int, context_length: int, vocab_size: int, N_BLOCKS: int) -> None:
        """
        Initializes the Transformer model.

        Args:
            n_head (int): Number of attention heads.
            n_embed (int): Embedding dimension.
            context_length (int): Maximum sequence length.
            vocab_size (int): Size of the vocabulary.
            N_BLOCKS (int): Number of transformer blocks.
        """
        super().__init__()
        self.context_length = context_length
        self.N_BLOCKS = N_BLOCKS
        # Opt-in activation (gradient) checkpointing; off by default so behaviour and
        # numerics are unchanged unless a caller explicitly turns it on (e.g. the
        # pretraining script's --grad-checkpointing flag to save VRAM). See issue #5.
        self.gradient_checkpointing = False
        self.token_embed = nn.Embedding(vocab_size, n_embed)
        self.position_embed = nn.Embedding(context_length, n_embed)
        self.attn_blocks = nn.ModuleList([Block(n_head, n_embed, context_length) for _ in range(N_BLOCKS)])
        self.layer_norm = nn.LayerNorm(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)
        self.register_buffer('pos_idxs', torch.arange(context_length))

    def _pre_attn_pass(self, idx: torch.Tensor) -> torch.Tensor:
        """
        Combines token and position embeddings.

        Args:
            idx (torch.Tensor): Input token indices.

        Returns:
            torch.Tensor: Sum of token and position embeddings.
        """
        B, T = idx.shape
        tok_embedding = self.token_embed(idx)
        pos_embedding = self.position_embed(self.pos_idxs[:T])
        return tok_embedding + pos_embedding

    def forward_hidden(self, idx: torch.Tensor) -> torch.Tensor:
        """
        Run the backbone and return the final hidden states AFTER the final layer norm.

        This is exactly the tensor that ``lm_head`` consumes, so it is the right
        representation for auxiliary heads added during post-training (a scalar value
        head for PPO, a scalar reward head for the reward model). Keeping it as a
        separate method lets those heads reuse the backbone without duplicating the
        forward logic or rewriting ``forward``.

        Args:
            idx (torch.Tensor): Input token indices, shape (B, T).

        Returns:
            torch.Tensor: Final hidden states, shape (B, T, n_embed).
        """
        x = self._pre_attn_pass(idx)
        for block in self.attn_blocks:
            if self.gradient_checkpointing and self.training:
                # Recompute block activations in backward instead of storing them,
                # trading compute for a large activation-memory saving. use_reentrant=False
                # is the modern, correct variant.
                x = checkpoint.checkpoint(block, x, use_reentrant=False)
            else:
                x = block(x)
        return self.layer_norm(x)

    def forward(self, idx: torch.Tensor, targets: torch.Tensor = None) -> tuple[torch.Tensor, torch.Tensor | None]:
        """
        Forward pass through the Transformer.

        Args:
            idx (torch.Tensor): Input token indices.
            targets (torch.Tensor, optional): Target token indices for loss calculation. Defaults to None.

        Returns:
            tuple: Logits and loss (if targets are provided).
        """
        x = self.forward_hidden(idx)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            # reshape (not view): targets come from a non-contiguous slice of the
            # batch tensor, which makes .view() raise on CPU (the cross-device .to('cuda')
            # copy happens to make it contiguous, which is why this only bit CPU runs).
            flat_logits = logits.reshape(B * T, C)
            targets = targets.reshape(B * T).long()
            loss = F.cross_entropy(flat_logits, targets)
        return logits, loss

    def forward_embedding(self, idx: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Forward pass focusing on the embedding and attention blocks.

        Args:
            idx (torch.Tensor): Input token indices.

        Returns:
            tuple: Output after attention blocks and the residual.
        """
        x = self._pre_attn_pass(idx)
        residual = x
        for block in self.attn_blocks:
            x, residual = block.forward_embedding(x)
        return x, residual

    def generate(self, idx: torch.Tensor, max_new_tokens: int) -> torch.Tensor:
        """
        Generates new tokens given a starting sequence.

        Args:
            idx (torch.Tensor): Initial sequence of token indices.
            max_new_tokens (int): Number of tokens to generate.

        Returns:
            torch.Tensor: The extended sequence of tokens.
        """
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.context_length:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

if __name__ == '__main__':
    # Example Usage (optional, for testing the module independently)
    batch_size = 2
    sequence_length = 5
    vocab_size = 100
    embedding_dim = 32
    num_heads = 4
    num_blocks = 2
    context_len = 5
    input_indices = torch.randint(0, vocab_size, (batch_size, sequence_length))

    transformer_model = Transformer(n_head=num_heads, n_embed=embedding_dim, context_length=context_len, vocab_size=vocab_size, N_BLOCKS=num_blocks)
    logits, loss = transformer_model(input_indices, targets=input_indices) # Using input as target for simplicity

    print("Transformer Logits Shape:", logits.shape)
    print("Transformer Loss:", loss)

    # Example of generating tokens
    start_indices = input_indices[:, :1]  # Take the first token of each sequence as start
    generated_tokens = transformer_model.generate(start_indices, max_new_tokens=5)
    print("Generated Tokens Shape:", generated_tokens.shape)


Writing src/models/transformer.py


In [58]:
%%writefile src/models/transformer_block.py
import torch
import torch.nn as nn
from src.models.attention import MultiHeadAttention
from src.models.mlp import MLP

class Block(nn.Module):
    """
    A single Transformer block.

    This block consists of a multi-head attention layer followed by an MLP,
    with layer normalization and residual connections.

    Args:
        n_head (int): The number of attention heads in the multi-head attention layer.
        n_embed (int): The dimensionality of the input embedding.
        context_length (int): The maximum length of the input sequence.
    """
    def __init__(self, n_head: int, n_embed: int, context_length: int) -> None:
        """
        Initializes the Transformer block.

        Args:
            n_head (int): The number of attention heads.
            n_embed (int): The dimensionality of the embedding space.
            context_length (int): The maximum sequence length.
        """
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embed)
        self.attn = MultiHeadAttention(n_head, n_embed, context_length)
        self.ln2 = nn.LayerNorm(n_embed)
        self.mlp = MLP(n_embed)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass through the Transformer block.

        Args:
            x (torch.Tensor): Input tensor.

        Returns:
            torch.Tensor: Output tensor after the block.
        """
        # Apply multi-head attention with residual connection
        x = x + self.attn(self.ln1(x))
        # Apply MLP with residual connection
        x = x + self.mlp(self.ln2(x))
        return x

    def forward_embedding(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Forward pass focusing on the embedding and attention parts.

        Args:
            x (torch.Tensor): Input tensor.

        Returns:
            tuple: A tuple containing the output after MLP embedding and the residual.
        """
        res = x + self.attn(self.ln1(x))
        x = self.mlp.forward_embedding(self.ln2(res))
        return x, res

if __name__ == '__main__':
    # Example Usage (optional, for testing the module independently)
    batch_size = 2
    sequence_length = 5
    embedding_dim = 32
    num_heads = 4
    context_len = 5
    input_tensor = torch.randn(batch_size, sequence_length, embedding_dim)

    transformer_block = Block(n_head=num_heads, n_embed=embedding_dim, context_length=context_len)
    output_tensor = transformer_block(input_tensor)

    print("Transformer Block Input Shape:", input_tensor.shape)
    print("Transformer Block Output Shape:", output_tensor.shape)


Writing src/models/transformer_block.py


In [59]:
%%writefile src/post_training/__init__.py
"""
From-scratch post-training suite for ``train-llm-from-scratch``.

This package adds Supervised Fine-Tuning (SFT), reward modeling, PPO, DPO and
GRPO/RLVR on top of the repo's own custom :class:`~src.models.transformer.Transformer`.
Everything here is written in plain PyTorch in the same teaching style as the rest
of the repo -- no ``trl`` / ``peft`` / ``transformers`` for the algorithms.

Module map
----------
- ``chat_template``  : tiktoken r50k_base chat formatting + loss masking
- ``rollout``        : sampling + per-token log-prob utilities (the RL core)
- ``value_head``     : actor-critic wrapper for PPO
- ``reward_model``   : scalar reward head wrapper
- ``utils``          : frozen copies, checkpoint I/O, masked reductions, seeding
- ``distributed``    : optional pure-``torch.distributed`` DDP helpers
- ``sft`` / ``reward_train`` / ``dpo`` / ``ppo`` / ``grpo`` : the algorithms
- ``rewards``        : verifiable reward functions (GSM8K, arithmetic, format)
"""


Writing src/post_training/__init__.py


In [60]:
%%writefile src/post_training/chat_template.py
"""
Chat formatting + loss masking for post-training.

The repo's tokenizer is tiktoken ``r50k_base``, whose ONLY special token is
``<|endoftext|>`` (id 50256). We therefore cannot register new special tokens for
chat roles. Instead we use plain-text role markers that simply tokenize as ordinary
multi-token strings -- the model learns them during SFT just like any other text.
``<|endoftext|>`` (EOT) is reused as the turn/sequence terminator and the single
generation stop token.

Conversation format (one user turn -> one assistant turn)::

    <|user|>
    {user content}<|endoftext|><|assistant|>
    {assistant content}<|endoftext|>

For reasoning / RLVR tasks the assistant content carries the structure itself::

    <think>step by step ...</think><answer>42</answer>

which needs no special handling here -- it is just the assistant message body.

Two outputs matter for training:
- ``ids``       : the full token id sequence.
- ``loss_mask`` : 1 on tokens the model should be trained to PRODUCE (the assistant
                  content plus its terminating EOT), 0 on prompt / role-marker tokens.
                  This is exactly the SFT prompt-masking mechanism and the RL
                  ``response_mask`` for generated positions.
"""

from __future__ import annotations

from functools import lru_cache
from typing import Iterable

import tiktoken

# tiktoken r50k_base end-of-text id; the only true special token and our stop token.
EOT_ID = 50256

# Plain-text role markers (ordinary tokens, NOT registered specials).
USER_HEADER = "<|user|>\n"
ASSISTANT_HEADER = "<|assistant|>\n"
SYSTEM_HEADER = "<|system|>\n"

# Reasoning structure markers (also ordinary tokens). Exposed so reward parsing and
# data generation share a single source of truth.
THINK_OPEN, THINK_CLOSE = "<think>", "</think>"
ANSWER_OPEN, ANSWER_CLOSE = "<answer>", "</answer>"


@lru_cache(maxsize=1)
def get_tokenizer() -> "tiktoken.Encoding":
    """Return the shared r50k_base encoder (cached so we build it once)."""
    return tiktoken.get_encoding("r50k_base")


def _encode_ordinary(text: str) -> list[int]:
    """Encode text as ordinary tokens (no special-token handling)."""
    return get_tokenizer().encode_ordinary(text)


def _header_for(role: str) -> str:
    if role == "user":
        return USER_HEADER
    if role == "assistant":
        return ASSISTANT_HEADER
    if role == "system":
        return SYSTEM_HEADER
    raise ValueError(f"Unknown chat role: {role!r}")


def render_chat(messages: Iterable[dict], add_generation_prompt: bool = False) -> str:
    """
    Render a list of ``{"role", "content"}`` messages to the plain-text chat format.

    This is purely for display/debugging; tokenization goes through :func:`encode_chat`
    so that loss masks stay exactly aligned with token boundaries.

    Args:
        messages: iterable of ``{"role": "user"|"assistant"|"system", "content": str}``.
        add_generation_prompt: if True, append a trailing ``<|assistant|>\\n`` so the
            model is cued to generate (used at inference / rollout time).
    """
    parts: list[str] = []
    for m in messages:
        parts.append(_header_for(m["role"]))
        parts.append(m["content"])
        parts.append("<|endoftext|>")
    if add_generation_prompt:
        parts.append(ASSISTANT_HEADER)
    return "".join(parts)


def encode_chat(
    messages: Iterable[dict],
    add_generation_prompt: bool = False,
) -> tuple[list[int], list[int]]:
    """
    Tokenize a conversation and build an aligned per-token loss mask.

    The mask is 1 over assistant content (and its terminating EOT) and 0 everywhere
    else (role markers, user/system content). When ``add_generation_prompt`` is True
    the conversation ends with an ``<|assistant|>\\n`` header and no completion, so the
    returned mask is all zeros -- this is the prompt form used for rollouts.

    Returns:
        (ids, loss_mask) as equal-length python lists of ints.
    """
    ids: list[int] = []
    mask: list[int] = []

    for m in messages:
        role = m["role"]
        # Role header: always masked out (the model should not be trained to emit it,
        # and at rollout time it is part of the fixed prompt).
        header_ids = _encode_ordinary(_header_for(role))
        ids.extend(header_ids)
        mask.extend([0] * len(header_ids))

        content_ids = _encode_ordinary(m["content"])
        is_completion = role == "assistant"
        ids.extend(content_ids)
        mask.extend([1 if is_completion else 0] * len(content_ids))

        # Turn terminator. Trained on (mask=1) only when it closes an assistant turn,
        # so the model learns to stop.
        ids.append(EOT_ID)
        mask.append(1 if is_completion else 0)

    if add_generation_prompt:
        header_ids = _encode_ordinary(ASSISTANT_HEADER)
        ids.extend(header_ids)
        mask.extend([0] * len(header_ids))

    return ids, mask


def encode_prompt(messages: Iterable[dict]) -> list[int]:
    """Token ids for the prompt form (ends in the assistant header, ready to generate)."""
    ids, _ = encode_chat(messages, add_generation_prompt=True)
    return ids


def decode(ids: Iterable[int]) -> str:
    """
    Decode token ids back to text (for eval / verifier parsing / sample dumps).

    Defensive: drops the EOT terminator and any id >= EOT (50256), since the model's vocab
    is padded to 50304 but r50k_base can only decode ordinary tokens 0..50255. An
    under-trained model can emit those padding ids, which would otherwise crash decoding.
    """
    clean = [t for t in ids if 0 <= t < EOT_ID]
    return get_tokenizer().decode(clean)


Writing src/post_training/chat_template.py


In [61]:
%%writefile src/post_training/cli.py
"""
Shared CLI helper: turn any stage config dataclass into ``--field value`` arguments so
every training script can override hyperparameters without bespoke argparse blocks.
"""

from __future__ import annotations

import argparse
import json
from dataclasses import asdict, fields
from types import SimpleNamespace

from config.loader import load_config


def _add_typed(p: argparse.ArgumentParser, name: str, type_str) -> None:
    t = str(type_str)
    if "int" in t and "|" not in t:
        p.add_argument(f"--{name}", type=int)
    elif "float" in t:
        p.add_argument(f"--{name}", type=float)
    elif t == "bool" or t == "<class 'bool'>":
        p.add_argument(f"--{name}", type=lambda s: s.lower() in ("1", "true", "yes"))
    else:
        # str, optionals, unions -> accept as string (dataclass keeps None defaults otherwise)
        p.add_argument(f"--{name}", type=str)


def parse_config(cfg_cls, extra: dict | None = None):
    """
    Parse CLI overrides for a config dataclass.

    Returns ``(cfg, extras)`` where ``cfg`` is an instance of ``cfg_cls`` with any provided
    ``--field`` overrides applied, and ``extras`` is a namespace of any non-config args
    declared via ``extra`` (mapping ``"--flag" -> dict(argparse kwargs)``).
    """
    p = argparse.ArgumentParser()
    field_names = {f.name for f in fields(cfg_cls)}
    for f in fields(cfg_cls):
        _add_typed(p, f.name, f.type)
    for flag, kwargs in (extra or {}).items():
        p.add_argument(flag, **kwargs)
    args = vars(p.parse_args())

    overrides = {k: v for k, v in args.items() if k in field_names and v is not None}
    extras = {k: v for k, v in args.items() if k not in field_names}
    return cfg_cls(**overrides), SimpleNamespace(**extras)


def parse_config_with_json(cfg_cls, default_json: str, extra: dict | None = None):
    """
    Like :func:`parse_config`, but resolves the config from a JSON file too.

    Adds two flags on top of the per-field ``--field`` overrides:
      - ``--config PATH``   : the stage JSON to load (default ``default_json``).
      - ``--print-config``  : print the fully resolved config as JSON and exit.

    Resolution order (low -> high): dataclass defaults < ``configs/base.json`` < the stage
    JSON < CLI ``--field`` overrides (see :func:`config.loader.load_config`).

    Returns ``(cfg, extras)`` exactly like :func:`parse_config`.
    """
    p = argparse.ArgumentParser()
    field_names = {f.name for f in fields(cfg_cls)}
    for f in fields(cfg_cls):
        _add_typed(p, f.name, f.type)
    p.add_argument("--config", default=default_json, help="stage JSON config to load")
    p.add_argument("--print-config", action="store_true", help="print resolved config and exit")
    for flag, kwargs in (extra or {}).items():
        p.add_argument(flag, **kwargs)
    args = vars(p.parse_args())

    reserved = {"config", "print_config"}
    overrides = {k: v for k, v in args.items() if k in field_names and v is not None}
    extras = {k: v for k, v in args.items() if k not in field_names and k not in reserved}

    cfg = load_config(cfg_cls, args["config"], overrides)
    if args["print_config"]:
        print(json.dumps(asdict(cfg), indent=2))
        raise SystemExit(0)
    return cfg, SimpleNamespace(**extras)


Writing src/post_training/cli.py


In [62]:
%%writefile src/post_training/distributed.py
"""
Optional, from-scratch multi-GPU helpers (pure ``torch.distributed`` -- no accelerate).

Every training script works on a single GPU with plain ``python scripts/train_*.py`` and
on N GPUs with ``torchrun --standalone --nproc_per_node=N scripts/train_*.py``. When not
launched under torchrun, :func:`ddp_setup` returns a world_size=1 context and all the
helpers degrade to no-ops, so the scripts have a single code path.
"""

from __future__ import annotations

import os
from dataclasses import dataclass

import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP


@dataclass
class DDPContext:
    rank: int
    local_rank: int
    world_size: int
    device: str

    @property
    def is_main(self) -> bool:
        return self.rank == 0

    @property
    def enabled(self) -> bool:
        return self.world_size > 1


def ddp_setup(device: str = "cuda") -> DDPContext:
    """Initialize the process group if launched under torchrun; otherwise single-process.

    Reads ``RANK`` / ``LOCAL_RANK`` / ``WORLD_SIZE`` from the environment (set by
    torchrun). Uses the NCCL backend on CUDA, gloo on CPU.
    """
    world_size = int(os.environ.get("WORLD_SIZE", "1"))
    if world_size == 1:
        dev = device if (device == "cuda" and torch.cuda.is_available()) else "cpu"
        if dev == "cuda":
            torch.cuda.set_device(0)
        return DDPContext(rank=0, local_rank=0, world_size=1, device=dev)

    rank = int(os.environ["RANK"])
    local_rank = int(os.environ["LOCAL_RANK"])
    backend = "nccl" if (device == "cuda" and torch.cuda.is_available()) else "gloo"
    dist.init_process_group(backend=backend, rank=rank, world_size=world_size)
    if device == "cuda":
        torch.cuda.set_device(local_rank)
        dev = f"cuda:{local_rank}"
    else:
        dev = "cpu"
    return DDPContext(rank=rank, local_rank=local_rank, world_size=world_size, device=dev)


def ddp_wrap(model: torch.nn.Module, ctx: DDPContext, find_unused_parameters: bool = False) -> torch.nn.Module:
    """Wrap a model in DDP when running multi-GPU; return it unchanged otherwise.

    Only the *trainable* model should be wrapped. Reference / old-policy / reward models
    carry no gradients and must NOT be wrapped (they are replicated identically per rank).

    Pass ``find_unused_parameters=True`` for a model where some parameters do not receive a
    gradient on every step (for example the reward model, which uses the backbone's
    ``forward_hidden`` and a reward head but never its ``lm_head``). Without it, DDP raises
    a "did not get a gradient" error on the first backward.
    """
    if not ctx.enabled:
        return model
    device_ids = [ctx.local_rank] if ctx.device.startswith("cuda") else None
    return DDP(model, device_ids=device_ids, find_unused_parameters=find_unused_parameters)


def is_main_process(ctx: DDPContext) -> bool:
    return ctx.is_main


def reduce_scalar(value: float, ctx: DDPContext, average: bool = True) -> float:
    """All-reduce a python scalar across ranks (sum or mean). No-op when single-process."""
    if not ctx.enabled:
        return value
    t = torch.tensor([value], device=ctx.device, dtype=torch.float32)
    dist.all_reduce(t, op=dist.ReduceOp.SUM)
    if average:
        t /= ctx.world_size
    return t.item()


def barrier(ctx: DDPContext) -> None:
    if ctx.enabled:
        dist.barrier()


def cleanup(ctx: DDPContext) -> None:
    if ctx.enabled and dist.is_initialized():
        dist.destroy_process_group()


def shard_indices(num_items: int, ctx: DDPContext) -> range:
    """Return this rank's strided slice of ``range(num_items)`` for data sharding."""
    if not ctx.enabled:
        return range(num_items)
    return range(ctx.rank, num_items, ctx.world_size)


Writing src/post_training/distributed.py


In [63]:
%%writefile src/post_training/dpo.py
"""
Direct Preference Optimization and two reference-free / unpaired variants.

All operate on sequence-level log-probabilities of the chosen/rejected responses (summed
over response tokens via :func:`src.post_training.rollout.sequence_logprobs`).

- ``dpo_loss``  : the standard DPO objective; aligns the policy to preferences using a
                  frozen reference model and temperature ``beta``.
- ``orpo_loss`` : ORPO -- reference-FREE; combines the SFT NLL on the chosen response with
                  an odds-ratio preference term (folds SFT + alignment into one stage).
- ``kto_loss``  : KTO -- works from a per-example desirable/undesirable signal (here read
                  off the chosen/rejected pair) with a reference KL baseline.
"""

from __future__ import annotations

import torch
import torch.nn.functional as F


def dpo_loss(
    policy_chosen_logps: torch.Tensor,
    policy_rejected_logps: torch.Tensor,
    ref_chosen_logps: torch.Tensor,
    ref_rejected_logps: torch.Tensor,
    beta: float = 0.1,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Standard DPO loss. Inputs are summed response log-probs (B,).

    Returns ``(loss, chosen_reward, rejected_reward)`` where the implicit rewards
    ``beta * (policy_logp - ref_logp)`` are detached diagnostics.
    """
    pi_logratios = policy_chosen_logps - policy_rejected_logps
    ref_logratios = ref_chosen_logps - ref_rejected_logps
    logits = pi_logratios - ref_logratios
    loss = -F.logsigmoid(beta * logits).mean()
    chosen_reward = beta * (policy_chosen_logps - ref_chosen_logps).detach()
    rejected_reward = beta * (policy_rejected_logps - ref_rejected_logps).detach()
    return loss, chosen_reward, rejected_reward


def _log1mexp(x: torch.Tensor) -> torch.Tensor:
    """Numerically stable log(1 - exp(x)) for x < 0."""
    return torch.where(x > -0.6931, torch.log(-torch.expm1(x)), torch.log1p(-torch.exp(x)))


def orpo_loss(
    policy_chosen_logps: torch.Tensor,
    policy_rejected_logps: torch.Tensor,
    chosen_n_tokens: torch.Tensor,
    rejected_n_tokens: torch.Tensor,
    orpo_lambda: float = 1.0,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    ORPO (reference-free). Uses per-token MEAN log-probs.

    ``L = NLL(chosen) + lambda * -log sigmoid(log_odds_chosen - log_odds_rejected)``
    where ``log_odds = mean_logp - log(1 - exp(mean_logp))``.
    """
    chosen_mean = policy_chosen_logps / chosen_n_tokens.clamp(min=1)
    rejected_mean = policy_rejected_logps / rejected_n_tokens.clamp(min=1)
    log_odds = (chosen_mean - _log1mexp(chosen_mean)) - (rejected_mean - _log1mexp(rejected_mean))
    or_loss = -F.logsigmoid(log_odds).mean()
    nll = -chosen_mean.mean()
    loss = nll + orpo_lambda * or_loss
    # Implicit rewards for logging: the mean log-probs themselves.
    return loss, chosen_mean.detach(), rejected_mean.detach()


def kto_loss(
    policy_chosen_logps: torch.Tensor,
    policy_rejected_logps: torch.Tensor,
    ref_chosen_logps: torch.Tensor,
    ref_rejected_logps: torch.Tensor,
    beta: float = 0.1,
    desirable_weight: float = 1.0,
    undesirable_weight: float = 1.0,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    KTO from paired data: chosen = desirable, rejected = undesirable, with a reference-KL
    baseline estimated (detached) from the batch's mean log-ratio.
    """
    chosen_logratio = policy_chosen_logps - ref_chosen_logps
    rejected_logratio = policy_rejected_logps - ref_rejected_logps
    kl = torch.cat([chosen_logratio, rejected_logratio]).mean().clamp(min=0).detach()
    chosen_losses = 1.0 - torch.sigmoid(beta * (chosen_logratio - kl))
    rejected_losses = 1.0 - torch.sigmoid(beta * (kl - rejected_logratio))
    loss = (desirable_weight * chosen_losses).mean() + (undesirable_weight * rejected_losses).mean()
    return loss, (beta * chosen_logratio).detach(), (beta * rejected_logratio).detach()


def implicit_accuracy(chosen_reward: torch.Tensor, rejected_reward: torch.Tensor) -> torch.Tensor:
    """Fraction of pairs where the implicit (DPO/KTO/ORPO) reward prefers chosen."""
    return (chosen_reward > rejected_reward).float().mean()


Writing src/post_training/dpo.py


In [64]:
%%writefile src/post_training/evaluation.py
"""
Shared evaluation utilities for post-training stages.

The headline metric across the whole pipeline is GSM8K accuracy (greedy decoding) on a
fixed held-out set, so Base -> SFT -> DPO -> PPO -> GRPO can be compared on one axis.
``batched_generate`` length-buckets prompts so the mask-free model can decode many at once.
"""

from __future__ import annotations

import torch

from src.post_training.chat_template import EOT_ID, decode, encode_prompt
from src.post_training.rollout import generate_with_logprobs
from src.post_training.rewards import gsm8k_gold_answer, is_correct


def _model_context_length(model) -> int:
    inner = getattr(model, "transformer", model)
    return getattr(inner, "context_length", 1024)


@torch.no_grad()
def batched_generate(
    model,
    prompts: list[list[int]],
    max_new_tokens: int,
    *,
    device: str,
    temperature: float = 1.0,
    top_k: int | None = None,
    top_p: float | None = None,
    greedy: bool = False,
    micro_batch: int = 32,
) -> list[str]:
    """
    Generate a response string for each tokenized prompt, returned in input order.

    Prompts are grouped by identical length (the model has no padding-aware attention
    mask) and decoded in micro-batches. ``greedy=True`` forces argmax decoding (top_k=1),
    which is what we use for comparable eval numbers.
    """
    if greedy:
        temperature, top_k, top_p = 1.0, 1, None

    cap = _model_context_length(model)
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    results: list[str | None] = [None] * len(prompts)

    i = 0
    while i < len(order):
        j = i
        L = len(prompts[order[i]])
        # Gather a run of same-length prompts, up to micro_batch.
        while j < len(order) and len(prompts[order[j]]) == L and (j - i) < micro_batch:
            j += 1
        idxs = order[i:j]
        i = j
        budget = min(max_new_tokens, cap - L)
        if budget <= 0:  # prompt already fills the context; nothing to generate
            for k in idxs:
                results[k] = ""
            continue
        batch = torch.tensor([prompts[k] for k in idxs], dtype=torch.long, device=device)
        rb = generate_with_logprobs(model, batch, budget, temperature=temperature,
                                    top_k=top_k, top_p=top_p, stop_tokens=(EOT_ID,), pad_id=EOT_ID)
        gen = rb.sequences[:, L:]
        for row, k in enumerate(idxs):
            toks = gen[row].tolist()
            if EOT_ID in toks:
                toks = toks[: toks.index(EOT_ID)]
            results[k] = decode(toks)
    return [r or "" for r in results]


@torch.no_grad()
def gsm8k_accuracy(
    model,
    qa_pairs: list[tuple[str, str]],
    *,
    device: str,
    max_new_tokens: int = 300,
    greedy: bool = True,
    return_samples: int = 0,
) -> dict:
    """
    Compute greedy GSM8K accuracy over ``(question, answer_field)`` pairs.

    Returns ``{"accuracy", "n", "correct", "samples"}`` where ``samples`` holds a few
    (question, response, gold, correct) tuples for qualitative inspection.
    """
    prompts = [encode_prompt([{"role": "user", "content": q}]) for q, _ in qa_pairs]
    responses = batched_generate(model, prompts, max_new_tokens, device=device, greedy=greedy)

    correct = 0
    samples = []
    for (q, ans), resp in zip(qa_pairs, responses):
        gold = gsm8k_gold_answer(ans)
        ok = is_correct(resp, gold)
        correct += int(ok)
        if len(samples) < return_samples:
            samples.append({"q": q, "response": resp, "gold": gold, "correct": ok})
    n = len(qa_pairs)
    return {"accuracy": correct / max(1, n), "n": n, "correct": correct, "samples": samples}


def load_gsm8k_eval(split: str = "test", limit: int | None = 200) -> list[tuple[str, str]]:
    """Load ``(question, answer_field)`` pairs from GSM8K for evaluation."""
    import os
    os.environ.setdefault("HF_HOME", "/ephemeral/hf_cache")
    from datasets import load_dataset

    ds = load_dataset("openai/gsm8k", "main", split=split)
    if limit:
        ds = ds.select(range(min(limit, len(ds))))
    return [(ex["question"], ex["answer"]) for ex in ds]


Writing src/post_training/evaluation.py


In [65]:
%%writefile src/post_training/grpo.py
"""
GRPO (Group Relative Policy Optimization) core -- the DeepSeek-R1 RL algorithm.

GRPO drops PPO's value network. For each prompt it samples a GROUP of G completions,
scores them with a verifiable reward, and uses the group's own mean/std to compute a
relative advantage -- so the baseline is the group, not a learned critic. The update is a
token-level clipped surrogate plus a per-token KL penalty to the reference policy.
"""

from __future__ import annotations

import torch

from src.post_training.utils import masked_mean


def group_advantages(rewards: torch.Tensor, group_size: int, eps: float = 1e-4) -> torch.Tensor:
    """
    Group-relative advantage: ``(r - group_mean) / (group_std + eps)``.

    ``rewards`` is (num_prompts * group_size,) laid out group-contiguously (all G samples
    of prompt 0, then prompt 1, ...). Returns advantages of the same shape.
    """
    r = rewards.view(-1, group_size)
    mean = r.mean(dim=1, keepdim=True)
    std = r.std(dim=1, keepdim=True)
    adv = (r - mean) / (std + eps)
    return adv.reshape(-1)


def k3_kl(new_logp: torch.Tensor, ref_logp: torch.Tensor) -> torch.Tensor:
    """Per-token unbiased, non-negative KL estimator (Schulman's k3) for KL(policy||ref)."""
    diff = ref_logp - new_logp
    return torch.exp(diff) - diff - 1.0


def grpo_loss(
    new_logp: torch.Tensor,
    old_logp: torch.Tensor,
    ref_logp: torch.Tensor,
    advantages: torch.Tensor,
    resp_mask: torch.Tensor,
    clip: float = 0.2,
    kl_coef: float = 0.04,
) -> tuple[torch.Tensor, dict]:
    """
    Token-level clipped surrogate + KL penalty.

    Args:
        new_logp/old_logp/ref_logp: (B, L) per-token log-probs (policy / sampling / ref).
        advantages: (B,) one scalar per completion, broadcast over its tokens.
        resp_mask:  (B, L) bool over response tokens.

    Returns:
        (loss, stats) with mean KL and clip fraction for logging.
    """
    adv = advantages[:, None]
    ratio = torch.exp(new_logp - old_logp)
    surr1 = ratio * adv
    surr2 = torch.clamp(ratio, 1.0 - clip, 1.0 + clip) * adv
    surrogate = torch.min(surr1, surr2)
    kl = k3_kl(new_logp, ref_logp)

    per_token = surrogate - kl_coef * kl
    loss = -masked_mean(per_token, resp_mask)
    stats = {
        "kl": masked_mean(kl, resp_mask).item(),
        "clipfrac": masked_mean(((ratio - 1.0).abs() > clip).float(), resp_mask).item(),
    }
    return loss, stats


Writing src/post_training/grpo.py


In [66]:
%%writefile src/post_training/inference.py
"""
Inference helpers shared by the chat CLI and the eval scripts.

Loads any stage checkpoint (base / sft / dpo / ppo / grpo) by reading the model
dimensions from the checkpoint's stored ``cfg`` (so you never have to repeat them), and
generates a reply either in chat-template form (for instruction-tuned models) or as raw
continuation (for the base model).
"""

from __future__ import annotations

import torch

from src.models.transformer import Transformer
from src.post_training.chat_template import EOT_ID, decode, encode_prompt, get_tokenizer
from src.post_training.evaluation import batched_generate


def load_model_from_ckpt(ckpt_path: str, device: str, overrides: dict | None = None) -> Transformer:
    """Build a :class:`Transformer` from a checkpoint's stored cfg and load its backbone
    weights (tolerates ``module.``/``transformer.`` prefixes and reward-head extras)."""
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    cfg = {**(ck.get("cfg") or {}), **(overrides or {})}
    model = Transformer(
        n_head=cfg.get("n_head", 16), n_embed=cfg.get("n_embed", 1024),
        context_length=cfg.get("context_length", 1024), vocab_size=cfg.get("vocab_size", 50304),
        N_BLOCKS=cfg.get("n_blocks", 24),
    )
    state = ck["model_state_dict"] if "model_state_dict" in ck else ck
    state = {k.removeprefix("module.").removeprefix("transformer."): v for k, v in state.items()}
    keys = set(model.state_dict().keys())
    model.load_state_dict({k: v for k, v in state.items() if k in keys}, strict=False)
    return model.to(device).eval()


@torch.no_grad()
def generate_reply(
    model,
    user_text: str,
    *,
    device: str,
    system: str | None = None,
    raw: bool = False,
    max_new_tokens: int = 256,
    temperature: float = 0.8,
    top_k: int | None = None,
    top_p: float | None = 0.95,
    greedy: bool = False,
) -> str:
    """
    Generate a response to ``user_text``.

    - chat mode (default): wraps the prompt in the chat template (optionally with a
      ``system`` message) and returns the decoded assistant turn.
    - raw mode (``raw=True``): treats ``user_text`` as a prefix and returns the base
      model's continuation (no chat template) -- the right mode for the pretrained base.
    """
    if raw:
        ids = get_tokenizer().encode_ordinary(user_text)
        out = batched_generate(model, [ids], max_new_tokens, device=device, temperature=temperature,
                               top_k=top_k, top_p=top_p, greedy=greedy)
        return out[0]

    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": user_text})
    prompt_ids = encode_prompt(messages)
    out = batched_generate(model, [prompt_ids], max_new_tokens, device=device, temperature=temperature,
                           top_k=top_k, top_p=top_p, greedy=greedy)
    return out[0]


Writing src/post_training/inference.py


In [67]:
%%writefile src/post_training/logging_utils.py
"""
Lightweight metrics logging shared by every training stage.

Always writes one JSON object per logged step to a JSONL file under the log dir (so runs
are reproducible and plottable without any external service). Optionally mirrors to
Weights & Biases when ``use_wandb=True`` and the package is importable. Only the main
process should construct/use a logger (the trainers guard this).
"""

from __future__ import annotations

import json
import os
import time
from typing import Any


class MetricsLogger:
    def __init__(
        self,
        stage: str,
        log_dir: str,
        *,
        use_wandb: bool = False,
        wandb_project: str = "train-llm-from-scratch-posttrain",
        config: dict | None = None,
        run_name: str | None = None,
    ) -> None:
        os.makedirs(log_dir, exist_ok=True)
        self.stage = stage
        # No Date.now in scripts here; use time.time() (allowed) for unique filenames.
        stamp = int(time.time())
        self.path = os.path.join(log_dir, f"{stage}_{stamp}.jsonl")
        self._fh = open(self.path, "a")
        self._wandb = None
        if use_wandb:
            try:
                import wandb

                wandb.init(project=wandb_project, name=run_name or f"{stage}-{stamp}", config=config or {})
                self._wandb = wandb
            except Exception as e:  # noqa: BLE001 - logging must never crash training
                print(f"[logger] wandb disabled ({e}); JSONL logging only -> {self.path}")
        print(f"[logger] stage={stage} -> {self.path}")

    def log(self, step: int, metrics: dict[str, Any]) -> None:
        record = {"step": step, "wall": time.time(), **metrics}
        self._fh.write(json.dumps(record) + "\n")
        self._fh.flush()
        if self._wandb is not None:
            self._wandb.log(metrics, step=step)

    def close(self) -> None:
        try:
            self._fh.close()
        finally:
            if self._wandb is not None:
                self._wandb.finish()


Writing src/post_training/logging_utils.py


In [68]:
%%writefile src/post_training/optim.py
"""
Optimizer and learning-rate-schedule helpers shared across stages.

- ``configure_optimizer`` builds AdamW with the standard weight-decay split (decay the 2D
  weight matrices, don't decay biases / LayerNorm / embeddings).
- ``cosine_lr`` is a linear-warmup + cosine-decay schedule returning the LR for a step.
"""

from __future__ import annotations

import math

import torch
import torch.nn as nn


def configure_optimizer(
    model: nn.Module,
    lr: float,
    weight_decay: float,
    betas: tuple[float, float] = (0.9, 0.95),
) -> torch.optim.AdamW:
    """AdamW with weight decay applied only to >=2D parameters (matrices), not to
    biases / norms / 1D params. Standard GPT recipe."""
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.dim() >= 2:
            decay.append(p)
        else:
            no_decay.append(p)
    groups = [
        {"params": decay, "weight_decay": weight_decay},
        {"params": no_decay, "weight_decay": 0.0},
    ]
    return torch.optim.AdamW(groups, lr=lr, betas=betas)


def cosine_lr(step: int, *, warmup_steps: int, max_steps: int, lr: float, min_lr: float) -> float:
    """Linear warmup to ``lr`` over ``warmup_steps``, then cosine decay to ``min_lr`` by
    ``max_steps`` (constant ``min_lr`` afterwards)."""
    if step < warmup_steps:
        return lr * (step + 1) / max(1, warmup_steps)
    if step >= max_steps:
        return min_lr
    progress = (step - warmup_steps) / max(1, max_steps - warmup_steps)
    coeff = 0.5 * (1.0 + math.cos(math.pi * progress))
    return min_lr + coeff * (lr - min_lr)


Writing src/post_training/optim.py


In [69]:
%%writefile src/post_training/ppo.py
"""
PPO core (the classic InstructGPT RLHF update), written from scratch.

The orchestration (rollout -> score -> GAE -> clipped epochs) lives in
``scripts/train_ppo.py``; this module holds the reusable, testable pieces:

- ``compute_gae``      : Generalized Advantage Estimation over response tokens.
- ``whiten``           : advantage normalization (masked).
- ``ppo_policy_loss``  : clipped surrogate objective.
- ``ppo_value_loss``   : clipped value-function loss.

All tensors are in the "action frame" of length ``L = T-1``: index ``t`` corresponds to
producing token ``t+1``, with state-value ``V(s_t)``, log-prob of the produced token, and
the reward received for that action. ``resp_mask`` marks which actions are response tokens.
"""

from __future__ import annotations

import torch

from src.post_training.utils import masked_mean


def compute_gae(
    rewards: torch.Tensor,
    values: torch.Tensor,
    values_next: torch.Tensor,
    resp_mask: torch.Tensor,
    gamma: float = 1.0,
    lam: float = 0.95,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Generalized Advantage Estimation over the response.

    Args (all (B, L) in the action frame; ``resp_mask`` bool):
        rewards:     per-action reward (KL penalty per token + task reward at the last
                     response token).
        values:      V(s_t).
        values_next: V(s_{t+1}).
        resp_mask:   True where action t is a response token. The episode is terminal
                     after the last response token (no bootstrap past it).

    Returns:
        (advantages, returns) each (B, L), zeroed outside the response.
    """
    B, L = rewards.shape
    adv = torch.zeros_like(rewards)
    lastgae = torch.zeros(B, device=rewards.device)
    m = resp_mask.float()
    for t in reversed(range(L)):
        # Bootstrap only if the NEXT action is still a response action.
        nonterminal = m[:, t + 1] if t + 1 < L else torch.zeros(B, device=rewards.device)
        delta = rewards[:, t] + gamma * values_next[:, t] * nonterminal - values[:, t]
        lastgae = delta + gamma * lam * nonterminal * lastgae
        adv[:, t] = lastgae
    returns = adv + values
    return adv * m, returns * m


def whiten(advantages: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """Normalize advantages to zero mean / unit std over masked (response) positions."""
    m = mask.float()
    mean = masked_mean(advantages, m)
    var = masked_mean((advantages - mean) ** 2, m)
    return ((advantages - mean) / (var.sqrt() + 1e-8)) * m


def ppo_policy_loss(
    new_logp: torch.Tensor,
    old_logp: torch.Tensor,
    advantages: torch.Tensor,
    mask: torch.Tensor,
    clip: float = 0.2,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Clipped surrogate policy loss. Returns (loss, clip_fraction)."""
    ratio = torch.exp(new_logp - old_logp)
    surr1 = ratio * advantages
    surr2 = torch.clamp(ratio, 1.0 - clip, 1.0 + clip) * advantages
    loss = -masked_mean(torch.min(surr1, surr2), mask)
    clipped = ((ratio - 1.0).abs() > clip).float()
    return loss, masked_mean(clipped, mask)


def ppo_value_loss(
    new_values: torch.Tensor,
    old_values: torch.Tensor,
    returns: torch.Tensor,
    mask: torch.Tensor,
    vf_clip: float = 0.2,
) -> torch.Tensor:
    """Clipped value-function loss (0.5 * max of clipped/unclipped squared error)."""
    v_clipped = old_values + torch.clamp(new_values - old_values, -vf_clip, vf_clip)
    loss_unclipped = (new_values - returns) ** 2
    loss_clipped = (v_clipped - returns) ** 2
    return 0.5 * masked_mean(torch.max(loss_unclipped, loss_clipped), mask)


def approx_kl(new_logp: torch.Tensor, old_logp: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """Mean approximate KL(old || new) over response tokens (a PPO health metric)."""
    return masked_mean(old_logp - new_logp, mask)


Writing src/post_training/ppo.py


In [70]:
%%writefile src/post_training/reward_model.py
"""
Reward model: a scalar reward head on top of a :class:`Transformer` backbone.

Following InstructGPT, the reward of a full sequence is read off the hidden state of its
**last real token**. We train it with the Bradley-Terry pairwise loss (see
``reward_train.py``) on (chosen, rejected) preference pairs. The trained RM then provides
the scalar reward signal for PPO.
"""

from __future__ import annotations

import torch
import torch.nn as nn

import torch as _torch

from src.models.transformer import Transformer
from src.post_training.utils import build_model_from_config, gather_last, unwrap


def load_reward_model(cfg, ckpt_path: str, device: str) -> "RewardModel":
    """Reconstruct a trained :class:`RewardModel` (backbone + reward head) from a reward
    checkpoint saved by ``scripts/train_reward.py``."""
    backbone = build_model_from_config(cfg)
    rm = RewardModel(backbone)
    ck = _torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state = ck["model_state_dict"] if "model_state_dict" in ck else ck
    if any(k.startswith("module.") for k in state):
        state = {k.removeprefix("module."): v for k, v in state.items()}
    rm.load_state_dict(state, strict=True)
    rm.to(device).eval()
    for p in rm.parameters():
        p.requires_grad_(False)
    return rm


class RewardModel(nn.Module):
    """Wrap a :class:`Transformer` and add a scalar reward head (no ``lm_head`` used)."""

    def __init__(self, transformer: Transformer) -> None:
        super().__init__()
        self.transformer = transformer
        n_embed = transformer.lm_head.in_features
        self.reward_head = nn.Linear(n_embed, 1, bias=False)
        nn.init.zeros_(self.reward_head.weight)  # start near-zero rewards

    @property
    def context_length(self) -> int:
        return self.transformer.context_length

    def token_rewards(self, idx: torch.Tensor) -> torch.Tensor:
        """Per-token scalar reward (B, T) -- useful for diagnostics / dense shaping."""
        hidden = self.transformer.forward_hidden(idx)
        return self.reward_head(hidden).squeeze(-1)

    def forward(self, idx: torch.Tensor, seq_lengths: torch.Tensor | None = None) -> torch.Tensor:
        """
        Scalar reward per sequence (B,).

        Args:
            idx: (B, T) token ids.
            seq_lengths: (B,) number of real (non-pad) tokens per row. If None, uses the
                full length T (assumes no padding).
        """
        rewards = self.token_rewards(idx)  # (B, T)
        if seq_lengths is None:
            return rewards[:, -1]
        return gather_last(rewards, seq_lengths)


Writing src/post_training/reward_model.py


In [71]:
%%writefile src/post_training/reward_train.py
"""
Reward-model training objective: the Bradley-Terry pairwise loss used by InstructGPT.

Given scalar rewards for a preferred (``chosen``) and dispreferred (``rejected``) response
to the same prompt, the model is trained so the chosen reward exceeds the rejected one:

    L = -log sigmoid(r_chosen - r_rejected)

Preference accuracy (fraction with r_chosen > r_rejected) is the headline eval metric.
"""

from __future__ import annotations

import torch
import torch.nn.functional as F


def bradley_terry_loss(chosen_rewards: torch.Tensor, rejected_rewards: torch.Tensor) -> torch.Tensor:
    """Mean ``-log sigmoid(chosen - rejected)`` over a batch of preference pairs."""
    return -F.logsigmoid(chosen_rewards - rejected_rewards).mean()


def preference_accuracy(chosen_rewards: torch.Tensor, rejected_rewards: torch.Tensor) -> torch.Tensor:
    """Fraction of pairs where the model scores the chosen response higher."""
    return (chosen_rewards > rejected_rewards).float().mean()


def reward_margin(chosen_rewards: torch.Tensor, rejected_rewards: torch.Tensor) -> torch.Tensor:
    """Mean reward gap (chosen - rejected); a useful training diagnostic."""
    return (chosen_rewards - rejected_rewards).mean()


Writing src/post_training/reward_train.py


In [72]:
%%writefile src/post_training/rewards/__init__.py
"""Verifiable (programmatic) reward functions for RLVR -- GSM8K math + arithmetic warm-up."""

from src.post_training.rewards.parsing import (
    extract_answer,
    extract_think,
    parse_number,
    gsm8k_gold_answer,
)
from src.post_training.rewards.verifiers import (
    reward_gsm8k,
    reward_arithmetic,
    reward_format,
    is_correct,
)

__all__ = [
    "extract_answer",
    "extract_think",
    "parse_number",
    "gsm8k_gold_answer",
    "reward_gsm8k",
    "reward_arithmetic",
    "reward_format",
    "is_correct",
]


Writing src/post_training/rewards/__init__.py


In [73]:
%%writefile src/post_training/rewards/parsing.py
"""
Tolerant parsing of model outputs and gold answers for verifiable rewards.

The model is trained (via SFT) to answer in the form::

    <think> step by step reasoning ... </think><answer>42</answer>

but a small model is inconsistent, so parsing falls back gracefully: prefer an
``<answer>`` tag, then a GSM8K-style ``#### N``, then the last number in the text.
"""

from __future__ import annotations

import re

from src.post_training.chat_template import (
    ANSWER_OPEN, ANSWER_CLOSE, THINK_OPEN, THINK_CLOSE,
)

# A number: optional sign, digits with optional thousands commas, optional decimal.
_NUMBER_RE = re.compile(r"-?\$?\d[\d,]*(?:\.\d+)?")
_ANSWER_RE = re.compile(re.escape(ANSWER_OPEN) + r"(.*?)" + re.escape(ANSWER_CLOSE), re.DOTALL)
_THINK_RE = re.compile(re.escape(THINK_OPEN) + r"(.*?)" + re.escape(THINK_CLOSE), re.DOTALL)
_HASH_RE = re.compile(r"####\s*(-?\$?\d[\d,]*(?:\.\d+)?)")


def parse_number(s: str | None) -> float | None:
    """Parse the first number out of a string, tolerating ``$`` and thousands commas."""
    if s is None:
        return None
    m = _NUMBER_RE.search(s)
    if not m:
        return None
    raw = m.group(0).replace(",", "").replace("$", "")
    try:
        return float(raw)
    except ValueError:
        return None


def extract_think(text: str) -> str | None:
    """Return the content of the first ``<think>...</think>`` block, if any."""
    m = _THINK_RE.search(text)
    return m.group(1).strip() if m else None


def extract_answer(text: str) -> float | None:
    """
    Extract the model's final numeric answer with graceful fallback:
    1) inside an ``<answer>...</answer>`` tag, else
    2) after a GSM8K-style ``#### N``, else
    3) the last number appearing anywhere in the text.
    """
    m = _ANSWER_RE.search(text)
    if m:
        n = parse_number(m.group(1))
        if n is not None:
            return n
    m = _HASH_RE.search(text)
    if m:
        n = parse_number(m.group(1))
        if n is not None:
            return n
    nums = _NUMBER_RE.findall(text)
    if nums:
        return parse_number(nums[-1])
    return None


def gsm8k_gold_answer(answer_field: str) -> float | None:
    """Extract the numeric gold answer from a GSM8K ``answer`` field (ends in ``#### N``)."""
    m = _HASH_RE.search(answer_field)
    if m:
        return parse_number(m.group(1))
    return parse_number(answer_field)


def has_well_formed_answer(text: str) -> bool:
    """True iff there is exactly one well-formed ``<answer>...</answer>`` block."""
    return len(_ANSWER_RE.findall(text)) == 1


Writing src/post_training/rewards/parsing.py


In [74]:
%%writefile src/post_training/rewards/verifiers.py
"""
Verifiable reward functions (RLVR). Each maps a decoded model response (+ a gold answer)
to a scalar reward. The design is deliberately **correctness-dominant** with only a small,
bounded format bonus, to limit reward hacking (a tiny model will happily emit empty
``<answer></answer>`` tags or repeat tokens if the format bonus is too large).

Reward decomposition (clipped to [0, 1.2]):
    +1.0  if the parsed final answer matches the gold answer (exact, with float tolerance)
    +0.2  if there is exactly one well-formed <answer>...</answer> block (format shaping)
"""

from __future__ import annotations

import math

from src.post_training.rewards.parsing import extract_answer, has_well_formed_answer

CORRECT_BONUS = 1.0
FORMAT_BONUS = 0.2
REWARD_CLIP = 1.2
FLOAT_TOL = 1e-4


def _answers_match(pred: float | None, gold: float | None) -> bool:
    if pred is None or gold is None:
        return False
    return math.isclose(pred, gold, rel_tol=0.0, abs_tol=FLOAT_TOL)


def is_correct(text: str, gold: float | None) -> bool:
    """Whether the model's parsed answer matches the gold answer (used for accuracy eval)."""
    return _answers_match(extract_answer(text), gold)


def reward_gsm8k(text: str, gold: float | None) -> float:
    """Reward for a GSM8K response: correctness (dominant) + small format bonus, clipped."""
    r = 0.0
    if _answers_match(extract_answer(text), gold):
        r += CORRECT_BONUS
    if has_well_formed_answer(text):
        r += FORMAT_BONUS
    return min(r, REWARD_CLIP)


# Arithmetic warm-up uses the same logic (gold is a single number).
reward_arithmetic = reward_gsm8k


def reward_format(text: str) -> float:
    """Pure format reward (no gold): 1.0 if exactly one well-formed answer block else 0."""
    return 1.0 if has_well_formed_answer(text) else 0.0


Writing src/post_training/rewards/verifiers.py


In [75]:
%%writefile src/post_training/rollout.py
"""
Generation + per-token log-probability utilities -- the shared core of every RL
algorithm here (PPO and GRPO).

These are FREE FUNCTIONS rather than methods on ``Transformer`` on purpose: the exact
same log-prob math has to be run against four different parameter sets during PPO/GRPO
(the trainable policy, the frozen reference, the old-policy snapshot, and -- for the
value head -- the actor-critic wrapper), so ``f(model, ...)`` composes far better than a
bound method and keeps the educational model file untouched.

Two key constraints from the repo's model:
- Learned absolute positions cap any sequence at ``context_length``; we enforce
  ``prompt_len + max_new_tokens <= context_length``.
- There is no padding-aware attention mask, so callers pass **equal-length** prompts
  (length-bucketed). Padding only ever appears AFTER a row hits its stop token, and the
  ``response_mask`` makes the loss ignore it.

Numerical note: log-probs are always taken in fp32 (``logits.float()``) even under bf16
autocast, because PPO/GRPO/DPO subtract log-probs and bf16 rounding there is harmful.
"""

from __future__ import annotations

from dataclasses import dataclass

import torch
import torch.nn.functional as F

from src.post_training.chat_template import EOT_ID


@dataclass
class RolloutBatch:
    """Result of :func:`generate_with_logprobs`.

    Attributes:
        sequences:     (B, P+G) prompt tokens followed by generated tokens.
        response_mask: (B, P+G) bool. True only on generated positions that are real
                       (not post-stop padding); always False over the prompt.
        gen_logprobs:  (B, G) log-prob of each generated token under the sampling policy
                       (full-distribution log-prob at the sampling temperature). 0 on
                       padded/finished positions.
        prompt_len:    int P, the (shared) prompt length.
    """

    sequences: torch.Tensor
    response_mask: torch.Tensor
    gen_logprobs: torch.Tensor
    prompt_len: int


def _logits_from(model, idx: torch.Tensor) -> torch.Tensor:
    """Get logits whether ``model`` returns ``(logits, loss)`` (Transformer) or
    ``(logits, values)`` (value-head wrapper) or a bare logits tensor."""
    out = model(idx)
    return out[0] if isinstance(out, tuple) else out


def filter_logits(
    logits: torch.Tensor,
    temperature: float = 1.0,
    top_k: int | None = None,
    top_p: float | None = None,
) -> torch.Tensor:
    """
    Apply temperature, then optional top-k and nucleus (top-p) filtering to a batch of
    next-token logits ``(B, vocab)``. Returns logits with filtered entries set to -inf,
    ready to softmax-and-sample. Used ONLY to build the sampling distribution; the
    log-prob we record for the ratio is the full-distribution log-prob (see
    :func:`generate_with_logprobs`).
    """
    if temperature != 1.0:
        logits = logits / max(temperature, 1e-6)

    if top_k is not None and top_k > 0:
        k = min(top_k, logits.size(-1))
        kth = torch.topk(logits, k, dim=-1).values[..., -1, None]
        logits = logits.masked_fill(logits < kth, float("-inf"))

    if top_p is not None and 0.0 < top_p < 1.0:
        sorted_logits, sorted_idx = torch.sort(logits, descending=True, dim=-1)
        cumprobs = sorted_logits.softmax(dim=-1).cumsum(dim=-1)
        remove = cumprobs > top_p
        # Keep at least the top token; shift so the token that crosses top_p stays.
        remove[..., 1:] = remove[..., :-1].clone()
        remove[..., 0] = False
        remove = remove.scatter(-1, sorted_idx, remove)
        logits = logits.masked_fill(remove, float("-inf"))

    return logits


@torch.no_grad()
def generate_with_logprobs(
    model,
    prompt_ids: torch.Tensor,
    max_new_tokens: int,
    *,
    temperature: float = 1.0,
    top_k: int | None = None,
    top_p: float | None = None,
    stop_tokens: tuple[int, ...] = (EOT_ID,),
    pad_id: int = EOT_ID,
    context_length: int | None = None,
) -> RolloutBatch:
    """
    Autoregressively sample a completion for each prompt and record per-token log-probs.

    ``prompt_ids`` must be (B, P) with a shared length P (length-bucket your prompts).
    Rows stop individually once they emit any ``stop_tokens``; subsequent positions are
    filled with ``pad_id`` and excluded from ``response_mask``.

    No KV cache (kept for clarity): each step re-runs the prefix, which is fine for the
    short sequences here on an H100.
    """
    model.eval()
    device = prompt_ids.device
    B, P = prompt_ids.shape

    cap = context_length if context_length is not None else getattr(model, "context_length", P + max_new_tokens)
    cap = getattr(getattr(model, "transformer", model), "context_length", cap)
    max_new_tokens = min(max_new_tokens, cap - P)
    if max_new_tokens <= 0:
        raise ValueError(f"Prompt length {P} leaves no room under context_length {cap}.")

    stop_set = torch.tensor(stop_tokens, device=device)
    sequences = prompt_ids.clone()
    gen_logprobs = torch.zeros(B, max_new_tokens, device=device)
    gen_mask = torch.zeros(B, max_new_tokens, dtype=torch.bool, device=device)
    finished = torch.zeros(B, dtype=torch.bool, device=device)

    for t in range(max_new_tokens):
        idx_cond = sequences[:, -cap:]
        logits = _logits_from(model, idx_cond)[:, -1, :]  # (B, vocab)

        # Full-distribution log-prob at the sampling temperature (matches recompute).
        full_logprobs = F.log_softmax(logits.float() / max(temperature, 1e-6), dim=-1)
        # Separate truncated distribution to actually sample from.
        probs = F.softmax(filter_logits(logits, temperature, top_k, top_p), dim=-1)
        next_tok = torch.multinomial(probs, num_samples=1).squeeze(-1)  # (B,)

        active = ~finished
        token_lp = full_logprobs.gather(-1, next_tok[:, None]).squeeze(-1)
        # Finished rows emit padding and contribute nothing.
        next_tok = torch.where(active, next_tok, torch.full_like(next_tok, pad_id))
        gen_logprobs[:, t] = torch.where(active, token_lp, torch.zeros_like(token_lp))
        gen_mask[:, t] = active

        sequences = torch.cat([sequences, next_tok[:, None]], dim=1)
        # A row is finished AFTER (and including) the step where it emits a stop token.
        hit_stop = (next_tok[:, None] == stop_set).any(dim=-1)
        finished = finished | (active & hit_stop)

        if bool(finished.all()):
            # Pad remaining columns so shapes stay (B, max_new_tokens).
            if t + 1 < max_new_tokens:
                rem = max_new_tokens - (t + 1)
                sequences = torch.cat(
                    [sequences, torch.full((B, rem), pad_id, dtype=sequences.dtype, device=device)], dim=1
                )
            break

    response_mask = torch.cat(
        [torch.zeros(B, P, dtype=torch.bool, device=device), gen_mask], dim=1
    )
    return RolloutBatch(
        sequences=sequences,
        response_mask=response_mask,
        gen_logprobs=gen_logprobs,
        prompt_len=P,
    )


def _cap_of(model) -> int:
    inner = getattr(model, "transformer", model)
    return getattr(inner, "context_length", 1024)


@torch.no_grad()
def rollout_prompts(
    model,
    prompts: list[list[int]],
    max_new_tokens: int,
    *,
    device: str,
    temperature: float = 1.0,
    top_k: int | None = None,
    top_p: float | None = None,
    pad_id: int = EOT_ID,
    micro_batch: int = 64,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Generate one completion per (possibly variable-length) tokenized prompt and merge the
    results into padded tensors. Length-buckets prompts (the model has no padding-aware
    attention mask) and pads all rollouts to a common width.

    Returns ``(sequences, response_mask, prompt_lens)``:
        sequences:     (N, Tmax) prompt+response, right-padded with ``pad_id``.
        response_mask: (N, Tmax) bool, True on real generated tokens.
        prompt_lens:   (N,) prompt length of each row.
    """
    cap = _cap_of(model)
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    pieces = []  # (orig_idxs, sequences, response_mask, prompt_len)
    i = 0
    while i < len(order):
        L = len(prompts[order[i]])
        j = i
        while j < len(order) and len(prompts[order[j]]) == L and (j - i) < micro_batch:
            j += 1
        idxs = order[i:j]
        i = j
        budget = max(1, min(max_new_tokens, cap - L))
        batch = torch.tensor([prompts[k] for k in idxs], dtype=torch.long, device=device)
        rb = generate_with_logprobs(model, batch, budget, temperature=temperature,
                                    top_k=top_k, top_p=top_p, stop_tokens=(EOT_ID,), pad_id=pad_id)
        pieces.append((idxs, rb.sequences, rb.response_mask, L))

    N = len(prompts)
    Tmax = max(seq.size(1) for _, seq, _, _ in pieces)
    sequences = torch.full((N, Tmax), pad_id, dtype=torch.long, device=device)
    response_mask = torch.zeros((N, Tmax), dtype=torch.bool, device=device)
    prompt_lens = torch.zeros(N, dtype=torch.long, device=device)
    for idxs, seq, rmask, L in pieces:
        t = seq.size(1)
        for row, k in enumerate(idxs):
            sequences[k, :t] = seq[row]
            response_mask[k, :t] = rmask[row]
            prompt_lens[k] = L
    return sequences, response_mask, prompt_lens


def compute_logprobs(
    model,
    sequences: torch.Tensor,
    response_mask: torch.Tensor,
    *,
    temperature: float = 1.0,
    requires_grad: bool = True,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Teacher-forced recomputation of per-token log-probs of ``sequences`` under ``model``.

    Mirrors the model's training shift: ``logits[:, t]`` predicts ``sequences[:, t+1]``,
    so the returned tensors have length ``T-1`` and are aligned to target positions
    ``1..T-1``.

    Args:
        sequences:     (B, T) token ids (prompt + response).
        response_mask: (B, T) bool over response positions (as from :class:`RolloutBatch`).
        temperature:   divide logits by this before log-softmax, matching sampling.
        requires_grad: if False, runs under ``no_grad`` (use for ref / old-policy).

    Returns:
        (logprobs, mask) each (B, T-1). ``logprobs`` is the log-prob of the realized
        next token; ``mask`` is the response mask shifted to align with those targets.
    """
    ctx = torch.enable_grad() if requires_grad else torch.no_grad()
    with ctx:
        logits = _logits_from(model, sequences)[:, :-1, :]  # predict tokens 1..T-1
        logprobs_all = F.log_softmax(logits.float() / max(temperature, 1e-6), dim=-1)
        targets = sequences[:, 1:].unsqueeze(-1)
        logprobs = logprobs_all.gather(-1, targets).squeeze(-1)  # (B, T-1)
    mask = response_mask[:, 1:].to(torch.bool)
    return logprobs, mask


def sequence_logprobs(
    model,
    sequences: torch.Tensor,
    response_mask: torch.Tensor,
    *,
    temperature: float = 1.0,
    requires_grad: bool = True,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Sequence-level summed log-prob over response tokens (used by DPO/KTO/ORPO).

    Returns ``(sum_logprob, n_tokens)`` each shape (B,). The per-token mean is
    ``sum_logprob / n_tokens.clamp(min=1)``.
    """
    lp, mask = compute_logprobs(model, sequences, response_mask, temperature=temperature, requires_grad=requires_grad)
    m = mask.to(lp.dtype)
    return (lp * m).sum(dim=-1), m.sum(dim=-1)


def sequence_entropy(model, sequences: torch.Tensor, temperature: float = 1.0) -> torch.Tensor:
    """Per-token predictive entropy ``(B, T-1)`` of ``model`` on ``sequences`` (for an
    optional PPO entropy bonus / diagnostics)."""
    with torch.no_grad():
        logits = _logits_from(model, sequences)[:, :-1, :]
        logp = F.log_softmax(logits.float() / max(temperature, 1e-6), dim=-1)
        return -(logp.exp() * logp).sum(dim=-1)


Writing src/post_training/rollout.py


In [76]:
%%writefile src/post_training/sft.py
"""
Supervised Fine-Tuning (SFT) core: a prompt-masked language-modeling loss and a
sequence-packing utility.

The only difference from ordinary next-token pretraining is the per-token ``loss_mask``:
we train the model to produce the *assistant* tokens (mask=1) but not to reproduce the
prompt / role markers (mask=0). The mask comes from
:func:`src.post_training.chat_template.encode_chat`.
"""

from __future__ import annotations

import numpy as np
import torch
import torch.nn.functional as F


def sft_loss(logits: torch.Tensor, tokens: torch.Tensor, loss_mask: torch.Tensor) -> torch.Tensor:
    """
    Masked next-token cross-entropy.

    Args:
        logits:    (B, T, V) model logits for ``tokens``.
        tokens:    (B, T) packed token ids.
        loss_mask: (B, T) 1 on tokens that should be predicted (assistant completions).

    Returns:
        Scalar loss averaged over masked target positions.
    """
    # Predict token t+1 from position t (same shift the base model uses).
    logits = logits[:, :-1, :]
    targets = tokens[:, 1:]
    mask = loss_mask[:, 1:].to(logits.dtype)

    V = logits.size(-1)
    ce = F.cross_entropy(logits.reshape(-1, V).float(), targets.reshape(-1).long(), reduction="none")
    ce = ce.view(targets.shape) * mask
    return ce.sum() / mask.sum().clamp(min=1.0)


def pack_examples(
    examples: list[tuple[list[int], list[int]]],
    context_length: int,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Pack a list of ``(ids, loss_mask)`` examples into fixed-length ``context_length``
    rows by concatenating them and slicing. Examples are already EOT-terminated by the
    chat template, so the EOT acts as the separator between packed examples. Trailing
    tokens that don't fill a full row are dropped.

    Returns:
        (tokens, masks) int arrays of shape (N, context_length).
    """
    flat_ids: list[int] = []
    flat_mask: list[int] = []
    for ids, mask in examples:
        flat_ids.extend(ids)
        flat_mask.extend(mask)

    n_rows = len(flat_ids) // context_length
    ids_arr = np.asarray(flat_ids[: n_rows * context_length], dtype=np.int32).reshape(n_rows, context_length)
    mask_arr = np.asarray(flat_mask[: n_rows * context_length], dtype=np.int8).reshape(n_rows, context_length)
    return ids_arr, mask_arr


Writing src/post_training/sft.py


In [77]:
%%writefile src/post_training/utils.py
"""
Shared helpers for the post-training stack: model construction from configs,
frozen reference/old-policy copies, checkpoint I/O (keeping the repo's existing
checkpoint shape), masked reductions, and seeding.
"""

from __future__ import annotations

import contextlib
import copy
import os
import random
from dataclasses import asdict, is_dataclass
from typing import Any

import numpy as np
import torch
import torch.nn as nn

from src.models.transformer import Transformer


def amp_autocast(amp_dtype: str | None, device: str):
    """Return a bf16 autocast context on CUDA when requested, else a no-op context.

    bf16 needs no GradScaler (unlike fp16), which keeps the training loops clean.
    """
    if amp_dtype == "bf16" and str(device).startswith("cuda") and torch.cuda.is_available():
        return torch.autocast(device_type="cuda", dtype=torch.bfloat16)
    return contextlib.nullcontext()


# --- Reproducibility ---------------------------------------------------------

def set_seed(seed: int) -> None:
    """Seed python / numpy / torch (incl. CUDA) for reproducible runs."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# --- Model construction ------------------------------------------------------

def _cfg_get(cfg: Any, key: str) -> Any:
    """Read a field from either a dataclass/object (attr) or a dict."""
    if isinstance(cfg, dict):
        return cfg[key]
    return getattr(cfg, key)


def build_model_from_config(cfg: Any) -> Transformer:
    """
    Construct a fresh :class:`Transformer` from a config carrying the standard keys
    ``n_head, n_embed, context_length, vocab_size, n_blocks``. Works with the new
    post-training dataclasses and with the legacy ``default_config`` dict.
    """
    return Transformer(
        n_head=_cfg_get(cfg, "n_head"),
        n_embed=_cfg_get(cfg, "n_embed"),
        context_length=_cfg_get(cfg, "context_length"),
        vocab_size=_cfg_get(cfg, "vocab_size"),
        N_BLOCKS=_cfg_get(cfg, "n_blocks"),
    )


def _strip_ddp_prefix(state_dict: dict) -> dict:
    """Remove a leading ``module.`` from keys saved by DistributedDataParallel."""
    if any(k.startswith("module.") for k in state_dict):
        return {k.removeprefix("module."): v for k, v in state_dict.items()}
    return state_dict


def load_backbone_from_ckpt(cfg: Any, ckpt_path: str, device: str) -> Transformer:
    """
    Build a Transformer from ``cfg`` and load backbone weights from a checkpoint saved
    by the pretraining script or any post-training stage (``model_state_dict`` key).
    DDP ``module.`` prefixes are stripped. Auxiliary head weights (value/reward), if
    present, are ignored here -- wrappers add their own fresh heads.
    """
    model = build_model_from_config(cfg)
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt
    state = _strip_ddp_prefix(state)
    # Keep only keys that belong to the bare Transformer backbone.
    backbone_keys = set(model.state_dict().keys())
    filtered = {k: v for k, v in state.items() if k in backbone_keys}
    missing, unexpected = model.load_state_dict(filtered, strict=False)
    if missing:
        print(f"[load_backbone] {len(missing)} missing keys (e.g. {missing[:3]})")
    return model.to(device)


def unwrap(model: nn.Module) -> nn.Module:
    """Return the underlying module behind a DDP wrapper (or the model itself)."""
    return model.module if hasattr(model, "module") else model


def make_frozen_copy(model: nn.Module, device: str | None = None) -> nn.Module:
    """
    Deep-copy a model, put it in eval mode, and disable all gradients. Used for the
    DPO/PPO/GRPO reference model and the PPO old-policy snapshot. At the ~300M-1B
    scale this is cheap on an 80GB H100 and is the clearest possible implementation.
    """
    ref = copy.deepcopy(unwrap(model))
    if device is not None:
        ref = ref.to(device)
    ref.eval()
    for p in ref.parameters():
        p.requires_grad_(False)
    return ref


# --- Masked reductions -------------------------------------------------------

def masked_mean(values: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """Mean of ``values`` over positions where ``mask`` is truthy (safe if mask empty)."""
    mask = mask.to(values.dtype)
    total = (values * mask).sum()
    count = mask.sum().clamp(min=1.0)
    return total / count


def masked_mean_per_row(values: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """Per-row (B,) mean of ``values`` (B,T) over masked positions."""
    mask = mask.to(values.dtype)
    total = (values * mask).sum(dim=-1)
    count = mask.sum(dim=-1).clamp(min=1.0)
    return total / count


def gather_last(values: torch.Tensor, seq_lengths: torch.Tensor) -> torch.Tensor:
    """
    Given per-token ``values`` (B, T) and ``seq_lengths`` (B,), return the value at the
    last real token of each row, i.e. ``values[i, seq_lengths[i]-1]``. This is the
    InstructGPT convention for reading a scalar reward off a sequence model.
    """
    idx = (seq_lengths - 1).clamp(min=0).long()
    return values[torch.arange(values.size(0), device=values.device), idx]


# --- Checkpoint I/O ----------------------------------------------------------

def _cfg_to_dict(cfg: Any) -> Any:
    return asdict(cfg) if is_dataclass(cfg) else cfg


def save_stage_ckpt(
    path: str,
    model: nn.Module,
    optimizer: torch.optim.Optimizer | None,
    *,
    stage: str,
    cfg: Any,
    step: int,
    metrics: dict | None = None,
    extra: dict | None = None,
) -> None:
    """
    Save a checkpoint in the repo's existing shape (``model_state_dict`` /
    ``optimizer_state_dict``) plus post-training metadata (``stage``, ``cfg``, ``step``,
    ``metrics``). DDP wrappers are unwrapped first so checkpoints load cleanly on a
    single GPU.
    """
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    payload = {
        "model_state_dict": unwrap(model).state_dict(),
        "optimizer_state_dict": optimizer.state_dict() if optimizer is not None else None,
        "stage": stage,
        "cfg": _cfg_to_dict(cfg),
        "step": step,
        "metrics": metrics or {},
        "pytorch_version": torch.__version__,
        "cuda_version": torch.version.cuda,
    }
    if extra:
        payload.update(extra)
    torch.save(payload, path)


Writing src/post_training/utils.py


In [78]:
%%writefile src/post_training/value_head.py
"""
Actor-critic wrapper for PPO.

PPO needs a per-token *value* estimate V(s_t) alongside the policy logits. We get both
from a single backbone: the wrapper reuses the :class:`Transformer`'s ``forward_hidden``
and ``lm_head`` for the policy, and adds a small scalar value head on top of the same
hidden states. Sharing the backbone is the standard, memory-cheap choice; the value head
is initialized to output ~0 so early training is stable.
"""

from __future__ import annotations

import torch
import torch.nn as nn

from src.models.transformer import Transformer


class TransformerWithValueHead(nn.Module):
    """Wrap a :class:`Transformer` and add a scalar value head for PPO.

    ``forward(idx)`` returns ``(logits, values)`` where ``logits`` are the policy logits
    (B, T, vocab) and ``values`` are per-token value estimates (B, T).
    """

    def __init__(self, transformer: Transformer) -> None:
        super().__init__()
        self.transformer = transformer
        n_embed = transformer.lm_head.in_features
        self.value_head = nn.Sequential(
            nn.Linear(n_embed, n_embed),
            nn.ReLU(),
            nn.Linear(n_embed, 1),
        )
        # Start with values ~0 so the critic does not destabilize the policy early on.
        last = self.value_head[-1]
        nn.init.zeros_(last.weight)
        nn.init.zeros_(last.bias)

    # Expose the backbone's context length so rollout's cap detection works.
    @property
    def context_length(self) -> int:
        return self.transformer.context_length

    def forward(self, idx: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        hidden = self.transformer.forward_hidden(idx)        # (B, T, n_embed)
        logits = self.transformer.lm_head(hidden)            # (B, T, vocab)
        values = self.value_head(hidden).squeeze(-1)         # (B, T)
        return logits, values

    @torch.no_grad()
    def value_only(self, idx: torch.Tensor) -> torch.Tensor:
        """Per-token values without computing the (large) vocab logits -- used when
        scoring rollouts where only V(s_t) is needed."""
        hidden = self.transformer.forward_hidden(idx)
        return self.value_head(hidden).squeeze(-1)


Writing src/post_training/value_head.py


In [79]:
%%writefile tests/gpu_verify.sh
#!/usr/bin/env bash
# End-to-end GPU verification: run EVERY post-training script for a few steps on the GPU,
# on the REAL prepared datasets, with a tiny model (fits in the spare VRAM next to the
# running pretraining). Proves the full pipeline works on GPU + real data, not just CPU.
set -uo pipefail
cd "$(dirname "$0")/.."
export PYTHONPATH=. HF_HOME=/ephemeral/hf_cache PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
export CUDA_VISIBLE_DEVICES=${CUDA_VISIBLE_DEVICES:-0}
PY=/ephemeral/venv/bin/python
DIM="--vocab_size 50304 --context_length 1024 --n_embed 128 --n_head 4 --n_blocks 2 --device cuda --amp_dtype bf16"
CK=/ephemeral/ckpts/tiny_base.pt
ok() { echo "  [PASS] $1"; }; bad() { echo "  [FAIL] $1"; }

echo "== 1/6 SFT on real sft_packed.h5 (GPU) =="
$PY scripts/train_sft.py $DIM --pretrained_ckpt $CK --data_path /ephemeral/data/sft_packed.h5 \
  --out_ckpt /ephemeral/ckpts/gpuverify_sft.pt --batch_size 8 --epochs 1 --max_steps 12 \
  --eval_steps 1000 --warmup_steps 2 --save_every 1000 2>&1 | grep -E "step [0-9]+/|Done SFT" | tail -2 \
  && [ -f /ephemeral/ckpts/gpuverify_sft.pt ] && ok "SFT ran on GPU + saved" || bad "SFT"

echo "== 2/6 Reward Model on real preferences.jsonl (GPU) =="
$PY scripts/train_reward.py $DIM --sft_ckpt $CK --pref_path /ephemeral/data/prefs_small.jsonl \
  --out_ckpt /ephemeral/ckpts/gpuverify_reward.pt --batch_size 8 --epochs 1 --eval_steps 1000 \
  --warmup_steps 2 --max_len 512 --save_every 1000 2>&1 | grep -E "step [0-9]+/|Done RM" | tail -2 \
  && [ -f /ephemeral/ckpts/gpuverify_reward.pt ] && ok "Reward ran on GPU + saved" || bad "Reward"
# truncate to a few steps by capping rows via a tiny pref file
echo "== 3/6 DPO on real preferences.jsonl (GPU) =="
$PY scripts/train_dpo.py $DIM --sft_ckpt $CK --pref_path /ephemeral/data/prefs_small.jsonl \
  --out_ckpt /ephemeral/ckpts/gpuverify_dpo.pt --loss_type dpo --beta 0.1 --batch_size 8 --epochs 1 \
  --eval_steps 1000 --warmup_steps 2 --max_len 512 --save_every 1000 2>&1 | grep -E "step [0-9]+/|Done DPO" | tail -2 \
  && [ -f /ephemeral/ckpts/gpuverify_dpo.pt ] && ok "DPO ran on GPU + saved" || bad "DPO"

echo "== 4/6 PPO on real GSM8K prompts, verifier reward (GPU) =="
$PY scripts/train_ppo.py $DIM --sft_ckpt $CK --prompt_path /ephemeral/data/rl_prompts_train.jsonl \
  --reward_source verifier --iterations 2 --prompts_per_iter 4 --rollout_len 48 --ppo_epochs 2 \
  --minibatch_size 2 --eval_every 1000 --save_every 1000 --out_ckpt /ephemeral/ckpts/gpuverify_ppo.pt \
  2>&1 | grep -E "iter [0-9]+|Done PPO" | tail -2 \
  && [ -f /ephemeral/ckpts/gpuverify_ppo.pt ] && ok "PPO ran on GPU + saved" || bad "PPO"

echo "== 5/6 GRPO on real arithmetic+GSM8K prompts (GPU) =="
$PY scripts/train_grpo.py $DIM --sft_ckpt $CK --curriculum_path /ephemeral/data/arithmetic_prompts.jsonl \
  --prompt_path /ephemeral/data/rl_prompts_train.jsonl --curriculum_iters 1 --iterations 2 \
  --prompts_per_iter 2 --group_size 4 --rollout_len 48 --grpo_epochs 1 --eval_every 1000 --save_every 1000 \
  --out_ckpt /ephemeral/ckpts/gpuverify_grpo.pt 2>&1 | grep -E "iter [0-9]+|Done GRPO" | tail -2 \
  && [ -f /ephemeral/ckpts/gpuverify_grpo.pt ] && ok "GRPO ran on GPU + saved" || bad "GRPO"

echo "== 6/6 GSM8K eval benchmark on GPU =="
$PY scripts/eval_post_training.py --ckpt $CK --label tiny_base --limit 20 --max_new_tokens 48 --device cuda --samples 0 \
  2>&1 | grep -E "GSM8K test accuracy" | tail -1 \
  && ok "GSM8K eval ran on GPU" || bad "eval"

echo "== cleanup =="; rm -f /ephemeral/ckpts/gpuverify_*.pt; echo "done"


Writing tests/gpu_verify.sh


In [80]:
%%writefile tests/grpo_live.py
"""Live-progress GRPO-ascends-reward proof (prints every few iters). See verify_rl_optimizes.py."""
import torch
from src.models.transformer import Transformer
from src.post_training.grpo import group_advantages, grpo_loss
from src.post_training.rollout import generate_with_logprobs, compute_logprobs
from src.post_training.utils import make_frozen_copy, set_seed

VOCAB, TARGET, PL, GEN, G = 64, 7, 4, 8, 8
set_seed(0)
dev = "cuda" if torch.cuda.is_available() else "cpu"
policy = Transformer(n_head=4, n_embed=64, context_length=PL + GEN + 2, vocab_size=VOCAB, N_BLOCKS=2).to(dev)
ref = make_frozen_copy(policy, device=dev)
opt = torch.optim.Adam(policy.parameters(), lr=1e-2)
prompt = torch.zeros(G, PL, dtype=torch.long, device=dev)
hist = []
for it in range(40):
    rb = generate_with_logprobs(policy, prompt, GEN, temperature=1.0)
    seqs, rmask = rb.sequences, rb.response_mask
    rew = torch.tensor([(seqs[i, PL:] == TARGET).float().mean().item() for i in range(G)], device=dev)
    adv = group_advantages(rew, G)
    old, _ = compute_logprobs(policy, seqs, rmask, requires_grad=False)
    rf, _ = compute_logprobs(ref, seqs, rmask, requires_grad=False)
    new, _ = compute_logprobs(policy, seqs, rmask, requires_grad=True)
    loss, _ = grpo_loss(new, old, rf, adv, rmask[:, 1:], clip=0.2, kl_coef=0.0)
    opt.zero_grad(); loss.backward(); opt.step()
    hist.append(rew.mean().item())
    if it % 5 == 0 or it == 39:
        print(f"iter {it:2d} | mean reward (P[target token]) = {rew.mean().item():.3f}", flush=True)
start, end = sum(hist[:5]) / 5, sum(hist[-5:]) / 5
verdict = "PASS (RL ascends reward)" if end > start + 0.2 else "NO IMPROVEMENT"
print(f"\nRESULT: reward {start:.3f} -> {end:.3f}  => {verdict}", flush=True)


Writing tests/grpo_live.py


In [81]:
%%writefile tests/test_checkpoint_resume.py
"""
Regression tests for the legacy pretraining checkpoint/resume helpers.

Run from the repo root:
    PYTHONPATH=. python tests/test_checkpoint_resume.py
"""

from __future__ import annotations

import tempfile
import os
from unittest import mock

import torch

from scripts.train_transformer import (
    checkpoint_path,
    list_checkpoints,
    resolve_resume_path,
    restore_training_checkpoint,
    save_training_checkpoint,
    prune_old_checkpoints,
)
from src.models.transformer import Transformer


def _tiny_model():
    torch.manual_seed(0)
    return Transformer(n_head=2, n_embed=8, context_length=8, vocab_size=32, N_BLOCKS=1)


def _tiny_config():
    return {
        "t_lr": 1e-3,
        "t_lr_decayed": 1e-4,
        "t_lr_decay_step": 10,
        "device": "cpu",
    }


def test_checkpoint_round_trip_and_latest_resume():
    cfg = _tiny_config()
    model = _tiny_model()
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["t_lr"])

    # Populate optimizer state so the round trip proves optimizer resume works too.
    idx = torch.randint(0, 32, (2, 4))
    _, loss = model(idx, idx)
    loss.backward()
    optimizer.step()

    losses = [3.0, 2.0, 1.0]
    with tempfile.TemporaryDirectory() as tmp:
        first = checkpoint_path(tmp, 2)
        second = checkpoint_path(tmp, 3)
        save_training_checkpoint(first, model, optimizer, cfg, losses, step=2)
        save_training_checkpoint(second, model, optimizer, cfg, losses + [0.5], step=3)
        prune_old_checkpoints(tmp, keep_last=1)

        remaining = list_checkpoints(tmp)
        assert remaining == [second]
        assert resolve_resume_path("latest", tmp) == second

        restored = _tiny_model()
        restored_optim = torch.optim.AdamW(restored.parameters(), lr=cfg["t_lr"])
        next_step, restored_losses = restore_training_checkpoint(
            second,
            restored,
            restored_optim,
            cfg,
            "cpu",
        )

        assert next_step == 4
        assert restored_losses == losses + [0.5]
        assert restored_optim.state_dict()["state"]
        for original, loaded in zip(model.parameters(), restored.parameters()):
            assert torch.allclose(original, loaded)


def test_checkpoint_save_failure_does_not_leave_partial_file():
    cfg = _tiny_config()
    model = _tiny_model()
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["t_lr"])

    with tempfile.TemporaryDirectory() as tmp:
        target = checkpoint_path(tmp, 7)
        with mock.patch("scripts.train_transformer.torch.save", side_effect=RuntimeError("boom")):
            try:
                save_training_checkpoint(target, model, optimizer, cfg, [1.0], step=7)
                assert False, "save_training_checkpoint should re-raise save errors"
            except RuntimeError:
                pass

        assert not os.path.exists(target)
        assert not [name for name in os.listdir(tmp) if name.endswith(".tmp")]


if __name__ == "__main__":
    test_checkpoint_round_trip_and_latest_resume()
    test_checkpoint_save_failure_does_not_leave_partial_file()
    print("ALL CHECKPOINT RESUME TESTS PASSED")


Writing tests/test_checkpoint_resume.py


In [82]:
%%writefile tests/test_post_training_smoke.py
"""
Fast smoke tests for the post-training core (no training, no datasets, runs on CPU in
seconds). Verifies the load-bearing math so bugs surface before the long runs.

Run from the repo root:
    PYTHONPATH=. python tests/test_post_training_smoke.py
"""

import torch
import torch.nn.functional as F

from src.models.transformer import Transformer
from src.post_training import chat_template as ct
from src.post_training.rollout import (
    generate_with_logprobs, compute_logprobs, filter_logits,
)
from src.post_training.value_head import TransformerWithValueHead
from src.post_training.reward_model import RewardModel
from src.post_training.utils import make_frozen_copy, gather_last, masked_mean, build_model_from_config
from src.post_training.rewards import extract_answer, gsm8k_gold_answer, reward_gsm8k, is_correct


def _tiny_model(vocab=64, ctx=32):
    torch.manual_seed(0)
    return Transformer(n_head=4, n_embed=32, context_length=ctx, vocab_size=vocab, N_BLOCKS=2)


def test_forward_hidden_matches_forward():
    m = _tiny_model().eval()
    idx = torch.randint(0, 64, (2, 10))
    with torch.no_grad():
        logits_a, _ = m(idx)
        logits_b = m.lm_head(m.forward_hidden(idx))
    assert torch.allclose(logits_a, logits_b, atol=1e-5), "forward_hidden must reproduce forward logits"
    print("ok  forward_hidden matches forward")


def test_compute_logprobs_matches_manual():
    m = _tiny_model().eval()
    seq = torch.randint(0, 64, (3, 12))
    mask = torch.ones_like(seq, dtype=torch.bool)
    lp, shifted = compute_logprobs(m, seq, mask, temperature=1.0, requires_grad=False)
    with torch.no_grad():
        logits = m(seq)[0][:, :-1, :]
        manual = F.log_softmax(logits.float(), dim=-1).gather(-1, seq[:, 1:, None]).squeeze(-1)
    assert torch.allclose(lp, manual, atol=1e-5), "compute_logprobs disagrees with manual log_softmax/gather"
    assert shifted.shape == (3, 11)
    print("ok  compute_logprobs matches manual computation")


def test_rollout_logprobs_consistent():
    # Recorded sampling log-probs must equal a teacher-forced recompute under the same model+temp.
    m = _tiny_model().eval()
    prompt = torch.randint(0, 64, (4, 5))
    rb = generate_with_logprobs(m, prompt, max_new_tokens=10, temperature=1.0, top_k=None, top_p=None)
    assert rb.sequences.shape[1] == 5 + 10
    assert rb.response_mask[:, :5].sum() == 0, "prompt positions must not be in response_mask"
    lp, shifted = compute_logprobs(m, rb.sequences, rb.response_mask, temperature=1.0, requires_grad=False)
    # Align recorded gen_logprobs (B,G) to shifted recompute on generated positions.
    recorded = rb.gen_logprobs[rb.response_mask[:, 5:]]
    recomputed = lp[shifted]
    assert torch.allclose(recorded, recomputed, atol=1e-4), "sampling vs recompute log-probs diverge"
    print(f"ok  rollout/recompute log-probs consistent ({recorded.numel()} response tokens)")


def test_context_cap_enforced():
    m = _tiny_model(ctx=16).eval()
    prompt = torch.randint(0, 64, (1, 10))
    rb = generate_with_logprobs(m, prompt, max_new_tokens=100)  # asks for too many
    assert rb.sequences.shape[1] <= 16, "must clamp prompt+gen to context_length"
    print("ok  context_length cap enforced")


def test_value_and_reward_heads():
    m = _tiny_model()
    actor = TransformerWithValueHead(m)
    idx = torch.randint(0, 64, (2, 8))
    logits, values = actor(idx)
    assert logits.shape == (2, 8, 64) and values.shape == (2, 8)
    values.sum().backward()  # critic path differentiable
    assert actor.value_head[0].weight.grad is not None

    rm = RewardModel(_tiny_model())
    lengths = torch.tensor([8, 5])
    r = rm(idx, seq_lengths=lengths)
    assert r.shape == (2,)
    r.sum().backward()
    assert rm.reward_head.weight.grad is not None
    print("ok  value head + reward head shapes and backprop")


def test_frozen_copy_and_reductions():
    m = _tiny_model()
    ref = make_frozen_copy(m, device="cpu")
    assert all(not p.requires_grad for p in ref.parameters())
    vals = torch.tensor([[1.0, 2.0, 3.0]])
    mask = torch.tensor([[1, 1, 0]])
    assert abs(masked_mean(vals, mask).item() - 1.5) < 1e-6
    last = gather_last(torch.tensor([[10.0, 20.0, 30.0]]), torch.tensor([2]))
    assert last.item() == 20.0
    print("ok  frozen copy + masked_mean + gather_last")


def test_chat_template_masking():
    msgs = [{"role": "user", "content": "What is 2+2?"},
            {"role": "assistant", "content": "<answer>4</answer>"}]
    ids, mask = ct.encode_chat(msgs)
    assert len(ids) == len(mask)
    assert ids[-1] == ct.EOT_ID and mask[-1] == 1, "assistant turn must end in a trained EOT"
    # The assistant content tokens should be the only masked-in region (plus its EOT).
    assert sum(mask) > 0 and sum(mask) < len(mask)
    prompt_ids = ct.encode_prompt(msgs[:1])
    assert ct.decode(prompt_ids).endswith(ct.ASSISTANT_HEADER)
    print(f"ok  chat template ids/mask aligned ({sum(mask)}/{len(mask)} trained tokens)")


def test_reward_parsing():
    assert gsm8k_gold_answer("She has 18 apples.\n#### 18") == 18.0
    assert extract_answer("blah <answer>42</answer> done") == 42.0
    assert extract_answer("the total is 1,234 dollars") == 1234.0
    assert is_correct("<answer>42</answer>", 42.0)
    assert reward_gsm8k("<answer>42</answer>", 42.0) > 1.0           # correct + format
    assert reward_gsm8k("<answer>7</answer>", 42.0) == 0.2           # wrong but formatted
    assert reward_gsm8k("the answer is 42", 42.0) == 1.0            # correct, no format bonus
    print("ok  reward parsing + verifier scoring")


def test_build_from_config():
    from config.post_training_config import smoke, SFTConfig
    cfg = smoke(SFTConfig)
    m = build_model_from_config(cfg)
    assert m.context_length == 64 and m.lm_head.out_features == 256
    print("ok  build_model_from_config from dataclass")


if __name__ == "__main__":
    torch.manual_seed(0)
    test_forward_hidden_matches_forward()
    test_compute_logprobs_matches_manual()
    test_rollout_logprobs_consistent()
    test_context_cap_enforced()
    test_value_and_reward_heads()
    test_frozen_copy_and_reductions()
    test_chat_template_masking()
    test_reward_parsing()
    test_build_from_config()
    print("\nALL SMOKE TESTS PASSED")


Writing tests/test_post_training_smoke.py


In [83]:
%%writefile tests/test_rl_math.py
"""
Unit tests for the from-scratch RL math (PPO + GRPO). Pure-CPU, runs in milliseconds.

    PYTHONPATH=. python tests/test_rl_math.py
"""

import torch

from src.post_training.ppo import compute_gae, whiten, ppo_policy_loss, ppo_value_loss, approx_kl
from src.post_training.grpo import group_advantages, grpo_loss, k3_kl


def test_gae_reward_to_go():
    # lambda=1, zero values -> advantage is plain reward-to-go (=1 everywhere here).
    r = torch.tensor([[0.0, 0.0, 1.0]])
    v = torch.zeros(1, 3)
    m = torch.ones(1, 3, dtype=torch.bool)
    adv, ret = compute_gae(r, v, v, m, gamma=1.0, lam=1.0)
    assert torch.allclose(adv, torch.ones(1, 3)), adv
    assert torch.allclose(ret, torch.ones(1, 3)), ret
    # lambda<1 discounts earlier advantages.
    adv2, _ = compute_gae(r, v, v, m, gamma=1.0, lam=0.95)
    assert adv2[0, 0] < adv2[0, 1] < adv2[0, 2]
    print("ok  GAE reward-to-go + lambda discounting")


def test_gae_masks_outside_response():
    r = torch.tensor([[0.0, 1.0, 0.0]])
    v = torch.zeros(1, 3)
    m = torch.tensor([[0, 1, 0]], dtype=torch.bool)  # only middle token is a response token
    adv, ret = compute_gae(r, v, v, m, gamma=1.0, lam=1.0)
    assert adv[0, 0] == 0 and adv[0, 2] == 0
    print("ok  GAE zeros advantages outside the response")


def test_whiten():
    a = torch.tensor([[1.0, 2.0, 3.0, 0.0]])
    m = torch.tensor([[1, 1, 1, 0]], dtype=torch.bool)
    w = whiten(a, m)
    assert abs(w[0, :3].mean().item()) < 1e-5
    assert w[0, 3] == 0
    print("ok  whiten: zero-mean over mask, zero outside")


def test_ppo_losses():
    m = torch.ones(1, 3, dtype=torch.bool)
    # ratio == 1 -> policy loss = -mean(advantage)
    loss, clipf = ppo_policy_loss(torch.zeros(1, 3), torch.zeros(1, 3), torch.ones(1, 3), m, clip=0.2)
    assert abs(loss.item() + 1.0) < 1e-5 and clipf.item() == 0.0
    # value loss = 0.5 * MSE
    vl = ppo_value_loss(torch.zeros(1, 3), torch.zeros(1, 3), torch.ones(1, 3), m, vf_clip=0.2)
    assert abs(vl.item() - 0.5) < 1e-5
    # clipping engages for a large ratio
    _, cf = ppo_policy_loss(torch.full((1, 3), 1.0), torch.zeros(1, 3), torch.ones(1, 3), m, clip=0.2)
    assert cf.item() == 1.0
    print("ok  ppo policy/value loss + clip fraction")


def test_group_advantages():
    r = torch.tensor([1.0, 0.0, 0.0, 0.0, 2.0, 2.0, 2.0, 2.0])
    adv = group_advantages(r, 4)
    assert abs(adv[:4].mean().item()) < 1e-5
    assert adv[4:].abs().max().item() < 1e-3  # zero-variance group -> ~zero advantage
    assert adv[0] > 0 and adv[1] < 0
    print("ok  group_advantages: group-relative, zero-variance group ~0")


def test_grpo_loss_and_kl():
    B, L = 2, 3
    z = torch.zeros(B, L)
    m = torch.ones(B, L, dtype=torch.bool)
    loss, st = grpo_loss(z, z, z, torch.ones(B), m, clip=0.2, kl_coef=0.04)
    assert abs(loss.item() + 1.0) < 1e-5      # ratio=1, adv=1, kl=0 -> -1
    assert abs(st["kl"]) < 1e-6
    kl = k3_kl(torch.tensor([-1.0, 0.5, 0.0]), torch.zeros(3))
    assert (kl >= 0).all()                    # k3 KL is non-negative
    print("ok  grpo_loss + non-negative k3 KL")


if __name__ == "__main__":
    test_gae_reward_to_go()
    test_gae_masks_outside_response()
    test_whiten()
    test_ppo_losses()
    test_group_advantages()
    test_grpo_loss_and_kl()
    print("\nALL RL MATH TESTS PASSED")


Writing tests/test_rl_math.py


In [84]:
%%writefile tests/verify_data_and_eval.py
"""
Deep correctness checks on the REAL prepared data and the evaluation benchmark.
No training, no GPU -- inspects the actual files on disk and the scoring logic.

    PYTHONPATH=. python tests/verify_data_and_eval.py
"""

import json

import h5py
import numpy as np

from src.post_training.chat_template import EOT_ID, decode, encode_chat, ASSISTANT_HEADER
from src.post_training.rewards import gsm8k_gold_answer, extract_answer, is_correct, reward_gsm8k

DATA = "/ephemeral/data"
PASS, FAIL = "PASS", "FAIL"


def check(name, cond, detail=""):
    print(f"  [{PASS if cond else FAIL}] {name}" + (f" -- {detail}" if detail else ""))
    assert cond, f"FAILED: {name} {detail}"


def verify_pile():
    print("\n== Pile pretraining HDF5 ==")
    for split, path in [("train", f"{DATA}/pile_train.h5"), ("dev", f"{DATA}/pile_dev.h5")]:
        with h5py.File(path, "r") as f:
            d = f["tokens"]
            n = d.shape[0]
            head = np.asarray(d[:100000])
            check(f"{split}: nonempty", n > 1_000_000, f"{n:,} tokens")
            check(f"{split}: token ids in [0,50256]", head.min() >= 0 and head.max() <= 50256,
                  f"min {head.min()} max {head.max()}")
            check(f"{split}: contains EOT separators", (head == EOT_ID).sum() > 0,
                  f"{int((head==EOT_ID).sum())} EOT in first 100k")


def verify_sft_mask_alignment():
    """The critical SFT check: the loss_mask must be 1 EXACTLY on assistant-completion
    tokens and 0 on the prompt / role markers. We decode masked vs unmasked spans of a
    real packed row and confirm they read like completion vs prompt."""
    print("\n== SFT packed data: loss-mask alignment (real data) ==")
    with h5py.File(f"{DATA}/sft_packed.h5", "r") as f:
        tokens, masks = f["tokens"], f["loss_mask"]
        check("packed: nonempty", tokens.shape[0] > 100, f"{tokens.shape[0]} rows x {tokens.shape[1]}")
        check("packed: tokens/masks aligned shape", tokens.shape == masks.shape)
        row_t = np.asarray(tokens[0]); row_m = np.asarray(masks[0])
        check("packed: token ids valid", row_t.min() >= 0 and row_t.max() <= 50303)
        check("packed: mask is binary", set(np.unique(row_m)).issubset({0, 1}))
        frac = row_m.mean()
        check("packed: mask trains a sensible fraction", 0.02 < frac < 0.95, f"{frac:.2f} of tokens trained")

    # Re-derive a single example from the chat template and confirm the mask matches exactly.
    ids, mask = encode_chat([{"role": "user", "content": "What is 2+2?"},
                             {"role": "assistant", "content": "<answer>4</answer>"}])
    masked_text = decode([t for t, m in zip(ids, mask) if m == 1])
    unmasked_text = decode([t for t, m in zip(ids, mask) if m == 0])
    check("mask covers the assistant answer", "answer>4" in masked_text or "4" in masked_text,
          f"masked={masked_text!r}")
    check("mask excludes the user question", "2+2" in unmasked_text and "2+2" not in masked_text,
          f"unmasked={unmasked_text!r}")
    check("prompt form ends at assistant header", decode(encode_chat(
        [{"role": "user", "content": "Hi"}], add_generation_prompt=True)[0]).endswith(ASSISTANT_HEADER))


def verify_preferences():
    print("\n== Preference data (HH-RLHF + UltraFeedback) ==")
    for path in [f"{DATA}/preferences.jsonl", f"{DATA}/preferences_test.jsonl"]:
        rows = [json.loads(l) for l in open(path)]
        check(f"{path.split('/')[-1]}: nonempty", len(rows) > 100, f"{len(rows)} pairs")
        bad = [r for r in rows[:5000] if not (r.get("prompt") and r.get("chosen") and r.get("rejected"))]
        check("all pairs have prompt/chosen/rejected", len(bad) == 0)
        diff = [r for r in rows[:5000] if r["chosen"] == r["rejected"]]
        check("chosen != rejected", len(diff) == 0, f"{len(diff)} degenerate")
    # decode-roundtrip a real pair through the chat encoder
    r = json.loads(open(f"{DATA}/preferences.jsonl").readline())
    ids, _ = encode_chat([{"role": "user", "content": r["prompt"]}, {"role": "assistant", "content": r["chosen"]}])
    check("real pair encodes to tokens", len(ids) > 0, f"{len(ids)} tokens")


def verify_rl_prompts_and_gold():
    """Gold answers in the RL prompt files must match the GSM8K dataset's #### answer."""
    print("\n== RL prompts + gold answers ==")
    rows = [json.loads(l) for l in open(f"{DATA}/rl_prompts_test.jsonl")]
    check("gsm8k test prompts present", len(rows) > 100, f"{len(rows)} prompts")
    check("all have numeric gold", all(isinstance(r["gold"], (int, float)) for r in rows))
    # Cross-check gold against the live GSM8K dataset for a few rows.
    import os
    os.environ.setdefault("HF_HOME", "/ephemeral/hf_cache")
    from datasets import load_dataset
    ds = load_dataset("openai/gsm8k", "main", split="test")
    by_q = {ex["question"].strip(): ex["answer"] for ex in ds}
    checked = 0
    for r in rows[:50]:
        if r["prompt"] in by_q:
            assert gsm8k_gold_answer(by_q[r["prompt"]]) == r["gold"], (r["prompt"][:40], r["gold"])
            checked += 1
    check("gold matches GSM8K #### answer", checked >= 40, f"verified {checked}/50 against source")
    # arithmetic curriculum gold correctness
    ar = [json.loads(l) for l in open(f"{DATA}/arithmetic_prompts.jsonl")]
    check("arithmetic prompts present", len(ar) > 1000, f"{len(ar)} prompts")


def verify_eval_benchmark():
    """Prove the GSM8K scoring is correct independent of any model: a response containing
    the gold scores correct; a wrong number scores incorrect."""
    print("\n== Evaluation benchmark scoring (GSM8K verifier) ==")
    import os
    os.environ.setdefault("HF_HOME", "/ephemeral/hf_cache")
    from datasets import load_dataset
    ds = load_dataset("openai/gsm8k", "main", split="test").select(range(100))

    correct_when_right = correct_when_wrong = 0
    for ex in ds:
        gold = gsm8k_gold_answer(ex["answer"])
        good = f"<think>...</think><answer>{int(gold) if gold==int(gold) else gold}</answer>"
        bad = f"<answer>{gold + 7}</answer>"
        correct_when_right += int(is_correct(good, gold))
        correct_when_wrong += int(is_correct(bad, gold))
    check("perfect answers score correct", correct_when_right >= 98, f"{correct_when_right}/100")
    check("wrong answers score incorrect", correct_when_wrong == 0, f"{correct_when_wrong}/100 false positives")
    # reward shaping: correct+formatted > correct-only > wrong
    check("reward: correct+format > wrong", reward_gsm8k("<answer>5</answer>", 5.0) > reward_gsm8k("<answer>9</answer>", 5.0))
    check("reward: bounded", reward_gsm8k("<answer>5</answer>", 5.0) <= 1.2)
    # tolerant parsing variants
    check("parse: $ and commas", extract_answer("the total is $1,234") == 1234.0)
    check("parse: #### form", extract_answer("reasoning #### 18") == 18.0)


if __name__ == "__main__":
    verify_pile()
    verify_sft_mask_alignment()
    verify_preferences()
    verify_rl_prompts_and_gold()
    verify_eval_benchmark()
    print("\nALL DATA + EVAL CORRECTNESS CHECKS PASSED")


Writing tests/verify_data_and_eval.py


In [85]:
%%writefile tests/verify_rl_optimizes.py
"""
Prove the RL machinery actually OPTIMIZES its reward (not just runs without error).

Uses the real GRPO and PPO code paths (rollout -> reward -> advantage -> clipped update)
on a tiny model with a learnable synthetic reward: "emit the target token". If the loops
are correct, the policy raises the probability of the target token and the mean reward
climbs. Runs on CPU in well under a minute (tiny vocab).

    PYTHONPATH=. python tests/verify_rl_optimizes.py
"""

import torch

from src.models.transformer import Transformer
from src.post_training.grpo import group_advantages, grpo_loss
from src.post_training.ppo import compute_gae, whiten, ppo_policy_loss, ppo_value_loss
from src.post_training.rollout import generate_with_logprobs, compute_logprobs
from src.post_training.utils import make_frozen_copy, set_seed
from src.post_training.value_head import TransformerWithValueHead

VOCAB, TARGET = 64, 7
PROMPT_LEN, GEN = 4, 8


def dense_reward(seq, prompt_len, target=TARGET):
    """Fraction of generated tokens equal to the target token (dense, easy to learn)."""
    gen = seq[prompt_len:]
    return (gen == target).float().mean().item()


def tiny_model():
    set_seed(0)
    return Transformer(n_head=4, n_embed=64, context_length=PROMPT_LEN + GEN + 2, vocab_size=VOCAB, N_BLOCKS=2)


def verify_grpo_optimizes():
    print("\n== GRPO optimizes reward (real grpo_loss path) ==")
    policy = tiny_model()
    ref = make_frozen_copy(policy)
    opt = torch.optim.Adam(policy.parameters(), lr=1e-2)
    G = 8
    prompt = torch.zeros(1, PROMPT_LEN, dtype=torch.long)
    history = []
    for it in range(60):
        batch = prompt.repeat(G, 1)
        rb = generate_with_logprobs(policy, batch, GEN, temperature=1.0)
        seqs, rmask = rb.sequences, rb.response_mask
        rewards = torch.tensor([dense_reward(seqs[i], PROMPT_LEN) for i in range(G)])
        adv = group_advantages(rewards, G)
        old_lp, _ = compute_logprobs(policy, seqs, rmask, requires_grad=False)
        ref_lp, _ = compute_logprobs(ref, seqs, rmask, requires_grad=False)
        new_lp, _ = compute_logprobs(policy, seqs, rmask, requires_grad=True)
        loss, _ = grpo_loss(new_lp, old_lp, ref_lp, adv, rmask[:, 1:], clip=0.2, kl_coef=0.0)
        opt.zero_grad(); loss.backward(); opt.step()
        history.append(rewards.mean().item())
    start, end = sum(history[:5]) / 5, sum(history[-5:]) / 5
    print(f"  mean reward: start {start:.3f} -> end {end:.3f}")
    assert end > start + 0.2, f"GRPO did not improve reward ({start:.3f} -> {end:.3f})"
    print(f"  [PASS] GRPO raised P(target token); reward climbed {start:.2f} -> {end:.2f}")


def verify_ppo_optimizes():
    print("\n== PPO optimizes reward (real GAE + clipped losses path) ==")
    backbone = tiny_model()
    ref = make_frozen_copy(backbone)
    actor = TransformerWithValueHead(backbone)
    opt = torch.optim.Adam(actor.parameters(), lr=1e-2)
    B = 8
    prompt = torch.zeros(B, PROMPT_LEN, dtype=torch.long)
    history = []
    for it in range(60):
        rb = generate_with_logprobs(actor, prompt, GEN, temperature=1.0)
        seqs, rmask = rb.sequences, rb.response_mask
        resp = rmask[:, 1:]
        task_r = torch.tensor([dense_reward(seqs[i], PROMPT_LEN) for i in range(B)])

        with torch.no_grad():
            logits, values = actor(seqs)
            old_lp, _ = compute_logprobs(actor, seqs, rmask, requires_grad=False)
        old_v = values[:, :-1]
        # sparse terminal reward at last response token
        rewards = torch.zeros_like(resp, dtype=torch.float32)
        last = resp.float().cumsum(1).argmax(1)
        rewards[torch.arange(B), last] = task_r
        vnext = torch.cat([old_v[:, 1:], torch.zeros_like(old_v[:, :1])], 1)
        adv, ret = compute_gae(rewards, old_v, vnext, resp, gamma=1.0, lam=0.95)
        adv = whiten(adv, resp)
        for _ in range(4):
            logits, values = actor(seqs)
            new_lp, _ = compute_logprobs(actor, seqs, rmask, requires_grad=True)
            pl, _ = ppo_policy_loss(new_lp, old_lp, adv, resp, clip=0.2)
            vl = ppo_value_loss(values[:, :-1], old_v, ret, resp, vf_clip=0.2)
            loss = pl + 0.5 * vl
            opt.zero_grad(); loss.backward(); opt.step()
        history.append(task_r.mean().item())
    start, end = sum(history[:5]) / 5, sum(history[-5:]) / 5
    print(f"  mean reward: start {start:.3f} -> end {end:.3f}")
    assert end > start + 0.15, f"PPO did not improve reward ({start:.3f} -> {end:.3f})"
    print(f"  [PASS] PPO raised reward {start:.2f} -> {end:.2f}")


if __name__ == "__main__":
    verify_grpo_optimizes()
    verify_ppo_optimizes()
    print("\nRL OPTIMIZATION VERIFIED: both PPO and GRPO increase their reward.")


Writing tests/verify_rl_optimizes.py


In [86]:
%%writefile ui/__init__.py
"""Streamlit control-panel package for the training pipeline."""


Writing ui/__init__.py


In [87]:
%%writefile ui/app.py
"""
Train LLM From Scratch — control panel (Home).

Run with:  streamlit run ui/app.py
"""

from __future__ import annotations

import os

import streamlit as st

from ui import jobs, theme
from ui.stages import ABS_DOC, STAGES

theme.setup_page("Control Panel", "🧠")
theme.hero("🧠  Train LLM From Scratch — Control Panel",
           "Pretrain → SFT → Reward Model → DPO → PPO → GRPO · evaluate · chat — all from one place.")

# Master pipeline diagram (the hand-drawn overview).
overview = ABS_DOC("docs/diagrams/00_overview.png")
if os.path.exists(overview):
    st.image(overview, use_container_width=True)

st.markdown(
    "Use the **pages in the sidebar** to prepare data, configure and launch each training stage "
    "(with live logs + metric charts), evaluate on GSM8K, and chat with any checkpoint."
)

# --- GPU status ---
st.subheader("🖥️  GPUs")
gpus = theme.gpu_status()
if gpus:
    cols = st.columns(len(gpus))
    for col, (i, name, used, total, util) in zip(cols, gpus):
        col.metric(f"GPU {i} · {name.split(' ')[-1]}", f"{used/1024:.0f}/{total/1024:.0f} GB", f"{util}% util")
else:
    st.info("No GPUs detected (nvidia-smi unavailable) — CPU / smoke mode only.")

# --- Job status ---
st.subheader("📋  Jobs")
busy = jobs.gpu_busy()
if busy:
    st.warning(f"A GPU job is currently running: **{busy}**. Other GPU launches are guarded until it finishes.")
active = jobs.active_jobs()
if active:
    for rec in active:
        s = jobs.status(rec["job_id"])
        st.markdown(
            f'<div class="stage-card"><b>{rec["job_id"]}</b> &nbsp; {theme.status_badge(s)} '
            f'&nbsp; <code>{" ".join(rec["cmd"][:3])} …</code></div>', unsafe_allow_html=True)
else:
    st.caption("No jobs launched yet.")

# --- Pipeline map ---
st.subheader("🗺️  Pipeline")
for key, sgt in STAGES.items():
    s = jobs.status(key)
    st.markdown(
        f'<div class="stage-card" style="--c:{theme.COLORS.get("model")}">'
        f'{sgt.emoji} <b>{sgt.title}</b> &nbsp; {theme.status_badge(s)} '
        f'&nbsp; <span style="opacity:.7">{sgt.script}</span></div>', unsafe_allow_html=True)

st.divider()
st.caption("Docs: see the MkDocs site (`mkdocs serve`) or the `docs/` folder. "
           "CLI reference: `POST_TRAINING.md`.")


Writing ui/app.py


In [88]:
%%writefile ui/config_forms.py
"""
Render an editable config form from a stage's dataclass fields and write it back to JSON.

Base/runtime fields (shared) are saved to ``base.json``; the stage's own hyperparameters to
``configs/<stage>.json`` — matching the loader's merge model. Shows the exact resolved
launch command underneath.
"""

from __future__ import annotations

import json
import os
from dataclasses import fields

import streamlit as st

from config.loader import load_config
from config.post_training_config import BaseModelConfig
from ui.jobs import build_argv
from ui.stages import Stage

_BASE_FIELDS = {f.name for f in fields(BaseModelConfig)}


def _widget(f, value):
    t = str(f.type)
    label = f.name
    if t == "bool" or "bool" in t and "|" not in t:
        return st.checkbox(label, bool(value))
    if "int" in t and "|" not in t:
        return int(st.number_input(label, value=int(value), step=1, format="%d"))
    if "float" in t:
        return float(st.number_input(label, value=float(value), format="%g"))
    # str / str|None / unknown -> text; empty string means "unset" (-> null) for optionals
    return st.text_input(label, "" if value is None else str(value))


def _coerce_blank(f, value):
    """Map an empty text box back to None for optional (``| None``) string fields."""
    if isinstance(value, str) and value == "" and "None" in str(f.type):
        return None
    return value


def _write_json(path: str, obj: dict) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)
        f.write("\n")


def render_form(stage: Stage, smoke: bool = False, nproc: int = 1) -> None:
    json_path = stage.smoke_json if smoke else stage.config_json
    base_path = os.path.join(os.path.dirname(json_path), "base.json")
    cur = load_config(stage.cfg_cls, json_path)

    vals: dict = {}
    with st.expander("🧱 Model & runtime  ·  shared `base.json`", expanded=False):
        cols = st.columns(2)
        bf = [f for f in fields(stage.cfg_cls) if f.name in _BASE_FIELDS]
        for i, f in enumerate(bf):
            with cols[i % 2]:
                vals[f.name] = _coerce_blank(f, _widget(f, getattr(cur, f.name)))

    with st.expander(f"⚙️ {stage.title} hyperparameters  ·  `{json_path}`", expanded=True):
        cols = st.columns(2)
        sf = [f for f in fields(stage.cfg_cls) if f.name not in _BASE_FIELDS]
        for i, f in enumerate(sf):
            with cols[i % 2]:
                vals[f.name] = _coerce_blank(f, _widget(f, getattr(cur, f.name)))

    if st.button("💾 Save config", key=f"save_{stage.key}_{smoke}"):
        base_vals = {k: v for k, v in vals.items() if k in _BASE_FIELDS}
        stage_vals = {k: v for k, v in vals.items() if k not in _BASE_FIELDS}
        _write_json(base_path, base_vals)
        _write_json(json_path, stage_vals)
        st.success(f"Saved → `{base_path}` and `{json_path}`")

    st.caption("Resolved launch command")
    argv = build_argv(stage.script, json_path, nproc, stage.multi_gpu)
    st.code(" ".join(argv), language="bash")


Writing ui/config_forms.py


In [89]:
%%writefile ui/docs_render.py
"""
Render a ``docs/*.md`` theory page inside Streamlit.

Streamlit's ``st.markdown`` can't load local-disk images and won't render the ``<details>``
mermaid blocks, so we: (1) show the stage diagram with ``st.image`` (absolute path), and
(2) strip the markdown image line + the ``<details>…</details>`` mermaid block before
rendering the prose. The ``docs/*.md`` files stay the single source of truth (the MkDocs
site publishes the same files).
"""

from __future__ import annotations

import os
import re

from ui.stages import ABS_DOC


def _clean_markdown(text: str) -> str:
    # drop the embedded PNG line (we render it via st.image instead)
    text = re.sub(r"^!\[[^\]]*\]\([^)]*\)\s*$", "", text, flags=re.MULTILINE)
    # drop the <details>…</details> live-mermaid block
    text = re.sub(r"<details>.*?</details>", "", text, flags=re.DOTALL)
    # drop the leading H1 (the page already has a header)
    text = re.sub(r"^#\s+.*$", "", text, count=1, flags=re.MULTILINE)
    return text.strip()


def render_doc(st, doc_md: str, diagram_png: str | None = None) -> None:
    """Render the diagram (if any) + the prose of a docs markdown file."""
    if diagram_png:
        abs_png = ABS_DOC(diagram_png)
        if os.path.exists(abs_png):
            st.image(abs_png, use_container_width=True)
    abs_md = ABS_DOC(doc_md)
    if os.path.exists(abs_md):
        with open(abs_md) as f:
            st.markdown(_clean_markdown(f.read()))
    else:
        st.info(f"Theory doc not found: {doc_md}")


Writing ui/docs_render.py


In [90]:
%%writefile ui/jobs.py
"""
Background job manager for the control panel.

Launches the real training / data-prep scripts as detached subprocesses, captures their
output to a logfile, and records a tiny JSON registry under /ephemeral/ui_jobs/ so jobs
survive Streamlit reruns and page navigation. Includes a GPU-busy guard so we never start a
second multi-GPU job on top of a running one (which would OOM the H100s).
"""

from __future__ import annotations

import json
import os
import signal
import subprocess
import sys
import time

from ui.stages import REPO_ROOT

JOB_DIR = "/ephemeral/ui_jobs"
_TORCHRUN = os.path.join(os.path.dirname(sys.executable), "torchrun")


def _ensure_job_dir() -> None:
    os.makedirs(JOB_DIR, exist_ok=True)


def _reg(job_id: str) -> str:
    return os.path.join(JOB_DIR, f"{job_id}.json")


def _log(job_id: str) -> str:
    return os.path.join(JOB_DIR, f"{job_id}.log")


def _write_registry(job_id: str, data: dict) -> None:
    with open(_reg(job_id), "w") as f:
        json.dump(data, f)


def read_registry(job_id: str) -> dict | None:
    p = _reg(job_id)
    if not os.path.exists(p):
        return None
    try:
        with open(p) as f:
            return json.load(f)
    except json.JSONDecodeError:
        return None


def _alive(pid: int) -> bool:
    try:
        os.kill(pid, 0)
    except OSError:
        return False
    # A finished detached child lingers as a zombie (parent = this process) and still
    # passes kill(pid, 0). Treat zombies as dead and best-effort reap them.
    try:
        with open(f"/proc/{pid}/stat") as f:
            state = f.read().rsplit(") ", 1)[1].split(" ", 1)[0]
        if state == "Z":
            try:
                os.waitpid(pid, os.WNOHANG)
            except OSError:
                pass
            return False
    except (FileNotFoundError, IndexError):
        return False
    return True


def build_argv(script: str, config_json: str, nproc: int, multi_gpu: bool, extra: list[str] | None = None) -> list[str]:
    """Construct the exact command (torchrun for multi-GPU, else python)."""
    base = [script, "--config", config_json] + (extra or [])
    if multi_gpu and nproc > 1:
        return [_TORCHRUN, "--standalone", f"--nproc_per_node={nproc}", *base]
    return [sys.executable, *base]


def launch(job_id: str, argv: list[str], *, kind: str = "cpu") -> dict:
    """Start ``argv`` as a detached background job. ``kind`` is 'gpu' or 'cpu' (for the guard)."""
    _ensure_job_dir()
    log_path = _log(job_id)
    env = {**os.environ, "PYTHONPATH": REPO_ROOT}
    env.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
    env.setdefault("HF_HOME", "/ephemeral/hf_cache")
    logf = open(log_path, "wb")
    proc = subprocess.Popen(
        argv, cwd=REPO_ROOT, env=env, stdout=logf, stderr=subprocess.STDOUT,
        start_new_session=True,   # own process group -> killpg reaches all torchrun ranks
    )
    rec = dict(job_id=job_id, pid=proc.pid, cmd=argv, log=log_path, kind=kind,
               started=time.time(), status="running")
    _write_registry(job_id, rec)
    return rec


def status(job_id: str) -> str:
    """Return 'running' | 'finished' | 'failed' | 'none'."""
    rec = read_registry(job_id)
    if not rec:
        return "none"
    if _alive(rec["pid"]):
        return "running"
    tail = tail_log(job_id, 4000).lower()
    if "traceback" in tail or "error:" in tail or "aborted" in tail:
        return "failed"
    return "finished"


def stop(job_id: str) -> bool:
    """Terminate a running job and all its workers (process group)."""
    rec = read_registry(job_id)
    if not rec:
        return False
    try:
        os.killpg(os.getpgid(rec["pid"]), signal.SIGTERM)
    except OSError:
        return False
    rec["status"] = "stopped"
    _write_registry(job_id, rec)
    return True


def tail_log(job_id: str, max_bytes: int = 16000) -> str:
    p = _log(job_id)
    if not os.path.exists(p):
        return ""
    with open(p, "rb") as f:
        f.seek(0, os.SEEK_END)
        size = f.tell()
        f.seek(max(0, size - max_bytes))
        return f.read().decode("utf-8", errors="replace")


def gpu_busy() -> str | None:
    """Return the job_id of a *running* GPU job, if any (for the launch guard)."""
    if not os.path.isdir(JOB_DIR):
        return None
    for fn in os.listdir(JOB_DIR):
        if not fn.endswith(".json"):
            continue
        rec = read_registry(fn[:-5])
        if rec and rec.get("kind") == "gpu" and _alive(rec["pid"]):
            return rec["job_id"]
    return None


def active_jobs() -> list[dict]:
    out = []
    if not os.path.isdir(JOB_DIR):
        return out
    for fn in sorted(os.listdir(JOB_DIR)):
        if fn.endswith(".json"):
            rec = read_registry(fn[:-5])
            if rec:
                rec["live"] = _alive(rec["pid"])
                out.append(rec)
    return out


Writing ui/jobs.py


In [91]:
%%writefile ui/metrics.py
"""Read the newest ``/ephemeral/logs/<prefix>_*.jsonl`` metrics file into a DataFrame."""

from __future__ import annotations

import glob
import json
import os

from ui.stages import LOG_DIR


def latest_log(prefix: str) -> str | None:
    files = glob.glob(os.path.join(LOG_DIR, f"{prefix}_*.jsonl"))
    return max(files, key=os.path.getmtime) if files else None


def load_metrics(path: str):
    import pandas as pd

    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
    return pd.DataFrame(rows)


def metric_columns(df) -> list[str]:
    """Numeric columns worth plotting (drop bookkeeping)."""
    skip = {"step", "wall"}
    return [c for c in df.columns if c not in skip and df[c].dtype.kind in "fi"]


Writing ui/metrics.py


In [92]:
%%writefile ui/pages/1_Data.py
"""Data preparation page: download + preprocess every dataset as background jobs."""

from __future__ import annotations

import os
import re
import sys
import time

import streamlit as st

from ui import jobs, theme
from ui.stages import DATA_DIR, DATA_SCRIPTS

theme.setup_page("Data", "📦")
theme.hero("📦  Data preparation",
           "Download & preprocess the corpora — Pile (pretrain), Alpaca/Dolly/GSM8K (SFT), "
           "HH-RLHF/UltraFeedback (preferences), GSM8K + arithmetic (RL).")

st.caption("Each button launches the real `scripts/prepare_*.py` as a background job (sets "
           "`HF_HOME=/ephemeral/hf_cache`). Large downloads take a while — watch the log below.")

for name, args in DATA_SCRIPTS.items():
    job_id = "data_" + re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
    s = jobs.status(job_id)
    c1, c2 = st.columns([3, 1])
    c1.markdown(f'<div class="stage-card" style="--c:{theme.COLORS["data"]}">'
                f'<b>{name}</b> &nbsp; {theme.status_badge(s)}<br>'
                f'<code>{" ".join(args)}</code></div>', unsafe_allow_html=True)
    if c2.button("Prepare", key=f"prep_{job_id}", disabled=(s == "running")):
        env_argv = [sys.executable, *args]
        rec = jobs.launch(job_id, env_argv, kind="cpu")
        # data scripts need the HF cache env; relaunch path uses jobs env + HF_HOME
        os.environ.setdefault("HF_HOME", "/ephemeral/hf_cache")
        st.session_state["last_data_job"] = job_id
        st.rerun()

st.subheader("Live log")
last = st.session_state.get("last_data_job")
sel = st.selectbox("Job", [j["job_id"] for j in jobs.active_jobs() if j["job_id"].startswith("data_")] or ["—"],
                   index=0)
job_to_show = sel if sel != "—" else last
if job_to_show:
    st.code(jobs.tail_log(job_to_show, 10000) or "(no log yet)", language="text")
    if jobs.status(job_to_show) == "running" and st.toggle("Auto-refresh", value=True):
        time.sleep(3); st.rerun()

st.subheader("Files in /ephemeral/data")
if os.path.isdir(DATA_DIR):
    rows = []
    for fn in sorted(os.listdir(DATA_DIR)):
        p = os.path.join(DATA_DIR, fn)
        if os.path.isfile(p):
            rows.append({"file": fn, "size": f"{os.path.getsize(p)/1e6:.1f} MB"})
    st.dataframe(rows, use_container_width=True, hide_index=True) if rows else st.caption("empty")
else:
    st.caption("/ephemeral/data does not exist yet.")


Writing ui/pages/1_Data.py


In [93]:
%%writefile ui/pages/2_Pretrain.py
"""2_Pretrain stage page."""
from ui.stage_page import render_stage_page
render_stage_page("pretrain")


Writing ui/pages/2_Pretrain.py


In [94]:
%%writefile ui/pages/3_SFT.py
"""3_SFT stage page."""
from ui.stage_page import render_stage_page
render_stage_page("sft")


Writing ui/pages/3_SFT.py


In [95]:
%%writefile ui/pages/4_Reward_Model.py
"""4_Reward_Model stage page."""
from ui.stage_page import render_stage_page
render_stage_page("reward")


Writing ui/pages/4_Reward_Model.py


In [96]:
%%writefile ui/pages/5_DPO.py
"""5_DPO stage page."""
from ui.stage_page import render_stage_page
render_stage_page("dpo")


Writing ui/pages/5_DPO.py


In [97]:
%%writefile ui/pages/6_PPO.py
"""6_PPO stage page."""
from ui.stage_page import render_stage_page
render_stage_page("ppo")


Writing ui/pages/6_PPO.py


In [98]:
%%writefile ui/pages/7_GRPO.py
"""7_GRPO stage page."""
from ui.stage_page import render_stage_page
render_stage_page("grpo")


Writing ui/pages/7_GRPO.py


In [99]:
%%writefile ui/pages/8_Evaluate.py
"""Evaluate a checkpoint on GSM8K (greedy), in-process."""

from __future__ import annotations

import glob

import streamlit as st

from ui import theme

theme.setup_page("Evaluate", "📊")
theme.hero("📊  Evaluate on GSM8K", "Greedy GSM8K accuracy for any checkpoint, with sample generations.")

ckpts = sorted(glob.glob("/ephemeral/ckpts/*.pt"))
if not ckpts:
    st.warning("No checkpoints found in /ephemeral/ckpts. Train a stage first.")
    st.stop()

ckpt = st.selectbox("Checkpoint", ckpts, index=len(ckpts) - 1)
c1, c2, c3 = st.columns(3)
limit = c1.slider("Num questions", 5, 200, 20, step=5)
max_new = c2.slider("Max new tokens", 64, 400, 256, step=32)
device = c3.selectbox("Device", ["cuda", "cpu"], index=0)

if st.button("▶️ Run GSM8K eval", type="primary"):
    with st.spinner(f"Generating + scoring {limit} GSM8K problems on {device} …"):
        from src.post_training.evaluation import gsm8k_accuracy, load_gsm8k_eval
        from src.post_training.inference import load_model_from_ckpt
        model = load_model_from_ckpt(ckpt, device)
        qa = load_gsm8k_eval("test", limit=limit)
        res = gsm8k_accuracy(model, qa, device=device, max_new_tokens=max_new, greedy=True,
                             return_samples=min(5, limit))
    st.metric("GSM8K accuracy", f"{res['accuracy']*100:.1f}%", f"{res['correct']}/{res['n']} correct")
    st.subheader("Sample generations")
    for s in res["samples"]:
        with st.expander(("✅ " if s["correct"] else "❌ ") + s["q"][:90]):
            st.markdown(f"**Gold:** `{s['gold']}` · **Correct:** {s['correct']}")
            st.code(s["response"][:1200], language="text")


Writing ui/pages/8_Evaluate.py


In [100]:
%%writefile ui/pages/9_Chat.py
"""Chat with any checkpoint, in-process (no subprocess)."""

from __future__ import annotations

import glob

import streamlit as st

from ui import theme

theme.setup_page("Chat", "💬")
theme.hero("💬  Chat", "Talk to any checkpoint — chat template for instruction models, raw mode for the base.")

ckpts = sorted(glob.glob("/ephemeral/ckpts/*.pt"))
if not ckpts:
    st.warning("No checkpoints in /ephemeral/ckpts yet. Train a stage (or run a smoke job) first.")
    st.stop()

with st.sidebar:
    st.header("Generation")
    ckpt = st.selectbox("Checkpoint", ckpts, index=len(ckpts) - 1)
    device = st.selectbox("Device", ["cuda", "cpu"], index=0)
    raw = st.toggle("Raw mode (base continuation, no chat template)", value=False)
    greedy = st.toggle("Greedy (deterministic)", value=False)
    temperature = st.slider("Temperature", 0.1, 1.5, 0.8, 0.05, disabled=greedy)
    top_p = st.slider("top-p", 0.1, 1.0, 0.95, 0.05, disabled=greedy)
    max_new = st.slider("Max new tokens", 16, 512, 256, 16)
    system = st.text_input("System prompt (chat mode)", "")
    if st.button("🗑️ Clear history"):
        st.session_state.pop("chat_msgs", None)


@st.cache_resource(show_spinner="Loading checkpoint …")
def _load(path, dev):
    from src.post_training.inference import load_model_from_ckpt
    return load_model_from_ckpt(path, dev)


model = _load(ckpt, device)
st.caption(f"Loaded `{ckpt}` · {sum(p.numel() for p in model.parameters())/1e6:.0f}M params on {device} · "
           f"mode={'raw' if raw else 'chat'}")

st.session_state.setdefault("chat_msgs", [])
for m in st.session_state["chat_msgs"]:
    st.chat_message(m["role"]).markdown(m["content"])

prompt = st.chat_input("Ask something (e.g. 'What is 13 + 29?')")
if prompt:
    st.session_state["chat_msgs"].append({"role": "user", "content": prompt})
    st.chat_message("user").markdown(prompt)
    with st.chat_message("assistant"), st.spinner("Generating …"):
        from src.post_training.inference import generate_reply
        reply = generate_reply(model, prompt, device=device, system=system or None, raw=raw,
                               max_new_tokens=max_new, temperature=temperature,
                               top_p=top_p if top_p < 1 else None, greedy=greedy)
        st.markdown(reply if reply.strip() else "_(empty generation)_")
    st.session_state["chat_msgs"].append({"role": "assistant", "content": reply})


Writing ui/pages/9_Chat.py


In [101]:
%%writefile ui/stage_page.py
"""
Reusable renderer for a training-stage page: theory + diagram → config form → launch /
stop (with the GPU-busy guard) → live log → metric charts. Each stage page is a 3-liner
that calls :func:`render_stage_page`.
"""

from __future__ import annotations

import streamlit as st

from ui import jobs, metrics, theme
from ui.config_forms import render_form
from ui.docs_render import render_doc
from ui.stages import STAGES


def render_stage_page(stage_key: str) -> None:
    stage = STAGES[stage_key]
    theme.setup_page(stage.title, stage.emoji)
    theme.hero(f"{stage.emoji}  {stage.title}", f"Configure, launch, and monitor the {stage.title} stage.")

    job_status = jobs.status(stage.key)
    st.markdown(f"**Job status:** {theme.status_badge(job_status)}", unsafe_allow_html=True)

    tab_run, tab_theory = st.tabs(["🚀 Run", "📖 Theory"])

    with tab_theory:
        render_doc(st, stage.doc_md, stage.diagram_png)

    with tab_run:
        c1, c2 = st.columns([1, 1])
        with c1:
            smoke = st.toggle("Smoke mode (tiny model, few steps, CPU)", value=False, key=f"smoke_{stage.key}")
        with c2:
            ngpu = theme.gpu_status()
            max_gpu = max(1, len(ngpu))
            nproc = st.slider("GPUs (torchrun)", 1, max_gpu, min(2, max_gpu) if stage.multi_gpu and not smoke else 1,
                              disabled=smoke or not stage.multi_gpu, key=f"nproc_{stage.key}")
            if smoke:
                nproc = 1

        st.subheader("Configuration")
        render_form(stage, smoke=smoke, nproc=nproc)

        st.subheader("Launch")
        busy = jobs.gpu_busy()
        guard = (not smoke) and busy is not None and busy != stage.key
        if guard:
            st.warning(f"🚧 GPUs are occupied by job **{busy}**. Stop it, or use Smoke mode (CPU), "
                       f"before launching {stage.title} on GPU.")
        cols = st.columns(3)
        with cols[0]:
            if st.button("▶️ Launch", type="primary", disabled=(job_status == "running" or guard),
                         key=f"launch_{stage.key}"):
                cfg_json = stage.smoke_json if smoke else stage.config_json
                argv = jobs.build_argv(stage.script, cfg_json, nproc, stage.multi_gpu)
                kind = "cpu" if smoke else ("gpu" if stage.multi_gpu else "cpu")
                jobs.launch(stage.key, argv, kind=kind)
                st.rerun()
        with cols[1]:
            if st.button("⏹️ Stop", disabled=(job_status != "running"), key=f"stop_{stage.key}"):
                jobs.stop(stage.key)
                st.rerun()
        with cols[2]:
            auto = st.toggle("Auto-refresh", value=(job_status == "running"), key=f"auto_{stage.key}")

        st.subheader("Live log")
        st.code(jobs.tail_log(stage.key, 12000) or "(no log yet — launch the stage)", language="text")

        st.subheader("Metrics")
        log = metrics.latest_log(stage.log_prefix)
        if log:
            df = metrics.load_metrics(log)
            cols_to_plot = metrics.metric_columns(df) if not df.empty else []
            if "step" in df.columns and cols_to_plot:
                st.line_chart(df.set_index("step")[cols_to_plot])
                st.caption(f"from `{log}` · {len(df)} logged points")
            else:
                st.info("No metrics logged yet.")
        else:
            st.info(f"No `{stage.log_prefix}_*.jsonl` metrics found yet.")

        if auto:
            import time
            time.sleep(3)
            st.rerun()


Writing ui/stage_page.py


In [102]:
%%writefile ui/stages.py
"""
Single source of truth for the control panel: maps each pipeline stage to its config
dataclass, JSON file, training script, log prefix, theory doc, diagram, and whether it is a
multi-GPU (torchrun) stage. Every page / form / job reads from here.
"""

from __future__ import annotations

import os
from dataclasses import dataclass

from config.post_training_config import (
    PretrainConfig, SFTConfig, RewardConfig, DPOConfig, PPOConfig, GRPOConfig,
)

REPO_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
CKPT_DIR = "/ephemeral/ckpts"
DATA_DIR = "/ephemeral/data"
LOG_DIR = "/ephemeral/logs"


@dataclass(frozen=True)
class Stage:
    key: str                 # short id ("sft")
    title: str               # display ("Supervised Fine-Tuning")
    emoji: str
    cfg_cls: type            # the dataclass
    config_json: str         # configs/<stage>.json
    smoke_json: str          # configs/smoke/<stage>.json
    script: str              # scripts/<trainer>.py
    log_prefix: str          # JSONL files are <log_prefix>_<ts>.jsonl
    doc_md: str              # docs/<NN_stage>.md (theory)
    diagram_png: str         # docs/diagrams/<NN_stage>.png
    multi_gpu: bool          # True -> can use torchrun


STAGES: dict[str, Stage] = {
    "pretrain": Stage("pretrain", "Pretraining", "📚", PretrainConfig,
                      "configs/pretrain.json", "configs/smoke/pretrain.json",
                      "scripts/pretrain_base.py", "pretrain",
                      "docs/02_pretraining.md", "docs/diagrams/02_pretraining.png", True),
    "sft": Stage("sft", "Supervised Fine-Tuning", "🎯", SFTConfig,
                 "configs/sft.json", "configs/smoke/sft.json",
                 "scripts/train_sft.py", "sft",
                 "docs/03_sft.md", "docs/diagrams/03_sft.png", True),
    "reward": Stage("reward", "Reward Model", "🏅", RewardConfig,
                    "configs/reward.json", "configs/smoke/reward.json",
                    "scripts/train_reward.py", "reward",
                    "docs/04_reward_model.md", "docs/diagrams/04_reward_model.png", True),
    "dpo": Stage("dpo", "DPO / ORPO / KTO", "⚖️", DPOConfig,
                 "configs/dpo.json", "configs/smoke/dpo.json",
                 "scripts/train_dpo.py", "dpo",
                 "docs/05_dpo.md", "docs/diagrams/05_dpo.png", True),
    "ppo": Stage("ppo", "PPO (RLHF)", "🎮", PPOConfig,
                 "configs/ppo.json", "configs/smoke/ppo.json",
                 "scripts/train_ppo.py", "ppo",
                 "docs/06_ppo.md", "docs/diagrams/06_ppo.png", True),
    "grpo": Stage("grpo", "GRPO / RLVR", "🧠", GRPOConfig,
                  "configs/grpo.json", "configs/smoke/grpo.json",
                  "scripts/train_grpo.py", "grpo",
                  "docs/07_grpo.md", "docs/diagrams/07_grpo.png", True),
}

# Data-prep scripts (plain python jobs, no config dataclass).
DATA_SCRIPTS = {
    "Pretrain corpus (Pile → HDF5)": ["scripts/prepare_pretrain_data.py", "--split", "train",
                                      "--num_shards", "1", "--out", f"{DATA_DIR}/pile_train.h5"],
    "Pretrain dev (Pile val)": ["scripts/prepare_pretrain_data.py", "--split", "val",
                                "--out", f"{DATA_DIR}/pile_dev.h5"],
    "SFT (Alpaca · Dolly · GSM8K)": ["scripts/prepare_sft_data.py"],
    "Preferences (HH-RLHF · UltraFeedback)": ["scripts/prepare_preference_data.py", "--source", "both"],
    "RL prompts (GSM8K + arithmetic)": ["scripts/prepare_rl_prompts.py"],
}

ABS_DOC = lambda rel: os.path.join(REPO_ROOT, rel)   # noqa: E731


Writing ui/stages.py


In [103]:
%%writefile ui/theme.py
"""Shared look-and-feel: page config, CSS injection, header, GPU status, job badges."""

from __future__ import annotations

import streamlit as st

# Palette matching the hand-drawn diagrams.
COLORS = {
    "data": "#27ae60", "proc": "#2c6fbb", "store": "#16a085", "model": "#d48806",
    "rl": "#e67e22", "loss": "#c0392b", "eval": "#8e44ad", "ckpt": "#888888",
}

_CSS = """
<style>
.block-container {padding-top: 2rem; max-width: 1100px;}
h1, h2, h3 {letter-spacing: .2px;}
.hero {background: linear-gradient(135deg,#16a085 0%, #2c6fbb 100%);
       color: #fff; padding: 1.4rem 1.6rem; border-radius: 16px; margin-bottom: 1.2rem;}
.hero h1 {color:#fff; margin:0 0 .3rem 0; font-size: 1.7rem;}
.hero p {color:#eafaf5; margin:0; font-size: .98rem;}
.stage-card {border-left: 6px solid var(--c,#16a085); background: rgba(127,127,127,.06);
             padding: .8rem 1rem; border-radius: 10px; margin:.3rem 0;}
.badge {display:inline-block; padding:.12rem .6rem; border-radius:999px; font-size:.78rem;
        font-weight:600; color:#fff;}
.b-run{background:#d48806;} .b-ok{background:#27ae60;} .b-fail{background:#c0392b;}
.b-idle{background:#888;}
div[data-testid="stMetricValue"] {font-size: 1.5rem;}
</style>
"""

_BADGE = {"running": ("b-run", "running"), "finished": ("b-ok", "finished"),
          "failed": ("b-fail", "failed"), "stopped": ("b-idle", "stopped"), "none": ("b-idle", "idle")}


def setup_page(title: str, icon: str = "🧠") -> None:
    st.set_page_config(page_title=f"{title} · Train LLM From Scratch", page_icon=icon, layout="wide")
    st.markdown(_CSS, unsafe_allow_html=True)


def hero(title: str, subtitle: str) -> None:
    st.markdown(f'<div class="hero"><h1>{title}</h1><p>{subtitle}</p></div>', unsafe_allow_html=True)


def status_badge(status: str) -> str:
    cls, label = _BADGE.get(status, ("b-idle", status))
    return f'<span class="badge {cls}">{label}</span>'


def gpu_status():
    """Return a list of (idx, name, used_mb, total_mb, util) via nvidia-smi, or []."""
    import shutil
    import subprocess
    if not shutil.which("nvidia-smi"):
        return []
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total,utilization.gpu",
             "--format=csv,noheader,nounits"], text=True, timeout=5)
    except Exception:  # noqa: BLE001
        return []
    rows = []
    for line in out.strip().splitlines():
        i, name, used, total, util = [x.strip() for x in line.split(",")]
        rows.append((int(i), name, int(used), int(total), int(util)))
    return rows


Writing ui/theme.py


## Data dan checkpoint dari Google Drive (opsional)

Setiap notebook bisa mendapat runtime Colab baru. Jika data atau checkpoint sudah disimpan di Drive, ubah `RESTORE_FROM_DRIVE` ke `True` untuk menyalinnya ke lokasi yang dipakai kode repo.


In [104]:
RESTORE_FROM_DRIVE = False
if RESTORE_FROM_DRIVE:
    import shutil
    from google.colab import drive
    drive.mount('/content/drive')
    drive_root = Path('/content/drive/MyDrive/train-llm-artifacts')
    for subfolder in ('data', 'ckpts'):
        source_dir = drive_root / subfolder
        if source_dir.exists():
            for source in source_dir.iterdir():
                if source.is_file():
                    destination = ARTIFACT_ROOT / subfolder / source.name
                    shutil.copy2(source, destination)
                    print('Restored:', destination)


## 3. Profil Colab dan data

Profil awal memakai dimensi `configs/smoke` agar tidak langsung membuat model 400M. Semua tahap tetap memerlukan data sesuai format repo. Skrip persiapan data di atas tersedia, tetapi unduhan Pile dapat besar walau `--max_tokens` kecil. Anda juga bisa menaruh HDF5 yang sudah disiapkan di `/ephemeral/data`.


In [105]:
import json, torch
PROFILE = 'smoke'  # ubah ke 'full' hanya bila sumber daya mencukupi
source_dir = Path('configs/smoke' if PROFILE == 'smoke' else 'configs')
config_dir = Path('configs/colab')
config_dir.mkdir(exist_ok=True)
for source in source_dir.glob('*.json'):
    cfg = json.loads(source.read_text())
    if source.name == 'base.json':
        cfg['device'] = 'cuda' if torch.cuda.is_available() else 'cpu'
        cfg['amp_dtype'] = ('bf16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else None)
    (config_dir / source.name).write_text(json.dumps(cfg, indent=2) + '\n')
print('Config:', config_dir, '| Device:', json.loads((config_dir/'base.json').read_text())['device'])


Config: configs/colab | Device: cuda


In [106]:
import h5py
for name in ('pile_train.h5', 'pile_dev.h5'):
    path = ARTIFACT_ROOT / 'data' / name
    if path.exists():
        with h5py.File(path, 'r') as handle:
            print(name, len(handle['tokens']), 'tokens')
    else:
        print('Belum ada:', path)


Belum ada: /ephemeral/data/pile_train.h5
Belum ada: /ephemeral/data/pile_dev.h5


## 4. Persiapan data opsional

Jalankan persiapan hanya untuk data yang belum tersedia. Setiap tombol proses dipisah agar unduhan dan pemakaian disk tetap terkendali. Persiapan Pile mengunduh shard terlebih dahulu sebelum batas token berlaku.


In [107]:
import subprocess
RUN_PREP_PRETRAIN = True
if RUN_PREP_PRETRAIN:
    for split, filename in [('val', 'pile_dev.h5'), ('train', 'pile_train.h5')]:
        command = [sys.executable, 'scripts/prepare_pretrain_data.py',
                   '--split', split, '--raw_dir', str(ARTIFACT_ROOT/'data'/'pile_raw'),
                   '--out', str(ARTIFACT_ROOT/'data'/filename),
                   '--max_tokens', '200000']
        subprocess.run(command, check=True, cwd=PROJECT_ROOT)


In [108]:
RUN_PREP_SFT = True
if RUN_PREP_SFT:
    dims = json.loads((config_dir/'base.json').read_text())
    subprocess.run([sys.executable, 'scripts/prepare_sft_data.py',
                    '--context_length', str(dims['context_length']),
                    '--out_dir', str(ARTIFACT_ROOT/'data'),
                    '--limit_per_set', '2000'], check=True, cwd=PROJECT_ROOT)


CalledProcessError: Command '['/usr/bin/python3', 'scripts/prepare_sft_data.py', '--context_length', '256', '--out_dir', '/ephemeral/data', '--limit_per_set', '2000']' returned non-zero exit status 1.

In [ ]:
RUN_PREP_PREFERENCES = True
if RUN_PREP_PREFERENCES:
    subprocess.run([sys.executable, 'scripts/prepare_preference_data.py',
                    '--source', 'both', '--max_per_source', '2000',
                    '--out_dir', str(ARTIFACT_ROOT/'data')], check=True, cwd=PROJECT_ROOT)


In [ ]:
RUN_PREP_RL = True
if RUN_PREP_RL:
    subprocess.run([sys.executable, 'scripts/prepare_rl_prompts.py',
                    '--out_dir', str(ARTIFACT_ROOT/'data'),
                    '--train_limit', '2000', '--test_limit', '200',
                    '--arith_n', '1000'], check=True, cwd=PROJECT_ROOT)


## 5. Jalankan tahap yang dipilih

Ubah hanya tahap yang siap menjadi `True`. Urutan data dan checkpoint mengikuti pipeline: pretrain → SFT → reward/DPO → PPO/GRPO → evaluasi. Semua perintah berjalan di runtime Colab dan memerlukan file data masing-masing.


In [109]:
RUN_PRETRAIN = True
if RUN_PRETRAIN:
    import subprocess
    command = [sys.executable, 'scripts/pretrain_base.py',
               '--config', str(config_dir / 'pretrain.json')]
    subprocess.run(command, check=True, cwd=PROJECT_ROOT)


CalledProcessError: Command '['/usr/bin/python3', 'scripts/pretrain_base.py', '--config', 'configs/colab/pretrain.json']' returned non-zero exit status 1.

In [ ]:
RUN_SFT = True
if RUN_SFT:
    import subprocess
    command = [sys.executable, 'scripts/train_sft.py',
               '--config', str(config_dir / 'sft.json')]
    subprocess.run(command, check=True, cwd=PROJECT_ROOT)


In [ ]:
RUN_REWARD = True
if RUN_REWARD:
    import subprocess
    command = [sys.executable, 'scripts/train_reward.py',
               '--config', str(config_dir / 'reward.json')]
    subprocess.run(command, check=True, cwd=PROJECT_ROOT)


In [ ]:
RUN_DPO = False
if RUN_DPO:
    import subprocess
    command = [sys.executable, 'scripts/train_dpo.py',
               '--config', str(config_dir / 'dpo.json')]
    subprocess.run(command, check=True, cwd=PROJECT_ROOT)


In [ ]:
RUN_PPO = False
if RUN_PPO:
    import subprocess
    command = [sys.executable, 'scripts/train_ppo.py',
               '--config', str(config_dir / 'ppo.json')]
    subprocess.run(command, check=True, cwd=PROJECT_ROOT)


In [ ]:
RUN_GRPO = False
if RUN_GRPO:
    import subprocess
    command = [sys.executable, 'scripts/train_grpo.py',
               '--config', str(config_dir / 'grpo.json')]
    subprocess.run(command, check=True, cwd=PROJECT_ROOT)


## 6. Evaluasi dan chat opsional

Gunakan checkpoint tahap yang sudah selesai. Evaluasi GSM8K memerlukan unduhan dataset dan dapat lama.


In [ ]:
RUN_EVALUATION = False
EVAL_STAGE = 'sft'
if RUN_EVALUATION:
    subprocess.run([sys.executable, 'scripts/eval_post_training.py',
                    '--ckpt', str(ARTIFACT_ROOT/'ckpts'/(EVAL_STAGE+'.pt')),
                    '--label', EVAL_STAGE, '--limit', '20',
                    '--device', json.loads((config_dir/'base.json').read_text())['device']],
                   check=True, cwd=PROJECT_ROOT)


In [ ]:
RUN_CHAT = False
CHAT_STAGE = 'sft'
PROMPT = 'Halo, apa kabar?'
if RUN_CHAT:
    subprocess.run([sys.executable, 'scripts/chat.py',
                    '--ckpt', str(ARTIFACT_ROOT/'ckpts'/(CHAT_STAGE+'.pt')),
                    '--prompt', PROMPT, '--max_new_tokens', '64'],
                   check=True, cwd=PROJECT_ROOT)


## 7. Simpan artefak

Checkpoint berada di `/ephemeral/ckpts`. Salin ke Google Drive sebelum runtime berakhir. Mount Drive hanya bila Anda siap memberi akses pada notebook ini.


In [ ]:
MOUNT_DRIVE = False
if MOUNT_DRIVE:
    import shutil
    from google.colab import drive
    drive.mount('/content/drive')
    drive_root = Path('/content/drive/MyDrive/train-llm-artifacts')
    for subfolder in ('data', 'ckpts'):
        destination_dir = drive_root / subfolder
        destination_dir.mkdir(parents=True, exist_ok=True)
        for source in (ARTIFACT_ROOT / subfolder).iterdir():
            if source.is_file():
                shutil.copy2(source, destination_dir / source.name)
                print('Saved to Drive:', destination_dir / source.name)
